# Great Britain Study 05 — What is a British horse race?

## Bounded study question

**What is a British horse race as a sporting and administrative object, and which properties actually specify the contest?**

This study continues the Great Britain conceptual sequence. It begins with authoritative British racing terminology and only then tests how those concepts are represented in Inside Rails data.

The study does **not** begin by assuming that a source row, a race-type label, a class value, a set of runners or a result record is itself a complete definition of the race.

The evidence writes the story.


## Why this study follows Studies 01–04

The preceding Great Britain studies established:

- the scale and calendar structure of the observed British racing programme;
- the top-level Flat / Jump structure and the position of Hurdle, Chase and National Hunt Flat racing;
- the distinction between racecourse identity, constituent course/track identity and time-bounded characteristics;
- the distinction between a BHA fixture and the looser contextual term meeting.

Those layers tell us **where and when racing is organised** and **what broad kind of racing is taking place**.

They do not yet tell us what makes one individual race the contest that it is.

Study 05 therefore moves one conceptual layer down:

> fixture / programme context → **race and race conditions** → runners and result


## Scope and stopping rule

### In scope

Establish, using authoritative British racing evidence:

1. what the BHA means by a race and by race conditions;
2. which properties may govern eligibility to take part;
3. how weights, penalties and allowances relate to the race conditions;
4. how handicap and non-handicap races differ at the level needed to understand the contest;
5. how race class, race type and named categories relate to the underlying conditions;
6. which properties appear to define/specify the race and which are descriptive, derived or mutable;
7. how the governed Inside Rails fields represent those concepts.

### Out of scope unless the evidence makes it necessary

- betting strategy or profitability;
- predictive modelling;
- exhaustive analysis of every race subtype;
- a new fixture model;
- physical track assignment below governed racecourse identity;
- Database v5 design;
- automatically ingesting all newly discovered BHA data.

Stop when we can explain coherently what information specifies a British race and map that explanation to the governed data without inventing unsupported identities or semantics.


## Evidence and data boundary

### Authoritative evidence first

British Horseracing Authority material is the primary starting point for British racing terminology, including:

- Rules of Racing and General Instructions;
- race-planning / programme material;
- official glossary of race types;
- handicapping and weight guidance;
- official race/result material where a concrete example is useful.

The recent discovery that BHA systems expose rich structured information does **not** mean that every available field becomes part of the Inside Rails database. External BHA information can be used as bounded evidence or study-specific data where appropriate.

### Accepted analytical database

Use accepted Database v4 read-only:

`data/processed/database/releases/inside_rails_v4.sqlite3`

Normal race-level interface:

`view_reconciled_race_occurrences`

Great Britain racecourse-aware interface when racecourse identity is material:

`view_gb_reconciled_race_occurrences_with_racecourse`

Expected Great Britain race population in the latter view: **111,634 race occurrences**.

Do not query immutable Source Version 1 directly unless a question specifically concerns raw source evidence or source semantics.


In [1]:
from pathlib import Path

import pandas as pd

from inside_rails.source_sqlite import connect_read_only


def find_project_root(start: Path) -> Path:
    current = start.resolve()
    while current != current.parent:
        if (current / 'pyproject.toml').exists():
            return current
        current = current.parent
    raise RuntimeError('Could not locate Inside Rails project root')


PROJECT_ROOT = find_project_root(Path.cwd())
DATABASE = PROJECT_ROOT / 'data/processed/database/releases/inside_rails_v4.sqlite3'

assert DATABASE.exists(), f'Accepted Database v4 not found: {DATABASE}'

print(f'Project root: {PROJECT_ROOT}')
print(f'Study database: {DATABASE}')


Project root: /home/rob/Documents/inside-rails-horse-racing
Study database: /home/rob/Documents/inside-rails-horse-racing/data/processed/database/releases/inside_rails_v4.sqlite3


# Stage 1 — What are the conditions of a race?

## Current question

Before looking at our columns, establish:

> **What does the BHA mean by race conditions, and what parts of the contest can those conditions govern?**

This is deliberately narrower than trying to catalogue every British race type immediately.

Questions to resolve from authoritative material:

- What determines which horses are eligible to enter or run?
- Can eligibility depend on age, sex, rating, previous wins, sales history or other restrictions?
- What determines the weight each horse is due to carry?
- How do penalties and allowances alter those weights?
- What distinguishes a handicap from a conditions/weight-for-age contest?
- Where do race class, Pattern/Listed status and named race types fit?
- Which of these are part of the published conditions of the race rather than facts observed only after it has been run?


## Initial authoritative source set

Starting points:

- BHA Rules of Racing and guides  
  https://www.britishhorseracing.com/regulation/rules-guides/

- BHA glossary of race types  
  https://www.britishhorseracing.com/regulation/glossary-of-race-types/

- BHA guide to handicapping  
  https://www.britishhorseracing.com/regulation/guide-to-handicapping/

- BHA explanation of how handicap weights are determined  
  https://www.britishhorseracing.com/races-can-horse-run-handicap-rating/

- Current BHA Programme Book / race-programme change material where examples are needed  
  https://www.britishhorseracing.com/2026-programme-book-1-update/

These are evidence leads, not a pre-written conclusion. Each material claim used in the study should be tied to the authoritative source that establishes it.


## First evidence note

The starting BHA material already indicates that **race type alone is not enough to describe the contest**.

The BHA's glossary and handicapping guidance show that race conditions can affect at least two distinct things:

1. **eligibility** — which horses may take part; and
2. **weight terms** — how much weight eligible horses are required to carry, including the operation of ratings, penalties and allowances in different kinds of races.

Examples in the BHA glossary show restrictions based on concepts such as ratings, age, previous wins and sales-related criteria, while conditions races allocate weights by rules other than a horse's handicap mark.

### Working proposition — not yet a conclusion

A British race may be better understood as a **contest specified by a bundle of conditions** than as merely a race-type label attached to a set of runners.

The next step is to test that proposition against the formal Rules / race-planning material and establish the vocabulary precisely before touching the database.


## Next action

Read the relevant BHA Rules of Racing / race-planning material for formal terminology around:

- race conditions;
- qualification / eligibility;
- weights, penalties and allowances;
- handicap versus non-handicap conditions.

Then record exactly what that evidence establishes and only after that decide the first database question.

**No descriptive database analysis has yet been run in this study.**


## Evidence note 1 — race conditions govern more than one dimension of the contest

The first BHA evidence establishes that **race conditions are prospective rules governing the contest**, rather than merely a descriptive label applied after the event.

### 1. Race conditions can determine eligibility

A BHA notice concerning a Brighton race on 9 June 2023 provides a particularly clear example.

The race was restricted to horses that were:

- four years old or older;
- rated 46–65; and
- had not won more than two races.

Four horses were declared non-runners because they did not satisfy those conditions. The BHA explicitly described them as **ineligible to race**.

Source:

https://www.britishhorseracing.com/?p=32370

This establishes that at least some published race conditions determine **which horses are permitted to participate**.

### 2. Eligibility and weight allocation are separate concepts

The BHA glossary distinguishes races in which a handicap rating affects eligibility from races in which it determines weight.

For example, in **Classified Stakes**, eligibility may be restricted by handicap mark while all qualifying horses carry the same weight.

By contrast, in **Conditions races**, the weights carried are determined by the conditions of the race rather than directly by each horse's handicap mark. Those conditions may incorporate:

- the weight-for-age scale;
- sex allowances; and
- penalties for previous success.

Source:

https://www.britishhorseracing.com/regulation/glossary-of-race-types/

This means two separate questions must be kept distinct:

1. **Is the horse eligible for this race?**
2. **If eligible, what weight is it required to carry?**

### 3. Handicap weights can also be modified by additional rules

The BHA's handicapping guidance states that a horse's handicap rating is not the only factor affecting the weight it carries in a handicap.

Other factors can include:

- weight-for-age allowances;
- apprentice or conditional jockey allowances; and
- penalties.

Source:

https://www.britishhorseracing.com/races-can-horse-run-handicap-rating/

### What this establishes

At this stage, the evidence supports a narrower statement than our working proposition:

> **The conditions of a British race can independently govern both eligibility to participate and the weight terms under which eligible participants compete.**

It does **not yet establish an exhaustive definition of "race conditions"**, nor which other properties — distance, age band, sex restriction, class, race category, prize, course or other terms — belong to that concept.

## Next question

> **What other properties of a race are specified prospectively as part of its conditions?**

The next step is therefore to examine an actual published British race specification / race conditions entry and identify its components before attempting to classify them.

## Evidence note 2 — a race is specified across several prospective dimensions

The next BHA evidence shows that the conditions of a race can govern more than eligibility and weight.

### Distance can be part of the specification

For the Windsor Castle Stakes at Royal Ascot, the BHA announced two separate changes for 2026:

- the race distance would increase from 5 furlongs to 6 furlongs; and
- the race conditions would change so that eligibility depended on the distance over which the horse's sire had won.

The race remained the Windsor Castle Stakes, but important properties of the contest changed.

Source:

https://www.britishhorseracing.com/press_releases/changes-to-windsor-castle-stakes-at-royal-ascot-from-2026/

This is useful because it separates several concepts:

- **race identity/name**;
- **distance**;
- **eligibility conditions**;
- **race status/category**.

They are related properties of the contest, but they are not necessarily the same thing.

### Age, race type, rating bands and qualification rules can also be specified prospectively

The BHA's 2026 two-year-old programme describes races using combinations of:

- age;
- novice or maiden status;
- Open or Restricted status;
- auction / median-auction qualification bands;
- race distance;
- handicap rating bands;
- race class;
- weight allowances.

For example, the 2026 Nursery programme includes:

- 0–65 — Class 6;
- 0–75 — Class 5;
- 0–85 — Class 4;
- 0–95 — Class 3;
- Open — Class 2.

The same programme also contains specific rules determining when a two-year-old becomes eligible to compete in certain Nursery Handicaps.

Source:

https://www.britishhorseracing.com/changes-to-2026-two-year-old-programme/

This reinforces the distinction between:

> **what kind of race is being staged**

and

> **the detailed conditions under which a horse qualifies for that particular contest**.

### Prize money is also published within the race conditions

The BHA's guidance for owners states that the **total prize fund (TPF)** for each race is detailed in the race conditions.

Source:

https://www.britishhorseracing.com/regulation/ownership/owners-toolkit/

This establishes prize fund as another prospective published property of the race, although we have not yet established whether prize money should be considered part of the race's sporting definition or simply an administrative/economic condition attached to it.

## Current conceptual inventory

The evidence now gives us at least the following prospective properties:

| Property | What it appears to govern |
|---|---|
| race type/category | broad form of contest |
| age | eligibility |
| sex | potentially eligibility and/or weight |
| handicap rating | eligibility and/or weight |
| previous performance/wins | eligibility and/or penalties |
| sales/breeding criteria | eligibility and/or allowances |
| distance | physical specification of contest |
| weight terms | competitive terms |
| penalties | modification of weight terms |
| allowances | modification of weight terms |
| class | programme/classification level |
| prize fund | economic terms of the race |

This table is **not yet a formal model**. Some properties may overlap, some may be derived from others, and some may be attributes of the race rather than components of its conditions in the strict Rules-of-Racing sense.

### Important observation

A race therefore cannot be adequately represented simply as:

> `race type + runners`

Even before the race takes place, there is a substantial specification describing **who may compete, over what contest, and on what terms**.

## Next question

We now need to distinguish between three things that are beginning to get mixed together:

1. **race type** — e.g. Handicap, Maiden, Novice, Classified Stakes;
2. **race class/status** — e.g. Class 1–6, Listed, Group/Grade;
3. **race conditions** — the detailed rules governing a particular contest.

> **How do race type, class/status and individual race conditions relate to one another?**

That is the next terminology problem to resolve before we attempt a formal model of a British race.

### Publication note — identity is not the same as specification

The Windsor Castle Stakes example may be useful for later reader-facing work.

The BHA can alter important properties of an established race — including its **distance** and **eligibility conditions** — while continuing to stage it under the same named race identity.

That suggests a useful distinction:

> **A race can retain its identity while its specification changes.**

In other words:

- **race identity** answers *which race is this?*
- **race specification** answers *what contest is being staged this time?*
- **race conditions** form at least part of that specification.

This distinction may become important later when analysing named races through time. Treating every edition of a named race as mechanically identical could conceal meaningful changes in the contest itself.

Preserve this point for potential reader-facing explanation; do not yet turn it into a formal database identity rule.

## Evidence note 3 — race type, class/status and race conditions are different layers

The BHA evidence does not support treating **race type**, **class/status** and **race conditions** as interchangeable descriptions of a race.

They answer different questions.

### 1. Race type describes the form of contest

The BHA glossary identifies types such as:

- Maiden;
- Novice;
- Handicap;
- Classified Stakes;
- Conditions race;
- Seller;
- Claimer;
- Auction race;
- National Hunt Flat race;
- Hunter Chase.

Importantly, these labels are not always mutually exclusive.

The BHA explicitly describes:

- maiden handicaps;
- novice handicaps;
- restricted handicaps;
- apprentice/conditional races that may themselves be handicaps or non-handicaps.

Source:

https://www.britishhorseracing.com/regulation/glossary-of-race-types/

So a race does not necessarily possess one simple mutually exclusive `race_type`.

Instead, several concepts can combine to describe the form and restrictions of the contest.

### 2. Class describes another property of the race

BHA race programmes separately assign races to classes.

For example, official BHA race listings have contained combinations such as:

- Handicap Hurdle Race — Class 3;
- Novices' Steeple Chase — Class 1;
- Mares' Hurdle Race — Class 1, Grade 2;
- Handicap Steeple Chase — Class 1, Listed;
- National Hunt Flat Race — Class 6.

This demonstrates that **class is not itself the race type**.

Different race types can appear within different classes.

Example BHA source:

https://www.britishhorseracing.com/press_releases/stanjames-com-champion-hurdle-trial-transferred-to-doncaster-on-saturday-january-26th/

### 3. Pattern / Listed status adds another classification layer

The BHA describes Group and Grade races as part of the Pattern hierarchy:

- Group/Grade 1;
- Group/Grade 2;
- Group/Grade 3.

Listed races sit below the Pattern and, together with Group/Graded races, form the races commonly referred to as **black type**.

Pattern status is therefore not simply another word for race type.

A race can simultaneously possess:

- a sporting form/type;
- a class;
- and a Pattern/Listed status.

Source:

https://www.britishhorseracing.com/regulation/glossary-of-race-types/

### 4. Individual race conditions parameterise the particular contest

Even after knowing the broad race type and class/status, further conditions can determine who is actually eligible and on what terms.

The BHA gives examples of handicaps being additionally restricted to horses that:

- have not won since a specified date; or
- have never won above a specified value.

Current race-programme rules can also impose additional qualification requirements on particular groups of races.

For example, the BHA's 2026 Jump programme requires horses competing in Grade 1 Novices' and Juvenile Hurdles to have achieved a specified minimum assessment, with the requirement reflected in the relevant race conditions.

Source:

https://www.britishhorseracing.com/2026-programme-book-1-update/

This means that knowing only:

> `Novice Hurdle — Class 1 — Grade 1`

still does not necessarily tell us everything required to determine eligibility for the particular race.

## Current model

The evidence now suggests a layered description:

> **race type/form**  
> + **class / quality tier**  
> + **Pattern or Listed status where applicable**  
> + **individual race conditions**

These layers interact, and some terminology can combine, so this should not yet be treated as a rigid database schema.

But they are clearly not one interchangeable field.

### Important consequence for Inside Rails

A single source field labelled something like `race_type`, `class` or `conditions` should not be assumed to encode the complete specification of a British race.

Before analysing any such field, we need to establish exactly which conceptual layer it represents.

## Next question

We now have a more fundamental issue.

If a particular race is specified by a combination of properties and conditions:

> **Which properties describe the contest itself, and which describe the horses that eventually participate in it?**

For example:

- `for 3yo+` exists before the runners are known;
- the actual ages of the runners are observed participant properties;
- `0–85` is an eligibility condition;
- each runner's actual handicap mark is a participant property;
- a weight rule exists prospectively;
- the weight actually carried belongs to an individual runner.

The next step is to separate **race-level specification** from **runner-level realisation**.

## Evidence note 4 — the race specification is not the same thing as the realised field

The evidence now allows us to separate the **race itself as a prospective contest** from the horses that eventually take part in it.

### 1. The race exists before its final participants are known

The BHA describes owners entering horses into races and paying an entry fee.

It separately describes the later act of **declaring a horse to run**.

Source:

https://www.britishhorseracing.com/regulation/ownership/owners-toolkit/

This establishes that:

> **entry and declaration are different stages.**

A horse may therefore be associated with a race before it becomes part of the official declared field.

### 2. Even declarations have a provisional and official state

The BHA's declaration-tracking guidance distinguishes:

- provisional declarations;
- provisional runners and riders;
- formally checked and published declarations.

It also warns that subsequent changes can still affect race times and final fields.

Source:

https://www.britishhorseracing.com/press_releases/transparent-declaration-tracking-available-on-bha-website/

So the set of horses associated with a race can change while the underlying scheduled contest remains.

### 3. A declared horse is not necessarily a runner

A horse that has been declared to run can subsequently become a **non-runner**.

The distinction can survive remarkably late into the process.

Under the current BHA Rules, Stewards may in specified circumstances declare a horse a non-runner because of an incident at the start.

Source:

https://www.britishhorseracing.com/press_releases/bha-confirms-rule-change-regarding-non-runners-at-the-start-of-jumps-races/

This means:

> **declared field ≠ necessarily the field that actually participates in the race.**

The participant population of the realised race is therefore not fixed merely by knowing the declarations.

### 4. Some participant terms are realised at raceday

The BHA describes the jockey **weighing out** before a race.

The Clerk of the Scales checks that the rider and equipment meet the required weight shown on the racecard, allowing for any applicable claim.

Source:

https://www.britishhorseracing.com/about/faqs/

This gives another useful distinction:

- the race establishes the rules under which weights are allocated;
- a particular horse receives a specified weight under those rules;
- raceday procedures establish the weight actually carried by that participant.

These are related facts, but they exist at different conceptual levels.

## Emerging lifecycle

We can now sketch a race without yet treating this as a final formal model:

> **race specification / conditions**  
> ↓  
> **entries**  
> ↓  
> **declarations**  
> ↓  
> **final declared field**  
> ↓  
> **actual starters / runners**  
> ↓  
> **result**

The important point is that these stages should not be collapsed into one another.

### Race-level properties

Examples include:

- distance;
- age/sex restrictions;
- rating band;
- race type;
- class/status;
- eligibility rules;
- weight terms;
- prize fund.

These describe the prospective contest.

### Participant-level properties

Examples include:

- horse identity;
- age and sex of the particular horse;
- handicap mark;
- assigned weight;
- jockey;
- applicable jockey claim;
- declared equipment;
- whether the horse ultimately started;
- actual weight carried;
- finishing position.

These describe how a particular horse relates to or participates in that contest.

## Important consequence

The horses that happened to run cannot by themselves define what the race was.

For example, if a race is open to horses aged **3yo+**, but every actual runner happens to be four years old, the realised field does not turn the contest into a four-year-old-only race.

Likewise, a `0–85` race does not cease to be a `0–85` contest merely because the highest-rated horse that actually runs is rated 79.

> **Conditions describe the permitted contest; participants describe one realisation of it.**

This distinction will matter whenever we try to infer race conditions from result data.

## Next question

We now have another layer to resolve:

> **What makes a handicap fundamentally different from a non-handicap race?**

We know that both can contain eligibility rules and weight terms.

The next step is therefore not simply to list handicap race types, but to establish the organising principle:

> **Why are horses given different weights in a handicap, and what is the intended relationship between handicap ratings and those weights?**

## Evidence note 5 — a handicap is defined by what the weight is trying to do

The distinction between handicap and non-handicap racing is not simply that one race has different weights and another has equal weights.

The more fundamental difference is **why each horse is assigned its weight**.

### 1. In a handicap, weight is used to compensate for assessed differences in ability

The BHA describes the purpose of handicap races as enabling horses of different abilities to compete competitively through the allocation of weight.

A handicap rating is the BHA's numerical assessment of a horse's current level of ability.

The scale is expressed in pounds.

If Horse A is rated three pounds higher than Horse B, the handicapping principle is that Horse A should carry three pounds more than Horse B for the two horses to have theoretically equal chances, all else being equal.

Source:

https://www.britishhorseracing.com/regulation/guide-to-handicapping/

The BHA summarises the principle as:

> the better-rated horse carries more weight, while lower-rated horses carry less.

Source:

https://www.britishhorseracing.com/about/handicapping/

### 2. The handicap rating therefore performs two distinct jobs

A rating can affect:

1. **eligibility** — whether a horse falls within the permitted rating band for a particular race; and
2. **weight allocation** — how much weight it is due to carry relative to its opponents.

For example, a handicap might be advertised for horses rated **66–80**.

If the top-rated horse is rated 80 and carries 9st 7lb:

- an 80-rated horse carries 9st 7lb;
- a 70-rated horse would normally carry 10lb less;
- a 66-rated horse would normally carry 14lb less.

Source:

https://www.britishhorseracing.com/races-can-horse-run-handicap-rating/

So the rating is not merely descriptive information about the horse.

Within a handicap it becomes part of the mechanism used to establish the competitive terms of the race.

### 3. A handicap is attempting to reduce the advantage of superior ability

The underlying objective is important.

The BHA explicitly contrasts handicaps with largely level-weights elite racing.

At level weights, a substantially superior horse would normally be expected to beat an inferior horse.

A handicap deliberately asks the superior horse to carry additional weight in an attempt to give horses of differing assessed ability a more realistic chance of competing against one another.

This gives us an important conceptual distinction:

> **A handicap is not primarily a race for inferior horses or a particular class of horse. It is a method of constructing competition through ability-related weight allocation.**

### 4. Non-handicap does not mean "every horse carries the same weight"

This is an important correction to an easy assumption.

The BHA describes Conditions races as races in which weights are **not determined by the horses' handicap marks**.

Instead, their weights may be determined by conditions involving:

- weight-for-age;
- sex allowances;
- penalties for previous victories or achievements.

Source:

https://www.britishhorseracing.com/regulation/glossary-of-race-types/

Therefore:

> **handicap ≠ different weights**
>
> and
>
> **non-handicap ≠ equal weights**

Different weights can occur in both.

The distinction is the **rule by which those weights are derived**.

### 5. Pattern racing illustrates the alternative objective

The BHA describes the broad premise of Group races as largely level-weights contests intended to establish which horse is best.

Those races can still incorporate:

- weight-for-age;
- allowances for fillies and mares;
- penalties in some circumstances.

So even elite non-handicap racing need not mean literal identical weights.

Its competitive purpose is nevertheless different from handicapping.

A simplified contrast is:

| Handicap | Non-handicap / conditions-based contest |
|---|---|
| assessed ability helps determine weight | handicap mark does not determine weight |
| stronger-rated horses generally concede weight | weights follow the published conditions |
| seeks to make differing abilities more competitive | can seek to compare ability under prescribed terms |
| rating may govern eligibility and weight | eligibility and weight may depend on age, sex, wins, status or other conditions |

## Important conceptual result

We can now refine our model of a race.

**Weight carried** is not enough information by itself.

To understand the sporting terms of the race we also need to know:

> **Why was that weight assigned?**

Two horses carrying different weights could be doing so because of:

- their relative handicap ratings;
- their ages;
- sex allowances;
- previous victories;
- jockey claims;
- another condition of the race.

Those mechanisms have very different sporting meanings.

### Publication note — "giving every horse an equal chance"

The popular description of handicapping as "giving every horse an equal chance" should be treated carefully in later reader-facing work.

It describes the **objective of the handicap mechanism**, not a factual claim that every horse actually has an identical probability of winning.

Ratings are assessments of ability, horses' ability changes, and many factors other than weight affect race performance.

That distinction may become important when we later examine betting markets or model race outcomes.

## Next question

The handicap system has now introduced another concept that we need to understand properly:

> **What exactly is a handicap rating?**

In particular:

- what is the rating intended to measure?
- how is it established?
- when does a horse first receive one?
- how does it change after subsequent performances?
- what is the difference between a **handicap rating** and a **performance figure**?

We should resolve those concepts before treating a numerical rating as simply another horse attribute.

## Scope boundary — handicapping becomes a separate study

The evidence above establishes enough about handicapping for the purposes of Study 05:

> In a handicap, assessed ability contributes to the allocation of weight; in a non-handicap race, weight is determined through other published conditions.

Understanding **how handicap ratings are created, maintained and revised** is a separate conceptual problem.

Questions deferred to a dedicated handicapping study include:

- what exactly a handicap rating measures;
- how a horse becomes eligible for an initial rating;
- how ratings are derived from performances;
- how and when ratings change;
- the relationship between handicap ratings and performance figures;
- penalties and reassessment timing;
- differences between Flat and Jump handicapping where material.

These questions are important, but they are not required to answer the present study's bounded question:

> **What is a British horse race?**

Study 05 therefore treats handicapping only as one possible mechanism within the specification of a race and does not attempt to explain the handicapping system itself.

### Candidate next study

**Great Britain Study 06 — How does British handicapping work?**

## Evidence note 6 — what specifies an individual programmed race?

We can now look at how the BHA itself describes individual races within a race programme.

A useful example comes from the replacement Sunday Series fixture staged at Musselburgh on 26 April 2026.

The BHA published the revised programme using descriptions containing combinations such as:

- age eligibility;
- distance;
- sex restriction;
- race type;
- restricted-race band;
- class;
- prize value.

Examples included:

- a two-year-old, 5f, EBF Restricted Maiden, Band B;
- a three-year-old-and-up, 1m 1f, fillies-only, Class 5 Handicap;
- a four-year-old-and-up, 2m, Class 4 Handicap;
- a four-year-old-and-up, 1m, Apprentice, Class 4 Handicap.

Each race also had a stated prize value.

Source:

https://www.britishhorseracing.com/press_releases/musselburgh-to-host-replacement-sunday-series-fixture-on-26-april/

### Why this example is particularly useful

The fixture was replacing one originally programmed for Ayr.

Because Musselburgh has a different track configuration and different starting points, the BHA altered parts of the race programme.

Examples included changes from:

- 6f to 5f;
- 1m 2f to 1m 1f;
- 6f to 1m 4f 110y.

Other races in the programme were explicitly described as having **no change**.

This demonstrates that the programme contains individual race specifications whose properties can be deliberately amended when circumstances require it.

It also reinforces our earlier distinction:

> **The fixture provides the organisational context, while each race within that fixture has its own sporting specification.**

---

## Separating the properties

The evidence so far suggests that the information associated with a published race does not all perform the same function.

### A. Sporting specification

These properties determine the nature or permitted terms of the contest itself.

Examples include:

- code / broad form of racing;
- distance;
- race type;
- age restriction;
- sex restriction;
- rating restriction;
- novice / maiden / other qualification requirements;
- restricted-race qualification;
- handicap or non-handicap weight mechanism;
- penalties and allowances;
- other explicit eligibility conditions.

These answer questions such as:

> **Who may compete?**

> **Over what contest?**

> **Under what competitive terms?**

### B. Classification / status

Other properties position the race within the wider racing programme.

Examples include:

- Class 1–6;
- Group / Grade status;
- Listed status;
- restricted-race band where applicable.

These tell us something about the category, level or status of the contest but are not necessarily sufficient to reconstruct its full conditions.

### C. Economic terms

Prize money is prospectively attached to the race.

The BHA states that the Total Prize Fund for each race is detailed in its race conditions.

Source:

https://www.britishhorseracing.com/regulation/ownership/owners-toolkit/

Prize value is therefore unquestionably a published property of the race.

However, we should keep open the conceptual question of whether prize money forms part of the **sporting specification** of the contest or is better treated as an economic property attached to that contest.

### D. Fixture / scheduling context

A race also appears within a wider administrative context including:

- racecourse;
- fixture;
- date;
- scheduled time;
- position/order on the card.

These are important for locating and administering the race, but they should not automatically be treated as sporting conditions.

The BHA has previously permitted races at certain fixtures to be **re-ordered at declaration stage**, with their scheduled times moving while the programmed races themselves remained.

Source:

https://www.britishhorseracing.com/2024-programme-book-1-update/

This is strong evidence that:

> **scheduled time/order and sporting specification are different concepts.**

Changing when a race is run does not necessarily mean that a different sporting contest has been created.

---

## Emerging anatomy of a British race

The evidence now supports a more structured picture:

> **Fixture context**
>
> where and when the programme is being staged
>
> ↓
>
> **Individual race**
>
> possessing a prospective sporting specification
>
> ↓
>
> **Eligibility + competitive terms**
>
> determining which horses may participate and on what basis
>
> ↓
>
> **Entries / declarations / actual runners**
>
> one realised field drawn from the population permitted by those conditions
>
> ↓
>
> **Result**
>
> the outcome of that particular realisation of the contest

### Important qualification

We have **not yet established a formal identity rule** for an individual race.

For example, we should not yet claim that some combination such as:

`fixture + race number`

or

`course + date + time`

is the authoritative persistent identity of a race.

Study 04 already taught us the danger of turning convenient data combinations into sporting identities without evidence.

What we have established here is narrower:

> **An individual British race has a prospective specification containing multiple sporting properties, and that specification is conceptually distinct from both its fixture context and its eventual participants/result.**

## Next question

One major part of the sporting specification remains surprisingly vague:

> **What does race distance actually mean in British racing?**

Before treating `5f`, `1m`, `2m 4f` and similar values as simple measurements, we need to establish:

- how British race distances are specified;
- whether advertised and actual distances can differ;
- how yards/furlongs/miles are represented;
- how rail movements or course configuration can affect the distance actually covered;
- whether Flat and Jump racing handle this differently.

If that expands into a substantial independent subject, it should become another dedicated study rather than consuming Study 05.

## Evidence note 7 — first working definition of a British horse race

We now have enough evidence to attempt a provisional definition.

This is **not yet the final conclusion of Study 05**. Its purpose is to make our current understanding explicit so that the remaining work can test it rather than continuing to accumulate disconnected terminology.

### First working definition

> **A British horse race is an individual contest programmed within a fixture and specified prospectively by conditions governing the nature of the contest, who may compete, and the competitive terms under which eligible horses participate. The programmed contest is then realised by a particular set of runners and produces a result.**

This definition deliberately separates three layers.

### 1. The programmed contest

Before the race is run, the contest already exists as a specified sporting object.

Its specification can include properties such as:

- distance;
- race type;
- age restrictions;
- sex restrictions;
- rating restrictions;
- qualification requirements;
- handicap or non-handicap weight terms;
- penalties and allowances;
- race class/status;
- other explicit conditions.

These properties describe **what race has been programmed**.

### 2. The realised participation

The programmed race is subsequently populated by actual participants.

The participant process may include:

> entries → declarations → final declared field → actual starters/runners

The horses that eventually run are therefore one **realisation** of the previously specified contest.

They do not themselves define the full permitted scope of that contest.

### 3. The result

Once the race is run, it produces an outcome including:

- finishing positions;
- margins / beaten distances;
- winner;
- finishing times where available;
- other result-level observations.

The result is evidence about **what happened in that particular running**.

It is not the definition of the prospective race conditions.

---

## What this definition deliberately does not say

### It does not define a race as a database row

A row in Inside Rails is an analytical representation of a race occurrence.

The sporting concept must be established independently of the database implementation.

### It does not define a race by its runners

A race restricted to `3yo+` remains a `3yo+` contest even if every horse that actually starts is four years old.

### It does not define a race solely by its race type

`Handicap`, `Maiden`, `Novice` or another type label captures only part of the specification.

### It does not define race identity from date + course + time

Those properties may be useful identifiers in particular datasets, but Study 04 already demonstrated the danger of promoting convenient combinations into authoritative sporting identities without evidence.

### It does not require every associated property to be part of the sporting specification

Properties such as:

- prize money;
- scheduled time;
- race number/order;
- sponsorship/name;
- television coverage;

may be attached to a race without necessarily defining the competitive sporting contest itself.

Their exact conceptual status should be kept separate where material.

---

## Test of the working definition

Before accepting this definition, we should ask whether it survives some simple changes.

### Change 1 — scheduled time moves

If the same programmed race moves from 14:10 to 14:25:

> Is it still the same sporting contest?

Our evidence suggests **yes**.

### Change 2 — a horse becomes a non-runner

If a declared horse is withdrawn:

> Has the programmed race itself changed?

The realised field has changed, but the underlying conditions normally have not.

### Change 3 — race distance changes before the programme is finalised

If a race is changed from 5f to 6f:

> Has its sporting specification changed?

Clearly **yes**, even if its established race name survives.

### Change 4 — eligibility conditions change

If the age, rating or qualification restriction changes:

> Has the sporting specification changed?

Again, **yes**.

### Change 5 — prize money changes

This is less clear.

The economic terms of the race have changed, but the underlying competitive contest may otherwise remain identical.

This supports keeping **sporting specification** and **economic attributes** conceptually separate.

---

## Current conclusion

The evidence now supports a useful distinction:

> **race specification ≠ participants ≠ result**

and:

> **race specification ≠ fixture context**

The race sits between the fixture and the participants:

> fixture  
> ↓  
> **programmed race / sporting specification**  
> ↓  
> entries and declarations  
> ↓  
> actual runners  
> ↓  
> result

This is currently the strongest candidate for the conceptual structure Study 05 is trying to establish.

## Next question

The remaining task is now practical:

> **How well does Database v4 represent this conceptual structure?**

Before analysing the full Great Britain population, inspect the governed race-level interface and identify which fields correspond to:

1. fixture/context properties;
2. sporting-specification properties;
3. participant-derived properties;
4. result properties;
5. fields whose semantics remain incomplete or ambiguous.

This will be the first point in Study 05 where we deliberately look at the database.

## Methodological boundary — BHA-first interpretation and validation

The discovery of the BHA's structured racing data materially changes how Inside Rails should approach British racing studies.

For Great Britain, the British Horseracing Authority is the authoritative source for the official sporting and administrative concepts being studied.

Therefore Study 05 adopts the following hierarchy.

### 1. Establish the concept from BHA evidence

Definitions and relationships such as:

- race;
- race conditions;
- race type;
- eligibility;
- class/status;
- handicap/non-handicap status;
- distance;
- weights and allowances;
- entries, declarations and results;

should be established from official BHA material wherever suitable evidence exists.

Inside Rails source fields must not be used to define these concepts merely because their names appear plausible.

### 2. Use BHA structured data as the authoritative comparison dataset

Where the BHA exposes structured fixture, race, runner or result information relevant to the study, use that information to establish how official British races are represented in practice.

The BHA data should therefore be used to answer questions such as:

- which properties are recorded for an official race;
- how those properties correspond to the published conditions;
- whether particular Inside Rails values agree with the official record;
- which official information is absent from Inside Rails;
- whether apparently similar fields actually represent the same concept.

### 3. Treat Database v4 as the dataset being evaluated, not the authority being interpreted

Database v4 remains the current governed Inside Rails analytical release.

Its role in this stage is:

> **represent the existing Inside Rails understanding of the race and test that representation against authoritative BHA evidence.**

Agreement strengthens confidence in the governed field.

Disagreement must be investigated rather than silently choosing either value.

Possible causes include:

- an Inside Rails source error;
- an Inside Rails semantic misunderstanding;
- a transformation or governance defect;
- differing observation times;
- a BHA programme change between publication stages;
- incomplete or historically limited BHA data;
- genuinely different concepts represented by superficially similar fields.

### 4. Do not automatically copy the BHA dataset into Inside Rails

Authoritative external evidence does not imply automatic database ingestion.

BHA data may be used as:

1. conceptual authority;
2. bounded verification evidence;
3. a study-specific comparison dataset;
4. governed database input only where later evidence shows integration is necessary and justified.

Study 05 does **not** design Database v5.

### Revised next question

Instead of asking only:

> How well does Database v4 represent our conceptual model?

ask:

> **How does the BHA officially represent an individual British race, and how well does Database v4 reproduce that representation?**

The comparison should proceed field by field and concept by concept rather than assuming that similarly named fields are equivalent.

# Stage 2 — How does the BHA represent an individual race?

Study 05 now moves from conceptual evidence to structured official data.

We already established during the Great Britain race-population completeness audit that the BHA exposes individual race records through its structured racing service.

The successful 27 May 2026 pilot retrieved:

- **5 result-bearing BHA fixtures**;
- **34 individual BHA races**;
- **34 corresponding Database v4 race occurrences**.

The population reconciled **34/34** before detailed race-level comparison.

That previous acquisition work is inherited here as source-discovery evidence. Study 05 does not need to rediscover the BHA service or reverse-engineer its frontend again.

## Properties exposed by the BHA race record

Across those 34 official race records, the BHA exposed fields including:

| BHA field | Apparent concept |
|---|---|
| `raceId` | BHA race identifier |
| `yearOfRace` | BHA race-identity context |
| `divisionSequence` | division/instance context |
| `raceDate` | race date |
| `raceTime` | scheduled/published race time |
| `raceName` | published race title/description |
| `ageLimit` | age eligibility |
| `raceClass` | race class |
| `ratingBand` | rating eligibility band |
| `raceCriteriaRaceType` | broad Flat/Jump criterion |
| `rawDistanceText` | BHA distance description |
| `distanceValue` | structured distance value |
| `distanceText` | formatted distance |
| `distanceChange` | change to race distance |
| `distanceChangeText` | resulting distance after change |
| `prizeAmount` | prize amount |
| `prizeCurrency` | prize currency |
| `goingText` | going/surface condition |
| `blackTypeRace` | black-type indicator |
| `abandonedReasonCode` | abandonment status/reason |
| `winnersDetails` | realised result information |

The record therefore contains information from **several conceptual layers at once**.

### Prospective sporting specification

Examples:

- `ageLimit`
- `raceClass`
- `ratingBand`
- `raceCriteriaRaceType`
- distance fields
- parts of `raceName`

### Fixture / scheduling context

Examples:

- `raceDate`
- `raceTime`
- BHA fixture identifiers inherited from the parent fixture

### Economic/contextual attributes

Examples:

- `prizeAmount`
- `prizeCurrency`
- `goingText`

### Realised result information

Example:

- `winnersDetails`

This supports the conceptual model developed earlier in Study 05:

> **one BHA race record can contain both the prospective specification of the contest and information about its eventual realisation.**

It should therefore not be interpreted as though every field belongs to one conceptual layer.

## Concrete official example

One BHA race retrieved for Hamilton Park on 27 May 2026 was represented as:

- `raceId`: **2959**
- `raceTime`: **14:05**
- `raceName`: **THE WYVIS ROOFING EBF MAIDEN STAKES (CLASS 4) (Hamilton Park 2yo Series Qualifier) (GBB RACE)**
- `ageLimit`: **2YO**
- `raceClass`: **4**
- `ratingBand`: blank
- `raceCriteriaRaceType`: **FLAT**
- `distanceText`: **5f 7y**
- `prizeAmount`: **£11,500**

This is useful because the structured fields independently represent several concepts that are also embedded in the human-readable race title.

For example:

> `MAIDEN STAKES (CLASS 4)`

contains information about race type and class in prose, while the structured record separately provides `raceClass = 4`.

That gives us an opportunity to validate not merely values but **semantic interpretation**.

## Comparison rule

For each material race concept, Study 05 will now ask:

1. **What does the BHA say the concept means?**
2. **Which BHA structured field represents it?**
3. **Which Database v4 field claims to represent the same concept?**
4. **Are the two genuinely semantically equivalent?**
5. **Where they are equivalent, do the values agree?**
6. **Where information exists only in the BHA data, is that absence important to Inside Rails?**

Similarity of field names is not sufficient evidence of equivalence.

## Next bounded question

> **What race-level information does Database v4 currently expose for these same concepts?**

The next step is therefore a schema-level inspection of the governed race view before comparing any values.

## Inherited Inside Rails race-field semantics

Database v4's race-level fields have already been investigated and governed by the earlier source-field studies.

Study 05 therefore **does not rediscover their schema or infer their meaning from column names**.

Relevant inherited work includes:

- `docs/STUDY_DATABASE_REFERENCE.md`
- `docs/DATABASE_USER_GUIDE.md`
- `data/reference/source_field_governance.csv`
- `reports/notebook_16_race_classification_and_eligibility.md`
- the individual field-integration documents where applicable.

The important question for Study 05 is now different:

> **Does the existing Inside Rails representation correspond to the official BHA representation of the same British race concepts?**

### Existing race-level source concepts

Source Version 1 contains race-level fields including:

- `race_name`
- `type`
- `class`
- `pattern`
- `rating_band`
- `age_band`
- `sex_rest`
- `dist`
- `going`
- `ran`

Earlier governance established that these fields are internally race-level within the authorised Inside Rails race occurrence.

However, their presence does **not** establish that they are complete or authoritative representations of British race conditions.

The BHA evidence discovered subsequently gives us an authoritative British source against which those representations can now be tested.

---

### Race classification and eligibility

Notebook 16 established that:

- `class`, `pattern` and `rating_band` describe different properties;
- they do not form one universal classification hierarchy;
- `rating_band` cannot substitute for class;
- `age_band` has stable source syntax but cannot universally reconstruct official eligibility;
- `sex_rest` is shorthand rather than a complete official sex condition;
- race-name text can contain useful condition information but can also contain sponsorship and descriptive material.

This limitation was important because Source Version 1 does not contain the complete governing-authority race conditions in a standard structured form.

The BHA structured data now gives us an opportunity to compare those source representations with the official British record.

---

### Race type and racing discipline

Study 02 established the British sporting structure as:

- **Flat racing**
- **Jump racing**
  - Hurdle
  - Chase / Steeple Chase
  - National Hunt Flat

Inside Rails currently represents individual races using the more specific observed categories:

- `Flat`
- `Hurdle`
- `Chase`
- `NH Flat`

The BHA race-list records inspected during the 27 May 2026 pilot exposed a field named:

`raceCriteriaRaceType`

Observed values included:

- `FLAT`
- `JUMP`

At face value, this BHA field operates at a broader level than the existing Inside Rails four-category race-type field.

However, **this does not establish that the BHA structured system lacks a more specific Hurdle / Chase / National Hunt Flat classification**.

The BHA itself distinguishes those forms of Jump racing in its official programme and regulatory material.

In addition, the structured BHA service exposes a separate individual race-detail endpoint of the form:

`/bha/v1/races/{yearOfRace}/{raceId}/{divisionSequence}`

That detailed race representation has not yet been inspected in Study 05 for the purpose of identifying the most specific official race-discipline field.

Therefore:

> **No formal BHA ↔ Inside Rails race-type mapping is made yet.**

In particular, Study 05 must not currently assume that:

`Hurdle / Chase / NH Flat → JUMP`

is the deepest comparison available from BHA structured data.

The next structured-data check should inspect representative BHA detailed race records for:

- one Flat race;
- one Hurdle race;
- one Chase race;
- one National Hunt Flat race;

and determine whether the BHA supplies a more specific race type, discipline, obstacle type or equivalent classification.

Only after that evidence is inspected should the semantic relationship between BHA race type and the governed Inside Rails race-type field be specified.

---

### Distance

Inside Rails preserves source distance information and governed parsing separately from independently verified official-distance enrichments.

The BHA structured race-list record supplies its own official distance representation, including fields observed in the pilot such as:

- `rawDistanceText`
- `distanceValue`
- `distanceText`
- `distanceChange`
- `distanceChangeText`

These can therefore support an authoritative comparison.

The comparison must nevertheless preserve the distinction between:

> **source-reported distance**

and:

> **BHA official race distance**

Agreement should be treated as validation rather than as evidence that the third-party source itself was authoritative.

---

### Prize information

A particularly important non-equivalence already exists.

Source Version 1 `prize` is a **runner-level field**.

The BHA race record's `prizeAmount` is a **race-level prize-fund property**.

These are not automatically the same monetary concept and must not be compared merely because both contain monetary amounts.

Study 05 therefore does not include prize money in the first direct field comparison.

Prize semantics can be investigated separately if required.

---

### Race identity and context

The BHA structured records also expose identifiers and contextual properties such as:

- `raceId`
- `yearOfRace`
- `divisionSequence`
- `raceDate`
- `raceTime`
- parent fixture identifiers.

These are useful for authoritative source retrieval and reconciliation.

They are **not automatically adopted as Inside Rails identity rules**.

Study 04 already established that convenient combinations of date, racecourse and time must not be promoted into sporting or administrative identities without evidence.

Similarly, the existence of a BHA `raceId` does not by itself determine how Inside Rails should model persistent race identity across time or across different runnings of a named race.

For Study 05 these fields are primarily authoritative source identifiers and matching/context evidence.

---

## Revised comparison objective

We are therefore not asking:

> Does Database v4 contain all of the BHA's fields?

We are asking:

> **For each important part of the official British race specification, what does the BHA represent, what does Inside Rails represent, and how strong is the equivalence between them?**

Possible outcomes are:

1. **direct equivalent** — both represent the same concept at the same grain;
2. **transformable equivalent** — the same concept is represented at different granularity or vocabulary and an explicit mapping is justified;
3. **partial representation** — Inside Rails captures some but not all of the official concept;
4. **different concept** — superficially similar fields should not be compared;
5. **BHA-only information** — the official source contains a concept not represented in Database v4;
6. **Inside-Rails-only analytical representation** — a useful project construct that is not itself an official BHA field;
7. **unresolved equivalence** — more authoritative evidence is required before the relationship can be stated.

Similarity of field names is not sufficient evidence of semantic equivalence.

---

## Next bounded question

Before comparing race-type values across the 34 reconciled races:

> **Does the BHA detailed race record expose a more specific official racing-discipline classification than the `FLAT` / `JUMP` value observed in the race-list record?**

Inspect one representative detailed BHA record from each existing Inside Rails category:

- Flat;
- Hurdle;
- Chase;
- NH Flat.

If a deeper BHA classification exists, establish its meaning before constructing the comparison.

If it does not, record that limitation explicitly and only then determine the correct relationship between the BHA broad form and the more detailed Inside Rails classification.

In [2]:
# Inspect one detailed BHA Jump-race record for a more specific discipline field.
#
# WHY THIS CELL EXISTS
# --------------------
# The BHA race-list endpoint exposed raceCriteriaRaceType='JUMP', but that does
# not prove that the more detailed race object lacks a Hurdle / Chase / NH Flat
# classification.
#
# Notebook 26 already reconciled this official BHA race:
#
#   Newton Abbot — 27 May 2026 — 14:53
#   raceId=23016
#   yearOfRace=2026
#   divisionSequence=0
#
# Its published race name identifies it as a Hurdle race.
#
# This cell reuses the proven BHA acquisition method but prints only
# classification-relevant fields. The public frontend Authorization value is
# kept in memory and is never displayed or written to disk.

import json
import re
from urllib.request import Request, urlopen


BHA_APP_JS = (
    "https://www.britishhorseracing.com/"
    "wp-content/themes/bha/library/js/angular/app.js"
)

BHA_RACE_DETAIL_URL = (
    "https://api09.horseracing.software/"
    "bha/v1/races/2026/23016/0"
)

USER_AGENT = (
    "Mozilla/5.0 (X11; Linux x86_64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/131.0 Safari/537.36"
)


# Retrieve the current public BHA frontend configuration.
app_request = Request(
    BHA_APP_JS,
    headers={"User-Agent": USER_AGENT},
)

with urlopen(app_request, timeout=30) as response:
    app_js_text = response.read().decode("utf-8", errors="replace")


# Find the single active Angular Authorization assignment.
# Ignore commented-out historical assignments.
authorization_pattern = re.compile(
    r"""
    \$httpProvider
    \.defaults
    \.headers
    \.common
    \[['"]Authorization['"]\]
    \s*=\s*
    ['"]([^'"]+)['"]
    """,
    re.VERBOSE,
)

authorization_matches = []

for line in app_js_text.splitlines():
    stripped = line.lstrip()

    if stripped.startswith("//"):
        continue

    match = authorization_pattern.search(line)

    if match:
        authorization_matches.append(match.group(1))

assert len(authorization_matches) == 1, (
    "Expected exactly one active BHA Authorization assignment; "
    f"found {len(authorization_matches)}."
)

authorization_value = authorization_matches[0]


# Retrieve the detailed official BHA race object.
detail_request = Request(
    BHA_RACE_DETAIL_URL,
    headers={
        "Authorization": authorization_value,
        "Accept": "application/json",
        "Origin": "https://www.britishhorseracing.com",
        "Referer": "https://www.britishhorseracing.com/",
        "User-Agent": USER_AGENT,
    },
)

with urlopen(detail_request, timeout=30) as response:
    detail_payload = json.loads(
        response.read().decode("utf-8")
    )


# Search recursively rather than assuming the response structure.
# Show scalar fields whose paths could plausibly identify race discipline/type.
classification_terms = (
    "race",
    "type",
    "criteria",
    "discipline",
    "obstacle",
    "hurdle",
    "chase",
    "jump",
    "flat",
    "code",
    "category",
    "description",
)


def classification_fields(value, path=""):
    rows = []

    if isinstance(value, dict):
        for key, child in value.items():
            child_path = f"{path}.{key}" if path else key
            rows.extend(classification_fields(child, child_path))

    elif isinstance(value, list):
        for index, child in enumerate(value):
            child_path = f"{path}[{index}]"
            rows.extend(classification_fields(child, child_path))

    else:
        path_lower = path.lower()

        if any(term in path_lower for term in classification_terms):
            rows.append((path, value))

    return rows


fields = classification_fields(detail_payload)

print("BHA detailed race inspection")
print("----------------------------")
print("Race: Newton Abbot, 27 May 2026, 14:53")
print("Known source/BHA description: Hurdle")
print(f"HTTP payload type: {type(detail_payload).__name__}")
print()

for path, value in fields:
    print(f"{path}: {value!r}")

BHA detailed race inspection
----------------------------
Race: Newton Abbot, 27 May 2026, 14:53
Known source/BHA description: Hurdle
HTTP payload type: dict

data[0].raceId: 23016
data[0].yearOfRace: '2026'
data[0].raceNumber: '32833'
data[0].raceDate: '2026-05-27'
data[0].raceTime: '14:53:00'
data[0].raceName: "THE STOCK EXE BUILDING SUPPLIES MARES' 'NATIONAL HUNT' NOVICES' HURDLE RACE (CLASS 4) (GBB RACE)"
data[0].currentStageCode: 99
data[0].riderType: None
data[0].animalType: 'NOVICE'
data[0].atTheRaces: 1
data[0].racecardAvailable: 1
data[0].raceCriteriaRaceType: 'JUMP'
data[0].raceCriteriaMinimumWeight: 0
data[0].raceCriteriaWeightsRaised: 0
data[0].blackTypeRace: 0
data[0].abandonedReasonCode: 0
data[0].pastWinners[0].raceId: 23016
data[0].pastWinners[0].yearOfRace: '2025'
data[0].pastWinners[0].raceDate: '2025-05-28'
data[0].pastWinners[1].raceId: 23016
data[0].pastWinners[1].yearOfRace: '2024'
data[0].pastWinners[1].raceDate: '2024-05-29'
data[0].pastWinners[2].raceId: 23016


## Evidence note 8 — the detailed BHA race object reveals two different identity/type layers

We inspected the detailed BHA record for:

- Newton Abbot;
- 27 May 2026;
- 14:53;
- BHA `raceId = 23016`;
- `"THE STOCK EXE BUILDING SUPPLIES MARES' 'NATIONAL HUNT' NOVICES' HURDLE RACE (CLASS 4) (GBB RACE)"`.

This was already reconciled to an Inside Rails Hurdle race in the earlier BHA pilot.

### 1. The detailed record still exposes `JUMP` as the broad structured race type

The detailed BHA object contains:

`raceCriteriaRaceType = 'JUMP'`

It does not expose an obvious separate scalar field containing:

- `HURDLE`;
- `CHASE`;
- `NH FLAT`;

for this race.

The more specific form is instead explicit in the published race name:

`... NOVICES' HURDLE RACE ...`

The record also contains:

`animalType = 'NOVICE'`

but this is clearly not equivalent to Hurdle/Chase/NH Flat, because `NOVICE` describes another dimension of the race.

Therefore the evidence currently supports:

> **BHA `raceCriteriaRaceType` represents the broad Flat/Jump form, while the more specific Hurdle classification is not represented by that field.**

We have not yet established whether another BHA endpoint or structure supplies a dedicated Hurdle/Chase/NH Flat field.

A Chase example should be inspected before generalising this finding across Jump racing.

---

### 2. BHA `raceId` is not an individual race-occurrence identifier

The same detailed record contains a `pastWinners` structure.

For this race, the BHA returned:

| `raceId` | `yearOfRace` | date |
|---|---:|---|
| 23016 | 2026 | 2026-05-27 |
| 23016 | 2025 | 2025-05-28 |
| 23016 | 2024 | 2024-05-29 |
| 23016 | 2023 | 2023-05-31 |
| 23016 | 2022 | 2022-05-25 |
| 23016 | 2019 | 2019-05-29 |

This is strong evidence that:

> **BHA `raceId` does not uniquely identify one historical running of a race.**

The identifier persists across multiple annual runnings while `yearOfRace` and `raceDate` change.

Therefore a BHA race occurrence cannot be identified by `raceId` alone.

At minimum, the BHA system itself uses additional context such as:

- `yearOfRace`;
- and, where applicable, `divisionSequence`.

This is an important distinction from the current Inside Rails `source_race_occurrence_code`, which deliberately represents one particular observed race occurrence.

### Important qualification

We should **not yet assign a stronger semantic meaning to `raceId`**.

The evidence shows that it persists across annual runnings.

It does not yet establish whether the BHA formally means:

- named race identity;
- programme race identity;
- recurring race-series identity;
- race-condition template;
- or another administrative object.

That question requires further evidence.

---

## Consequence for the working model

Study 05 now needs to distinguish at least:

> **recurring/programmed race identity or lineage**  
> ↓  
> **one particular running / race occurrence**  
> ↓  
> **the specification applying to that running**  
> ↓  
> **participants**  
> ↓  
> **result**

This strengthens the earlier distinction between:

> **race identity**

and:

> **race specification**

because the BHA itself preserves one `raceId` while individual annual runnings occur on different dates.

## Next bounded question

Before interpreting `raceId` further:

> **Does a BHA Chase race show the same broad `JUMP` structured type while carrying its more specific Chase/Steeple Chase form elsewhere?**

If so, we will have stronger evidence that BHA `raceCriteriaRaceType` genuinely operates at the Flat/Jump level rather than the Hurdle/Chase/NH Flat level.

In [3]:
# Test whether a known BHA Chase race is also represented only as broad JUMP
# in the detailed structured record.
#
# Known reconciled race from the 27 May 2026 pilot:
#
#   Cartmel — 20:38
#   BHA raceId=1462
#   Inside Rails: Chase
#   BHA race name: ... HANDICAP STEEPLE CHASE ...
#
# Reuse the current in-memory Authorization value from the previous cell.

CHASE_DETAIL_URL = (
    "https://api09.horseracing.software/"
    "bha/v1/races/2026/1462/0"
)

chase_request = Request(
    CHASE_DETAIL_URL,
    headers={
        "Authorization": authorization_value,
        "Accept": "application/json",
        "Origin": "https://www.britishhorseracing.com",
        "Referer": "https://www.britishhorseracing.com/",
        "User-Agent": USER_AGENT,
    },
)

with urlopen(chase_request, timeout=30) as response:
    chase_payload = json.loads(
        response.read().decode("utf-8")
    )

chase_fields = classification_fields(chase_payload)

print("BHA detailed Chase inspection")
print("-----------------------------")
print("Race: Cartmel, 27 May 2026, 20:38")
print("Known Inside Rails classification: Chase")
print()

for path, value in chase_fields:
    print(f"{path}: {value!r}")

BHA detailed Chase inspection
-----------------------------
Race: Cartmel, 27 May 2026, 20:38
Known Inside Rails classification: Chase

data[0].raceId: 1462
data[0].yearOfRace: '2026'
data[0].raceNumber: '32823'
data[0].raceDate: '2026-05-27'
data[0].raceTime: '20:38:00'
data[0].raceName: "THE JJ CROSSFIELD'S SUNDOWNER HANDICAP STEEPLE CHASE (CLASS 5)"
data[0].currentStageCode: 99
data[0].riderType: None
data[0].animalType: 'NOT_APPLICABLE'
data[0].atTheRaces: 0
data[0].racecardAvailable: 1
data[0].raceCriteriaRaceType: 'JUMP'
data[0].raceCriteriaMinimumWeight: 142
data[0].raceCriteriaWeightsRaised: 5
data[0].blackTypeRace: 0
data[0].abandonedReasonCode: 0
data[0].pastWinners[0].raceId: 1462
data[0].pastWinners[0].yearOfRace: '2025'
data[0].pastWinners[0].raceDate: '2025-05-28'
data[0].pastWinners[1].raceId: 1462
data[0].pastWinners[1].yearOfRace: '2024'
data[0].pastWinners[1].raceDate: '2024-05-29'
data[0].pastWinners[2].raceId: 1462
data[0].pastWinners[2].yearOfRace: '2023'
data[0].p

In [4]:
# Find recent candidate NH Flat races for BHA inspection.

with connect_read_only(DATABASE) as connection:
    recent_nh_flat_candidates = pd.read_sql_query(
        """
        SELECT
            source_race_occurrence_code,
            raw_date AS race_date,
            governed_racecourse_name,
            advertised_start_course_local,
            race_name_raw,
            race_type_raw
        FROM view_gb_reconciled_race_occurrences_with_racecourse
        WHERE
            candidate_jurisdiction = 'Great Britain'
            AND race_type_raw = 'NH Flat'
            AND advertised_start_course_local IS NOT NULL
        ORDER BY
            raw_date DESC,
            advertised_start_course_local DESC
        LIMIT 10
        """,
        connection,
    )

recent_nh_flat_candidates

,source_race_occurrence_code,race_date,governed_racecourse_name,advertised_start_course_local,race_name_raw,race_type_raw
0,race:77b5dbbbfdee69d4d92a5826:000188980,2026-05-26,Plumpton,2026-05-26T20:39:00+01:00,Ladies Day At Season Opener Open National Hunt...,NH Flat
1,race:77b5dbbbfdee69d4d92a5826:000188855,2026-05-24,Kelso,2026-05-24T17:57:00+01:00,See You In September Open National Hunt Flat R...,NH Flat
2,race:77b5dbbbfdee69d4d92a5826:000188884,2026-05-24,Uttoxeter,2026-05-24T17:47:00+01:00,Wrights 100 Year Dash Open National Hunt Flat ...,NH Flat
3,race:77b5dbbbfdee69d4d92a5826:000188846,2026-05-24,Fontwell Park,2026-05-24T17:30:00+01:00,Land & Power Mares Open Maiden National Hunt F...,NH Flat
4,race:77b5dbbbfdee69d4d92a5826:000188775,2026-05-23,Bangor-on-Dee,2026-05-23T17:20:00+01:00,Wrexham Evening On Friday 26th June Mares Open...,NH Flat
5,race:77b5dbbbfdee69d4d92a5826:000188772,2026-05-22,Worcester,2026-05-22T18:06:00+01:00,CopyBet Overnight Best Odds Guaranteed Open Na...,NH Flat
6,race:77b5dbbbfdee69d4d92a5826:000188522,2026-05-17,Stratford-on-Avon,2026-05-17T17:35:00+01:00,Family Day On Sunday 12th July Open Maiden Nat...,NH Flat
7,race:77b5dbbbfdee69d4d92a5826:000188335,2026-05-15,Aintree,2026-05-15T20:40:00+01:00,Point To Point Bumper National Hunt Flat Race ...,NH Flat
8,race:77b5dbbbfdee69d4d92a5826:000188302,2026-05-14,Fontwell Park,2026-05-14T20:10:00+01:00,Best Of British Events Open National Hunt Flat...,NH Flat
9,race:77b5dbbbfdee69d4d92a5826:000188317,2026-05-14,Perth,2026-05-14T17:28:00+01:00,Edinburgh Gin Open National Hunt Flat Race (Ca...,NH Flat


In [5]:
# Find the official BHA race record corresponding to the
# Plumpton NH Flat candidate on 26 May 2026 at 20:39.
#
# The lookup happens in two stages:
#   1. fetch the BHA fixture for Plumpton on the target date;
#   2. fetch that fixture's race list and select the 20:39 race.
#
# The resulting race record gives us the BHA identifiers needed
# to inspect the full detailed race object in the next cell.

from urllib.parse import urlencode


# ---------------------------------------------------------------------------
# Fetch all result-bearing BHA fixtures for the target date.
# ---------------------------------------------------------------------------

target_date = "2026-05-26"

fixture_params = {
    "fromdate": target_date,
    "todate": target_date,
    "resultsAvailable": 1,
    "order": "asc",
    "page": 1,
    "per_page": 100,
}

fixture_url = (
    "https://api09.horseracing.software/bha/v1/fixtures/?"
    + urlencode(fixture_params)
)

fixture_request = Request(
    fixture_url,
    headers={
        "Authorization": authorization_value,
        "Accept": "application/json",
        "Origin": "https://www.britishhorseracing.com",
        "Referer": "https://www.britishhorseracing.com/",
        "User-Agent": USER_AGENT,
    },
)

with urlopen(fixture_request, timeout=30) as response:
    fixture_payload = json.loads(
        response.read().decode("utf-8")
    )


# ---------------------------------------------------------------------------
# Identify the Plumpton fixture for that date.
# ---------------------------------------------------------------------------

plumpton_fixtures = [
    fixture
    for fixture in fixture_payload["data"]
    if fixture.get("courseName") == "Plumpton"
]

assert len(plumpton_fixtures) == 1, (
    f"Expected one Plumpton fixture, found {len(plumpton_fixtures)}"
)

plumpton_fixture = plumpton_fixtures[0]


# ---------------------------------------------------------------------------
# Fetch the individual races belonging to that BHA fixture.
# ---------------------------------------------------------------------------

race_list_url = (
    "https://api09.horseracing.software/bha/v1/fixtures/"
    f"{plumpton_fixture['fixtureYear']}/"
    f"{plumpton_fixture['fixtureId']}/races"
)

race_list_request = Request(
    race_list_url,
    headers={
        "Authorization": authorization_value,
        "Accept": "application/json",
        "Origin": "https://www.britishhorseracing.com",
        "Referer": "https://www.britishhorseracing.com/",
        "User-Agent": USER_AGENT,
    },
)

with urlopen(race_list_request, timeout=30) as response:
    plumpton_race_payload = json.loads(
        response.read().decode("utf-8")
    )

plumpton_races = plumpton_race_payload["data"]


# ---------------------------------------------------------------------------
# Select the race scheduled for 20:39.
#
# This should correspond to the Inside Rails candidate:
# "Ladies Day At Season Opener Open National Hunt Flat Race ..."
# ---------------------------------------------------------------------------

nh_flat_candidates = [
    race
    for race in plumpton_races
    if race.get("raceTime") == "20:39:00"
]

assert len(nh_flat_candidates) == 1, (
    f"Expected one Plumpton race at 20:39, "
    f"found {len(nh_flat_candidates)}"
)

nh_flat_bha_race = nh_flat_candidates[0]


# Display the complete race-list record so we can verify the match
# and obtain its BHA identifiers before requesting the detailed object.

for key, value in nh_flat_bha_race.items():
    print(f"{key}: {value!r}")

raceId: 68080
yearOfRace: '2026'
divisionSequence: 0
raceDate: '2026-05-26'
raceTime: '20:39:00'
raceName: 'THE LADIES DAY AT SEASON OPENER OPEN NATIONAL HUNT FLAT RACE (CLASS 5) (Category 1 Elimination) (GBB RACE)'
ageLimit: '4-5YO'
prizeAmount: 4999
prizeCurrency: 'GBP'
raceClass: 5
ratingBand: ''
rawDistanceText: ' TWO MILES ABOUT ONE AND A HALF FURLONGS (2m 1f 169yds)'
distanceUnits: None
distanceValue: 3909
distanceText: '2m 1f 169y'
distanceChange: -29
timingType: None
goingText: 'Good, Good to Soft in places'
currentStageCode: 99
transparentWindowStatus: 'DECLARATIONS_OPEN'
raceCriteriaRaceType: 'JUMP'
abandonedReasonCode: 0
blackTypeRace: 0
distanceChangeText: '2m 1f 140y'
plus10: False
winnersDetails: [{'position': 1, 'jockeyName': 'Jonathan Burke', 'trainerName': 'James Owen', 'silkImage': 'https://draper.horseracing.software/silks?id=051816071618091618&desc=YELLOW, ROYAL BLUE braces, ROYAL BLUE sleeves, YELLOW stars and stars on cap.', 'racehorseName': 'Scotto (IRE)'}]
aroRa

In [6]:
# Inspect the detailed BHA record for the Plumpton NH Flat race.
#
# We already know from the fixture race list that:
#   raceId = 68080
#   yearOfRace = 2026
#   divisionSequence = 0
#   raceCriteriaRaceType = 'JUMP'
#
# This check asks whether the more detailed race object contains any
# additional structured field that identifies the race specifically as
# National Hunt Flat rather than only as broad JUMP.

NH_FLAT_DETAIL_URL = (
    "https://api09.horseracing.software/"
    "bha/v1/races/2026/68080/0"
)

nh_flat_detail_request = Request(
    NH_FLAT_DETAIL_URL,
    headers={
        "Authorization": authorization_value,
        "Accept": "application/json",
        "Origin": "https://www.britishhorseracing.com",
        "Referer": "https://www.britishhorseracing.com/",
        "User-Agent": USER_AGENT,
    },
)

with urlopen(nh_flat_detail_request, timeout=30) as response:
    nh_flat_detail_payload = json.loads(
        response.read().decode("utf-8")
    )


# Reuse the recursive inspection helper from the earlier Hurdle check.
# It displays classification-related scalar fields without assuming
# in advance where the BHA may store the more specific race form.

nh_flat_classification_fields = classification_fields(
    nh_flat_detail_payload
)

print("BHA detailed NH Flat inspection")
print("-------------------------------")
print("Race: Plumpton, 26 May 2026, 20:39")
print("BHA race-list type: JUMP")
print()

for path, value in nh_flat_classification_fields:
    print(f"{path}: {value!r}")

BHA detailed NH Flat inspection
-------------------------------
Race: Plumpton, 26 May 2026, 20:39
BHA race-list type: JUMP

data[0].raceId: 68080
data[0].yearOfRace: '2026'
data[0].raceNumber: '36057'
data[0].raceDate: '2026-05-26'
data[0].raceTime: '20:39:00'
data[0].raceName: 'THE LADIES DAY AT SEASON OPENER OPEN NATIONAL HUNT FLAT RACE (CLASS 5) (Category 1 Elimination) (GBB RACE)'
data[0].currentStageCode: 99
data[0].riderType: None
data[0].animalType: 'NOT_APPLICABLE'
data[0].atTheRaces: 1
data[0].racecardAvailable: 1
data[0].raceCriteriaRaceType: 'JUMP'
data[0].raceCriteriaMinimumWeight: 0
data[0].raceCriteriaWeightsRaised: 0
data[0].blackTypeRace: 0
data[0].abandonedReasonCode: 0


## Evidence note 10 — BHA structured race type closes the Flat/Jump comparison

A representative detailed BHA record has now been inspected for each of the
three principal forms of Jump racing represented in Inside Rails:

| Inside Rails form | Official BHA race | BHA `raceCriteriaRaceType` |
|---|---|---|
| Hurdle | Newton Abbot, 27 May 2026, 14:53 | `JUMP` |
| Chase | Cartmel, 27 May 2026, 20:38 | `JUMP` |
| NH Flat | Plumpton, 26 May 2026, 20:39 | `JUMP` |

For all three races, the specific form is explicit in the official BHA
`raceName`:

- `... HURDLE RACE ...`
- `... STEEPLE CHASE ...`
- `... NATIONAL HUNT FLAT RACE ...`

Neither the race-list record nor the detailed race object inspected exposes an
obvious dedicated scalar field corresponding to the Inside Rails
`Hurdle` / `Chase` / `NH Flat` distinction.

The structured BHA field:

`raceCriteriaRaceType`

therefore operates, on the evidence inspected, at the broader level:

- `FLAT`
- `JUMP`

This gives the following current semantic relationship:

> **BHA broad racing form:** Flat / Jump

> **Inside Rails structured race form:** Flat / Hurdle / Chase / NH Flat

The Inside Rails field therefore represents a finer subdivision of Jump racing
than `raceCriteriaRaceType`.

This does not make the Inside Rails field authoritative merely because it is
more detailed. Its Hurdle, Chase and NH Flat values must still correspond to
official BHA evidence for the individual race.

For the examples inspected, that official subtype evidence is present in the
BHA race name.

### Cross-study implication

This finding should be incorporated into the later editorial pass over
Great Britain Study 02 — Types of British Racing.

Study 02's sporting conclusion remains sound:

- Flat racing;
- Jump racing, including Hurdle, Chase and National Hunt Flat racing.

The new BHA structured-data evidence adds an important representation detail:

> the BHA API field `raceCriteriaRaceType` encodes the top-level Flat/Jump
> distinction, while the inspected structured race objects do not provide an
> equivalent dedicated Hurdle/Chase/NH Flat subtype field.

Study 05 retains this as authoritative structured-data evidence; Study 02 can
reference it when its notebook/report is edited.

## BHA concept — race class

Before comparing `raceClass` with the Inside Rails `class` fields, Study 05
must establish what race class represents in British racing.

### BHA race-programme role

The BHA Racing Department is responsible for compiling race programmes and
designing opportunities for horses across different levels of ability and
different kinds of racing.

Within that programme, **Class** is used as a classification of the level of
an individual race.

It is one property of the programmed contest rather than a complete description
of the race.

A race can therefore simultaneously have:

- a broad racing form;
- a specific race type;
- a Class;
- an eligibility or rating band;
- age and sex conditions;
- distance;
- weight conditions;
- Pattern / Listed / Grade status where applicable;
- and other race conditions.

These properties must not be collapsed into one another.

### Class is not the same concept as handicap rating or rating band

BHA programme material demonstrates that Class and rating eligibility are
related in some parts of the programme but are not identical concepts.

For example, the BHA's 2026 two-year-old Nursery Handicap programme specifies:

| Rating range | Class |
|---|---:|
| 0–65 | 6 |
| 0–75 | 5 |
| 0–85 | 4 |
| 0–95 | 3 |
| Open | 2 |

This shows that a Class can form part of the structure used to organise races
for horses of different ability.

However, the relationship is programme-specific rather than a universal rule
that allows Class to be derived from a rating band.

The distinction is reinforced by historical BHA programme changes.

The BHA has, for example, reclassified 0–105 Jump handicaps from Class 4 to
Class 5 while separately changing the minimum Class at which weight-for-age
Maiden, Novice and Juvenile races could be programmed.

A race's Class is therefore a **BHA race-programme classification**, not merely
another name for its rating restriction.

### Working Study 05 interpretation

For this study:

> **Race Class is a prospective classification assigned to an individual race
> within the British race programme, representing its programme level and
> operating alongside — rather than replacing — the race's type, eligibility
> conditions, rating restrictions and other conditions.**

The exact consequences of a particular Class can depend on the relevant race
type and programme rules.

Study 05 therefore does not assume that:

`Class 5 = one fixed rating range`

or that:

`Class = race quality expressed numerically`

without additional BHA evidence.

### Structured BHA representation

The BHA structured race record represents this concept through:

`raceClass`

For the Plumpton National Hunt Flat race on 26 May 2026 at 20:39:

`raceClass = 5`

The next question is therefore:

> **Does Database v4 represent the same BHA race-class concept and value for
> this individual race?**

In [7]:
# Compare the BHA race-class value with Database v4 for the same race.
#
# The BHA structured record supplies `raceClass`.
# Database v4 preserves the source class text and separately stores the
# governed parsed class number.
#
# This tests whether Inside Rails represents the same race-class concept and
# value for this individual race.

plumpton_source_race_code = (
    "race:77b5dbbbfdee69d4d92a5826:000188980"
)

with connect_read_only(DATABASE) as connection:
    plumpton_v4_class = pd.read_sql_query(
        """
        SELECT
            source_race_occurrence_code,
            raw_date AS race_date,
            governed_racecourse_name,
            advertised_start_course_local,
            race_name_raw,
            class_raw,
            class_number,
            class_parse_status
        FROM view_gb_reconciled_race_occurrences_with_racecourse
        WHERE source_race_occurrence_code = ?
        """,
        connection,
        params=[plumpton_source_race_code],
    )

assert len(plumpton_v4_class) == 1, (
    f"Expected one Inside Rails race, found {len(plumpton_v4_class)}"
)

bha_class = nh_flat_bha_race["raceClass"]

class_comparison = pd.DataFrame(
    [
        {
            "BHA_raceClass": bha_class,
            "Inside_Rails_class_raw": plumpton_v4_class.loc[0, "class_raw"],
            "Inside_Rails_class_number": plumpton_v4_class.loc[0, "class_number"],
            "Inside_Rails_parse_status": plumpton_v4_class.loc[0, "class_parse_status"],
            "numeric_agreement": (
                bha_class
                == plumpton_v4_class.loc[0, "class_number"]
            ),
        }
    ]
)

class_comparison

,BHA_raceClass,Inside_Rails_class_raw,Inside_Rails_class_number,Inside_Rails_parse_status,numeric_agreement
0,5,Class 5,5,canonical,True


## Evidence note 11 — race class is directly represented for the inspected race

The Plumpton National Hunt Flat race on 26 May 2026 at 20:39 provides a clean
comparison of BHA race class with Database v4.

The BHA structured record reports:

`raceClass = 5`

Database v4 reports:

- `class_raw = "Class 5"`
- `class_number = 5`
- `class_parse_status = "canonical"`

The governed numeric value therefore agrees exactly with the BHA value.

### Interpretation

For this inspected race, the Inside Rails `class` representation is a:

> **direct semantic equivalent**

of the BHA race-class concept.

The only representational difference is formatting:

- BHA stores the class as the integer `5`;
- Source Version 1 stores the text `"Class 5"`;
- Database v4 parses that canonical source form to `class_number = 5`.

This is therefore not merely a textual resemblance.

The governed Inside Rails parser has extracted the same prospective race-class
value represented by the official BHA record.

### Current conclusion

For this race:

> `BHA raceClass 5`  
> =  
> `Inside Rails class_number 5`

with the original source value `"Class 5"` preserved separately.

This establishes one positive equivalence case but does not by itself establish
population-wide agreement.

## Next bounded question

> **How consistently does Database v4 reproduce BHA `raceClass` across the
> already reconciled 34-race BHA pilot?**

Because race class is now conceptually understood and one exact equivalence has
been demonstrated, the next useful step is to test the field across the bounded
34-race official BHA sample rather than inspecting additional single races.

In [8]:
# Compare BHA race class with Database v4 across all 34 races in the
# previously reconciled 27 May 2026 pilot.
#
# This cell:
#   1. retrieves the five result-bearing BHA fixtures for the pilot date;
#   2. retrieves every race belonging to those fixtures;
#   3. loads the corresponding 34 Database v4 race occurrences;
#   4. reconciles them by date, governed racecourse and advertised start time;
#   5. compares BHA `raceClass` with governed `class_number`.

pilot_date = "2026-05-27"


# ---------------------------------------------------------------------------
# Retrieve the BHA fixtures for the pilot date.
# ---------------------------------------------------------------------------

fixture_params = {
    "fromdate": pilot_date,
    "todate": pilot_date,
    "resultsAvailable": 1,
    "order": "asc",
    "page": 1,
    "per_page": 100,
}

fixture_url = (
    "https://api09.horseracing.software/bha/v1/fixtures/?"
    + urlencode(fixture_params)
)

fixture_request = Request(
    fixture_url,
    headers={
        "Authorization": authorization_value,
        "Accept": "application/json",
        "Origin": "https://www.britishhorseracing.com",
        "Referer": "https://www.britishhorseracing.com/",
        "User-Agent": USER_AGENT,
    },
)

with urlopen(fixture_request, timeout=30) as response:
    fixture_payload = json.loads(
        response.read().decode("utf-8")
    )

bha_fixtures = fixture_payload["data"]


# ---------------------------------------------------------------------------
# Retrieve every BHA race belonging to those fixtures.
# ---------------------------------------------------------------------------

bha_race_rows = []

for fixture in bha_fixtures:
    race_list_url = (
        "https://api09.horseracing.software/bha/v1/fixtures/"
        f"{fixture['fixtureYear']}/"
        f"{fixture['fixtureId']}/races"
    )

    race_list_request = Request(
        race_list_url,
        headers={
            "Authorization": authorization_value,
            "Accept": "application/json",
            "Origin": "https://www.britishhorseracing.com",
            "Referer": "https://www.britishhorseracing.com/",
            "User-Agent": USER_AGENT,
        },
    )

    with urlopen(race_list_request, timeout=30) as response:
        race_payload = json.loads(
            response.read().decode("utf-8")
        )

    for race in race_payload["data"]:
        bha_race_rows.append(
            {
                "race_date": race["raceDate"],
                "governed_racecourse_name": fixture["courseName"],
                "advertised_time": race["raceTime"],
                "bha_race_id": race["raceId"],
                "bha_race_name": race["raceName"],
                "bha_race_class": race["raceClass"],
            }
        )

bha_classes = pd.DataFrame(bha_race_rows)

assert len(bha_classes) == 34, (
    f"Expected 34 BHA races, found {len(bha_classes)}"
)


# ---------------------------------------------------------------------------
# Load the corresponding Database v4 race records.
#
# The local advertised time is extracted from the governed ISO timestamp
# so that it can be compared directly with BHA `raceTime`.
# ---------------------------------------------------------------------------

with connect_read_only(DATABASE) as connection:
    v4_classes = pd.read_sql_query(
        """
        SELECT
            source_race_occurrence_code,
            raw_date AS race_date,
            governed_racecourse_name,
            substr(advertised_start_course_local, 12, 8) AS advertised_time,
            race_name_raw,
            class_raw,
            class_number,
            class_parse_status
        FROM view_gb_reconciled_race_occurrences_with_racecourse
        WHERE raw_date = ?
        """,
        connection,
        params=[pilot_date],
    )

assert len(v4_classes) == 34, (
    f"Expected 34 Database v4 races, found {len(v4_classes)}"
)


# ---------------------------------------------------------------------------
# Reconcile the two representations of the same races.
# ---------------------------------------------------------------------------

class_pilot = bha_classes.merge(
    v4_classes,
    on=[
        "race_date",
        "governed_racecourse_name",
        "advertised_time",
    ],
    how="outer",
    indicator=True,
)

assert (class_pilot["_merge"] == "both").all(), (
    "The 34-race reconciliation did not produce 34 exact matches."
)

assert len(class_pilot) == 34


# ---------------------------------------------------------------------------
# Classify the class-value comparison for each reconciled race.
# ---------------------------------------------------------------------------

def compare_class_values(row):
    bha_value = row["bha_race_class"]
    inside_rails_value = row["class_number"]

    if pd.isna(bha_value) and pd.isna(inside_rails_value):
        return "both_blank"

    if pd.isna(bha_value):
        return "bha_blank_only"

    if pd.isna(inside_rails_value):
        return "inside_rails_blank_only"

    if int(bha_value) == int(inside_rails_value):
        return "agreement"

    return "disagreement"


class_pilot["class_comparison"] = class_pilot.apply(
    compare_class_values,
    axis=1,
)

class_summary = (
    class_pilot["class_comparison"]
    .value_counts(dropna=False)
    .rename_axis("comparison")
    .reset_index(name="races")
)

class_summary

,comparison,races
0,agreement,34


## Evidence note 12 — Database v4 reproduces BHA race class across the 34-race pilot

The race-class comparison was extended from the single Plumpton example to all
34 reconciled BHA races on 27 May 2026.

For each race:

- BHA `raceClass` was treated as the authoritative structured race-class value;
- Database v4 `class_number` was the governed Inside Rails comparison value;
- races were reconciled by date, governed racecourse and advertised start time.

The result was:

| Comparison | Races |
|---|---:|
| agreement | 34 |

Therefore:

> **Database v4 reproduced the BHA race-class value for all 34 races in the bounded pilot.**

### Interpretation

For the races inspected, the relationship is:

> **direct equivalent**

between:

`BHA raceClass`

and:

`Inside Rails class_number`

The source representation differs only in format.

For example:

`Class 5`

is preserved as:

`class_raw = "Class 5"`

and governed as:

`class_number = 5`

which corresponds directly to:

`BHA raceClass = 5`.

### What this establishes

The pilot provides strong evidence that the existing Inside Rails `class`
field is representing the same British race-programme classification as the BHA
structured `raceClass` field.

It also validates the existing governed parser for all 34 races in this sample.

### What this does not establish

The result is bounded to the 34 official races inspected on 27 May 2026.

It does not establish:

- population-wide zero disagreement;
- historical completeness of BHA class data;
- that every British race must have a non-null Class;
- or that Class can be interpreted independently of race type and the relevant
  BHA programme rules.

The result should therefore be recorded as:

> **34/34 agreement in the bounded BHA-v4 pilot, with no class discrepancies observed.**

## Next bounded question

> **What does the BHA mean by a race's rating band, how is that concept represented
> in structured BHA data, and how well does Database v4 reproduce it?**

As with Class, the concept should be established from BHA evidence before
comparing `ratingBand` with the Inside Rails rating-band fields.

## BHA concept — rating band

Before comparing BHA `ratingBand` with the Inside Rails rating-band fields,
Study 05 must establish what the rating band means in British racing.

### Handicap ratings belong to horses

The BHA assigns handicap ratings to horses as numerical assessments of ability.

Those ratings help determine:

- which handicap races a horse can enter; and
- the weight allocated to the horse in a handicap.

A higher-rated horse is assessed as having greater ability than a lower-rated
horse.

The **rating band**, however, is a property of the programmed race rather than
of an individual horse.

It states a rating range used in the conditions of that race.

For example, the BHA describes a handicap with a rating band of:

`66–80`

Broadly, a horse rated from 66 to 80 is within the intended rating range for
that contest.

The BHA also uses programme descriptions such as:

- `0–65`;
- `0–75`;
- `0–85`;
- `0–95`;
- `Open`.

### Rating band is an eligibility condition, not a description of the realised field

A race carrying a particular rating band retains that programmed condition even
if the horses that eventually participate occupy only part of the range.

For example:

> a `0–85` race does not become a `0–79` race merely because the highest-rated
> starter happens to be rated 79.

The band describes the **prospective race conditions**.

The eventual runners describe one realisation of those conditions.

This reinforces the Study 05 distinction:

> **race specification ≠ realised participants**

### The boundaries are not always absolute entry cut-offs

The BHA's own explanation of handicap conditions also shows that a rating band
must not be interpreted too mechanically.

For a race with a `66–80` band:

- horses below the lower boundary may sometimes be entered and run from out of
  the handicap;
- horses one or two pounds above the upper boundary may normally also be
  eligible under specified conditions;
- elimination rules can affect whether such horses actually obtain a place in
  the race.

The numerical band is therefore a central race condition, but the complete
eligibility rule can require additional conditions.

Study 05 must not translate a band such as `66–80` into:

> only horses rated exactly 66 through 80 are legally capable of participating.

### Rating band is distinct from Class

BHA race-programme material demonstrates that rating band and Class are related
but separate properties.

For example, the 2026 Nursery Handicap programme includes:

| Rating band | Class |
|---|---:|
| 0–65 | 6 |
| 0–75 | 5 |
| 0–85 | 4 |
| 0–95 | 3 |
| Open | 2 |

This does not make the two concepts interchangeable.

`0–85`

describes a rating condition applying to the race.

`Class 4`

describes its race-programme classification.

Both can belong simultaneously to the specification of the same race.

### Working Study 05 interpretation

For this study:

> **A rating band is a prospective race condition expressed on the BHA
> handicap-rating scale that defines the principal rating range for which the
> race is programmed, subject to any additional eligibility, weight and
> elimination rules applying to that race.**

It is therefore distinct from:

- the Class of the race;
- an individual horse's handicap rating;
- the weights actually carried;
- and the ratings of the horses that eventually run.

### Structured BHA representation

The BHA structured race record exposes this concept through:

`ratingBand`

Observed records in the existing BHA pilot include values such as:

`51-70`

while races for which no rating band is represented can return a blank value.

The next question is therefore:

> **Does Database v4 represent the same BHA rating-band concept and value for
> the same individual races?**

In [9]:
# Compare BHA rating bands with Database v4 across the 34-race pilot.
#
# The BHA supplies the programmed race rating band through `ratingBand`.
# Database v4 preserves the source text and separately stores governed
# lower/upper bounds where the source syntax is canonical.
#
# The comparison distinguishes:
#   - both sides blank;
#   - exact textual agreement;
#   - equivalent numerical bounds despite formatting differences;
#   - one-sided missing values;
#   - genuine disagreement.

import re


# ---------------------------------------------------------------------------
# Build the BHA rating-band dataset from the already identified pilot fixtures.
# ---------------------------------------------------------------------------

bha_rating_rows = []

for fixture in bha_fixtures:
    race_list_url = (
        "https://api09.horseracing.software/bha/v1/fixtures/"
        f"{fixture['fixtureYear']}/"
        f"{fixture['fixtureId']}/races"
    )

    race_list_request = Request(
        race_list_url,
        headers={
            "Authorization": authorization_value,
            "Accept": "application/json",
            "Origin": "https://www.britishhorseracing.com",
            "Referer": "https://www.britishhorseracing.com/",
            "User-Agent": USER_AGENT,
        },
    )

    with urlopen(race_list_request, timeout=30) as response:
        race_payload = json.loads(
            response.read().decode("utf-8")
        )

    for race in race_payload["data"]:
        bha_rating_rows.append(
            {
                "race_date": race["raceDate"],
                "governed_racecourse_name": fixture["courseName"],
                "advertised_time": race["raceTime"],
                "bha_race_id": race["raceId"],
                "bha_race_name": race["raceName"],
                "bha_rating_band": race["ratingBand"],
            }
        )

bha_rating_bands = pd.DataFrame(bha_rating_rows)

assert len(bha_rating_bands) == 34


# ---------------------------------------------------------------------------
# Load the existing Inside Rails rating-band representation for the same date.
# ---------------------------------------------------------------------------

with connect_read_only(DATABASE) as connection:
    v4_rating_bands = pd.read_sql_query(
        """
        SELECT
            source_race_occurrence_code,
            raw_date AS race_date,
            governed_racecourse_name,
            substr(advertised_start_course_local, 12, 8) AS advertised_time,
            race_name_raw,
            rating_band_raw,
            rating_lower_bound,
            rating_upper_bound,
            rating_band_parse_status
        FROM view_gb_reconciled_race_occurrences_with_racecourse
        WHERE raw_date = ?
        """,
        connection,
        params=[pilot_date],
    )

assert len(v4_rating_bands) == 34


# ---------------------------------------------------------------------------
# Reconcile BHA and Inside Rails records for the same race occurrences.
# ---------------------------------------------------------------------------

rating_band_pilot = bha_rating_bands.merge(
    v4_rating_bands,
    on=[
        "race_date",
        "governed_racecourse_name",
        "advertised_time",
    ],
    how="outer",
    indicator=True,
)

assert len(rating_band_pilot) == 34

assert (rating_band_pilot["_merge"] == "both").all(), (
    "The 34-race reconciliation did not produce 34 exact matches."
)


# ---------------------------------------------------------------------------
# Parse canonical BHA N-N bands so they can also be compared numerically.
# ---------------------------------------------------------------------------

def parse_bha_rating_band(value):
    if value is None:
        return None

    value = str(value).strip()

    if value == "":
        return None

    match = re.fullmatch(r"(\d+)-(\d+)", value)

    if not match:
        return None

    return int(match.group(1)), int(match.group(2))


# ---------------------------------------------------------------------------
# Classify the semantic relationship for each race.
# ---------------------------------------------------------------------------

def compare_rating_band(row):
    bha_raw = (
        ""
        if pd.isna(row["bha_rating_band"])
        else str(row["bha_rating_band"]).strip()
    )

    ir_raw = (
        ""
        if pd.isna(row["rating_band_raw"])
        else str(row["rating_band_raw"]).strip()
    )

    if bha_raw == "" and ir_raw == "":
        return "both_blank"

    if bha_raw == "" and ir_raw != "":
        return "bha_blank_only"

    if bha_raw != "" and ir_raw == "":
        return "inside_rails_blank_only"

    if bha_raw == ir_raw:
        return "exact_text_agreement"

    bha_bounds = parse_bha_rating_band(bha_raw)

    if (
        bha_bounds is not None
        and pd.notna(row["rating_lower_bound"])
        and pd.notna(row["rating_upper_bound"])
        and bha_bounds
        == (
            int(row["rating_lower_bound"]),
            int(row["rating_upper_bound"]),
        )
    ):
        return "numeric_bounds_agreement"

    return "disagreement"


rating_band_pilot["rating_band_comparison"] = rating_band_pilot.apply(
    compare_rating_band,
    axis=1,
)

rating_band_summary = (
    rating_band_pilot["rating_band_comparison"]
    .value_counts(dropna=False)
    .rename_axis("comparison")
    .reset_index(name="races")
)

rating_band_summary

,comparison,races
0,disagreement,14
1,both_blank,11
2,exact_text_agreement,9


In [10]:
# Inspect every rating-band disagreement in the 34-race pilot.
#
# The aim is to determine whether these are:
#   - formatting differences;
#   - different observation/programme states;
#   - different concepts;
#   - or genuine source disagreements.
#
# Show the race identity/context alongside both BHA and Inside Rails
# representations and the governed parsing result.

rating_band_disagreements = (
    rating_band_pilot.loc[
        rating_band_pilot["rating_band_comparison"] == "disagreement",
        [
            "race_date",
            "governed_racecourse_name",
            "advertised_time",
            "bha_race_id",
            "bha_race_name",
            "bha_rating_band",
            "race_name_raw",
            "rating_band_raw",
            "rating_lower_bound",
            "rating_upper_bound",
            "rating_band_parse_status",
        ],
    ]
    .sort_values(
        [
            "governed_racecourse_name",
            "advertised_time",
        ]
    )
    .reset_index(drop=True)
)

assert len(rating_band_disagreements) == 14

rating_band_disagreements

,race_date,governed_racecourse_name,advertised_time,bha_race_id,bha_race_name,bha_rating_band,race_name_raw,rating_band_raw,rating_lower_bound,rating_upper_bound,rating_band_parse_status
0,2026-05-27,Beverley,15:15:00,1107,THE WARD HOMES YORKSHIRE 10TH ANNIVERSARY HAND...,56-75,Ward Homes Yorkshire 10th Anniversary Handicap,0-75,0.0,75.0,canonical
1,2026-05-27,Beverley,15:45:00,24061,THE DR EDDIE MOLL HANDICAP STAKES (CLASS 5),56-75,Dr Eddie Moll Handicap,0-75,0.0,75.0,canonical
2,2026-05-27,Beverley,16:15:00,1105,THE CONNEXIN'S FULL FIBRE FOR ALL HANDICAP STA...,46-60,Connexins Full Fibre For All Handicap,0-60,0.0,60.0,canonical
3,2026-05-27,Beverley,16:50:00,1106,THE UP THE TIGERS HANDICAP STAKES (CLASS 6),46-60,Up The Tigers Handicap,0-60,0.0,60.0,canonical
4,2026-05-27,Beverley,17:20:00,1108,THE RACING AGAIN THIS SATURDAY APPRENTICE HAND...,46-60,Racing Again This Saturday Apprentice Handicap,0-60,0.0,60.0,canonical
5,2026-05-27,Hamilton Park,15:05:00,2974,THE ICB (WATER PROOFING) HANDICAP STAKES (CLAS...,61-80,ICB (Water Proofing) Handicap,0-80,0.0,80.0,canonical
6,2026-05-27,Hamilton Park,15:35:00,2977,THE CUPA SLATES HANDICAP STAKES (CLASS 4) (GBB...,61-80,Cupa Slates Handicap (GBBPlus Race),0-80,0.0,80.0,canonical
7,2026-05-27,Hamilton Park,16:05:00,66583,THE DAVID HARDIE ENGINEERING LTD HANDICAP STAK...,61-80,David Hardie Engineering Ltd Handicap,0-80,0.0,80.0,canonical
8,2026-05-27,Hamilton Park,16:35:00,67064,THE FIXING POINT CLADDING FASTENERS HANDICAP S...,56-75,Fixing Point Cladding Fasteners Handicap,0-75,0.0,75.0,canonical
9,2026-05-27,Hamilton Park,17:05:00,2973,THE COMPASS ROOFING HANDICAP STAKES (CLASS 5),51-70,Compass Roofing Handicap,0-70,0.0,70.0,canonical


## Evidence note 13 — `rating_band_raw` is not always a direct equivalent of BHA `ratingBand`

The BHA-versus-Database-v4 rating-band comparison across the 34-race pilot
produced:

| Comparison | Races |
|---|---:|
| exact text agreement | 9 |
| both blank | 11 |
| disagreement | 14 |

Inspection of all 14 disagreements revealed a highly systematic pattern.

Examples include:

| BHA `ratingBand` | Inside Rails `rating_band_raw` |
|---|---|
| `56-75` | `0-75` |
| `46-60` | `0-60` |
| `61-80` | `0-80` |
| `51-70` | `0-70` |
| `66-85` | `0-85` |
| `71-90` | `0-90` |

In every disagreement:

> **the upper bound agrees, while Database v4 replaces the BHA lower bound with zero.**

This is too systematic to treat as random transcription error.

### BHA evidence clarifies the distinction

The BHA's handicapping guidance describes races using closed rating bands such
as `66-80`.

For such a race, horses rated within that range are the horses for whom the
race is principally suitable.

The BHA also explains that a horse rated below the lower boundary can sometimes
still enter, but would run **out of the handicap** and carry the minimum weight.

Therefore the lower boundary in a BHA rating band has sporting meaning. It is
not merely decorative formatting.

BHA race-programme material also demonstrates that `0-X` notation genuinely
exists within British racing.

For example, BHA programme material has described Nursery Handicaps as:

- `0-65`;
- `0-75`;
- `0-85`;
- `0-95`.

Historical BHA programmes can even contain `0-X` races and closed-band races
such as `61-80`, `46-60` and `66-85` on the same card.

Therefore:

> **`0-X` and `N-X` are both real forms used in British race programming, and
> they must not be assumed to mean the same thing.**

### Consequence for Inside Rails

For the 14 discrepant races, the existing Inside Rails field has preserved the
same upper rating boundary as the BHA but has not preserved the lower boundary
reported by the official BHA structured record.

The correct current classification is therefore:

> **partial representation**

rather than:

> **direct equivalent**

for those races.

This does not yet establish why Source Version 1 contains `0-X`.

Possible explanations include:

- a deliberate source convention that records only the upper rating ceiling;
- a transformation from another race-programme description;
- a different observation stage;
- or loss of information from the complete official race conditions.

Those possibilities require evidence before one is accepted.

### Revised pilot result

Of the 34 races:

- 9 have exact non-blank BHA / Inside Rails rating-band agreement;
- 11 are blank on both sides;
- 14 preserve the BHA upper boundary but differ on the lower boundary.

Therefore the existing `rating_band_raw` field cannot currently be described
as a universally direct representation of BHA `ratingBand`.

The earlier Notebook 16 decision to preserve `rating_band_raw` separately from
its parsed bounds was consequently important: parsing `"0-80"` correctly does
not establish that `0` is the authoritative BHA lower rating boundary.

## Next bounded question

> **What distinguishes the 9 races where BHA and Inside Rails agree exactly
> from the 14 races where Inside Rails records `0-X` instead of the BHA closed
> rating band?**

The next step is to inspect those two groups by race type and race name before
trying to explain the source convention.

In [11]:
# Compare the two non-blank rating-band groups by race type and class.
#
# This adds the existing governed race descriptors from Database v4, then
# summarises where exact agreement and lower-bound disagreement occur.
# The detailed rows are retained so we can inspect the race names if the
# grouped pattern does not explain the distinction.

with connect_read_only(DATABASE) as connection:
    rating_context = pd.read_sql_query(
        """
        SELECT
            source_race_occurrence_code,
            race_type_raw,
            class_number,
            age_band_raw
        FROM view_gb_reconciled_race_occurrences_with_racecourse
        WHERE raw_date = ?
        """,
        connection,
        params=[pilot_date],
    )

rating_band_context = rating_band_pilot.merge(
    rating_context,
    on="source_race_occurrence_code",
    how="left",
    validate="one_to_one",
)

nonblank_rating_cases = rating_band_context[
    rating_band_context["rating_band_comparison"].isin(
        [
            "exact_text_agreement",
            "disagreement",
        ]
    )
].copy()

group_summary = (
    nonblank_rating_cases
    .groupby(
        [
            "rating_band_comparison",
            "race_type_raw",
            "class_number",
        ],
        dropna=False,
    )
    .size()
    .reset_index(name="races")
    .sort_values(
        [
            "rating_band_comparison",
            "race_type_raw",
            "class_number",
        ]
    )
)

display(group_summary)

display(
    nonblank_rating_cases[
        [
            "governed_racecourse_name",
            "advertised_time",
            "race_type_raw",
            "class_number",
            "age_band_raw",
            "bha_race_name",
            "bha_rating_band",
            "rating_band_raw",
            "rating_band_comparison",
        ]
    ]
    .sort_values(
        [
            "rating_band_comparison",
            "race_type_raw",
            "governed_racecourse_name",
            "advertised_time",
        ]
    )
    .reset_index(drop=True)
)

,rating_band_comparison,race_type_raw,class_number,races
0,disagreement,Flat,3,1
1,disagreement,Flat,4,5
2,disagreement,Flat,5,5
3,disagreement,Flat,6,3
4,exact_text_agreement,Chase,3,1
5,exact_text_agreement,Chase,5,2
6,exact_text_agreement,Hurdle,4,3
7,exact_text_agreement,Hurdle,5,3


,governed_racecourse_name,advertised_time,race_type_raw,class_number,age_band_raw,bha_race_name,bha_rating_band,rating_band_raw,rating_band_comparison
0,Beverley,15:15:00,Flat,5,4yo+,THE WARD HOMES YORKSHIRE 10TH ANNIVERSARY HAND...,56-75,0-75,disagreement
1,Beverley,15:45:00,Flat,5,4yo+,THE DR EDDIE MOLL HANDICAP STAKES (CLASS 5),56-75,0-75,disagreement
2,Beverley,16:15:00,Flat,6,4yo+,THE CONNEXIN'S FULL FIBRE FOR ALL HANDICAP STA...,46-60,0-60,disagreement
3,Beverley,16:50:00,Flat,6,4yo+,THE UP THE TIGERS HANDICAP STAKES (CLASS 6),46-60,0-60,disagreement
4,Beverley,17:20:00,Flat,6,4yo+,THE RACING AGAIN THIS SATURDAY APPRENTICE HAND...,46-60,0-60,disagreement
5,Hamilton Park,15:05:00,Flat,4,3yo,THE ICB (WATER PROOFING) HANDICAP STAKES (CLAS...,61-80,0-80,disagreement
6,Hamilton Park,15:35:00,Flat,4,4yo+,THE CUPA SLATES HANDICAP STAKES (CLASS 4) (GBB...,61-80,0-80,disagreement
7,Hamilton Park,16:05:00,Flat,4,4yo+,THE DAVID HARDIE ENGINEERING LTD HANDICAP STAK...,61-80,0-80,disagreement
8,Hamilton Park,16:35:00,Flat,5,3yo,THE FIXING POINT CLADDING FASTENERS HANDICAP S...,56-75,0-75,disagreement
9,Hamilton Park,17:05:00,Flat,5,4yo+,THE COMPASS ROOFING HANDICAP STAKES (CLASS 5),51-70,0-70,disagreement


## Evidence note 14 — the rating-band discrepancy separates cleanly by racing form

The 23 pilot races with a non-blank rating band separate perfectly by
race type.

| Inside Rails race type | BHA / Inside Rails relationship | Races |
|---|---|---:|
| Flat | lower-bound disagreement | 14 |
| Hurdle | exact agreement | 6 |
| Chase | exact agreement | 3 |

There are no Flat exact agreements and no Jump disagreements in this
bounded sample.

### Flat handicaps

For all 14 Flat races, Database v4 preserves the BHA upper boundary but
replaces the BHA lower boundary with zero.

Examples are:

| BHA | Database v4 |
|---|---|
| `46-60` | `0-60` |
| `51-70` | `0-70` |
| `56-75` | `0-75` |
| `61-80` | `0-80` |
| `66-85` | `0-85` |
| `71-90` | `0-90` |

This corresponds to genuine BHA Flat-programme terminology.

BHA race-programme material describes ordinary Flat handicaps using
closed bands such as:

- `46-60`;
- `51-70`;
- `56-75`;
- `61-80`;
- `66-85`;
- `71-90`.

The lower bound is therefore part of the official programmed race
condition.

For these races, Source Version 1 has not merely reformatted the BHA
rating band. It has discarded that lower-bound information.

### Jump handicaps

The nine Jump handicaps show a different pattern.

BHA and Database v4 agree exactly on values including:

- `0-95`;
- `0-100`;
- `0-105`;
- `0-115`;
- `0-120`;
- `0-135`.

BHA programme material independently confirms that `0-X` notation is
genuine Jump race-programme terminology.

Examples published by the BHA include `0-105` handicap hurdles and
steeple chases, as well as higher-rated `0-X` Jump handicaps.

Therefore the zero lower bound in these Jump records is not evidence of
information loss: it corresponds to the official BHA representation.

### Important qualification — `0-X` is not exclusively Jump terminology

The distinction cannot be simplified to:

> Flat = closed band  
> Jump = `0-X`

The BHA also uses genuine `0-X` bands in some Flat programmes.

A clear example is the Nursery programme, where BHA programme bands
include:

- `0-65`;
- `0-75`;
- `0-85`;
- `0-95`.

Therefore the correct rule is not based simply on racing code.

Instead:

> **The authoritative rating band is the band specified for the
> individual race by the BHA.**

The source value must be compared with that official condition rather
than interpreted through a universal `0-X` convention.

### Revised interpretation of the Inside Rails field

The 34-race pilot now shows two different behaviours.

For the nine Jump handicaps inspected:

> `rating_band_raw` is a direct equivalent of BHA `ratingBand`.

For the 14 Flat handicaps inspected:

> `rating_band_raw` is only a partial representation of BHA
> `ratingBand`: the upper boundary is preserved but the official lower
> boundary is lost.

The existing parser itself is behaving correctly.

For example:

`rating_band_raw = "0-80"`

is correctly parsed as:

`rating_lower_bound = 0`  
`rating_upper_bound = 80`

The problem lies upstream of the parser: the source text being parsed
does not contain the complete official BHA rating band.

This is therefore a **source-semantics / source-information-loss issue**,
not a parsing defect.

## Consequence for Database v4

Database v4 must not currently treat `rating_lower_bound` derived from
`rating_band_raw` as an authoritative British race-condition boundary.

For Great Britain:

- the upper bound appears useful in the pilot;
- the lower bound requires authoritative validation;
- exact `0-X` values may be genuine for some races;
- and an apparent `0-X` value cannot by itself establish that the
  official BHA lower boundary was zero.

## Next bounded question

> **How widespread is this source convention across the Great Britain
> population?**

Before deciding whether this requires database correction or a later
BHA enrichment, inspect how `rating_band_raw` is distributed across
Flat, Hurdle, Chase and NH Flat races in Database v4.

In [12]:
# Describe the forms of rating-band representation across the complete
# Database v4 Great Britain race population.
#
# Analytical question:
#
#   How are rating bands represented in Source Version 1 / Database v4,
#   and how does that representation vary by the source race-type label?
#
# This is a source-population description only. It does not require the old
# Study 02 post-v3 race-type overlay, so we deliberately do not use
# `build_race_overlay_query()` here.
#
# `race_type_raw` is retained under the historical column name
# `race_type_study` so the existing downstream notebook cells do not need to
# change. For this bounded rating-band investigation it is being used only as
# a descriptive grouping variable.
#
# No source off-time field is used.

with connect_read_only(DATABASE) as connection:
    rating_band_population = pd.read_sql_query(
        """
        WITH classified AS (
            SELECT
                race_type_raw AS race_type_study,

                CASE
                    WHEN rating_band_raw IS NULL
                      OR TRIM(rating_band_raw) = ''
                        THEN 'blank'

                    WHEN rating_lower_bound = 0
                     AND rating_upper_bound IS NOT NULL
                        THEN '0-X'

                    WHEN rating_lower_bound > 0
                     AND rating_upper_bound IS NOT NULL
                        THEN 'N-X'

                    ELSE 'other'
                END AS rating_band_form

            FROM view_gb_reconciled_race_occurrences_with_racecourse
        )

        SELECT
            race_type_study,
            rating_band_form,
            COUNT(*) AS races

        FROM classified

        GROUP BY
            race_type_study,
            rating_band_form

        ORDER BY
            race_type_study,
            rating_band_form
        """,
        connection,
    )


print(
    "GB races represented:",
    rating_band_population["races"].sum(),
)

display(rating_band_population)

GB races represented: 111634


,race_type_study,rating_band_form,races
0,Chase,0-X,11501
1,Chase,blank,4170
2,Flat,0-X,48675
3,Flat,N-X,15
4,Flat,blank,21528
5,Hurdle,0-X,12168
6,Hurdle,blank,10477
7,NH Flat,blank,3100


In [13]:
# Measure how the source rating-band representation is distributed across the
# full Great Britain race population.
#
# We already know from the 34-race BHA pilot that:
#
#   - some official rating bands genuinely begin at zero, such as 0-105;
#   - some official rating bands have a positive lower bound, such as 61-80;
#   - Source Version 1 sometimes stores 0-X where the BHA stores N-X.
#
# This population check does NOT attempt to decide which individual values are
# correct. It asks a simpler descriptive question:
#
#   How often does Database v4 contain a zero-lower-bound band, a positive-
#   lower-bound band, a blank band, or some other source form, and how does
#   that distribution differ between Flat, Hurdle, Chase and NH Flat?
#
# The existing governed rating-band parser is used to classify the notation.
# The pending race-type overlay is applied so the known verified race-type
# corrections are reflected in the four racing-form groups.

from inside_rails.study_overlay import build_race_overlay_query


# ---------------------------------------------------------------------------
# Supply the race-level fields needed both for this analysis and for the
# existing read-only post-release overlay.
#
# The overlay returns `race_type_study`, which is the study-facing race type
# after applying any verified corrections that are not yet native to v4.
# ---------------------------------------------------------------------------

base_query = """
SELECT
    source_race_occurrence_code,
    raw_date,
    raw_course,
    raw_off,
    race_type_raw,
    advertised_start_course_local,
    rating_band_raw,
    rating_lower_bound,
    rating_upper_bound,
    rating_band_parse_status
FROM view_gb_reconciled_race_occurrences_with_racecourse
"""

overlay_sql, overlay_params = build_race_overlay_query(base_query)


# ---------------------------------------------------------------------------
# Classify each race by the form of rating band preserved in Database v4.
#
# `canonical_zero_lower`
#     A successfully parsed N-N band whose stored lower boundary is zero.
#
# `canonical_positive_lower`
#     A successfully parsed N-N band whose stored lower boundary is above zero.
#
# `blank`
#     No source rating band was supplied.
#
# `other_source_form`
#     A non-blank value exists but is not one of the governed canonical N-N
#     forms. These remain separate rather than being forced into either group.
# ---------------------------------------------------------------------------

with connect_read_only(DATABASE) as connection:
    rating_band_population = pd.read_sql_query(
        f"""
        SELECT
            race_type_study,
            CASE
                WHEN rating_band_parse_status = 'blank'
                    THEN 'blank'

                WHEN rating_band_parse_status = 'canonical'
                     AND rating_lower_bound = 0
                    THEN 'canonical_zero_lower'

                WHEN rating_band_parse_status = 'canonical'
                     AND rating_lower_bound > 0
                    THEN 'canonical_positive_lower'

                ELSE 'other_source_form'
            END AS rating_band_form,
            COUNT(*) AS races
        FROM ({overlay_sql})
        GROUP BY
            race_type_study,
            rating_band_form
        ORDER BY
            race_type_study,
            rating_band_form
        """,
        connection,
        params=overlay_params,
    )


# ---------------------------------------------------------------------------
# Add within-race-type percentages so we can compare the structure of the
# four racing forms even though their total race counts are very different.
# ---------------------------------------------------------------------------

rating_band_population["race_type_total"] = (
    rating_band_population
    .groupby("race_type_study")["races"]
    .transform("sum")
)

rating_band_population["percent_of_race_type"] = (
    100
    * rating_band_population["races"]
    / rating_band_population["race_type_total"]
).round(1)

rating_band_population

,race_type_study,rating_band_form,races,race_type_total,percent_of_race_type
0,Chase,blank,4163,15664,26.6
1,Chase,canonical_zero_lower,11501,15664,73.4
2,Flat,blank,21518,70208,30.6
3,Flat,canonical_positive_lower,15,70208,0.0
4,Flat,canonical_zero_lower,48675,70208,69.3
5,Hurdle,blank,10485,22653,46.3
6,Hurdle,canonical_zero_lower,12168,22653,53.7
7,NH Flat,blank,3109,3109,100.0


## Evidence note 15 — Database v4 overwhelmingly stores rating bands as `0-X`

The full Great Britain population was examined using the governed rating-band
parser and the study-facing race-type classification.

The result is:

| Race type | Blank | Canonical `0-X` | Canonical positive-lower | Total |
|---|---:|---:|---:|---:|
| Chase | 4,163 | 11,501 | 0 | 15,664 |
| Flat | 21,518 | 48,675 | 15 | 70,208 |
| Hurdle | 10,485 | 12,168 | 0 | 22,653 |
| NH Flat | 3,109 | 0 | 0 | 3,109 |

### Jump racing

For both Hurdle and Chase, every non-blank canonical rating band in Database v4
has a zero lower bound.

This is consistent with the official BHA examples already observed in the
34-race pilot, including:

- `0-95`;
- `0-100`;
- `0-105`;
- `0-115`;
- `0-120`;
- `0-135`.

The population pattern therefore gives no immediate reason to treat the
zero lower boundary in Jump bands as suspicious.

National Hunt Flat races contain no rating band in Database v4, which is also
consistent with their different race form.

### Flat racing

Flat racing is much more significant.

Of 70,208 Flat races:

- 21,518 have no rating band;
- 48,675 contain a canonical `0-X` band;
- only 15 contain a canonical band with a positive lower boundary.

Therefore, among Flat races for which Database v4 contains a canonical rating
band, almost every stored value begins at zero.

This matters because the BHA pilot showed that ordinary Flat handicaps officially
represented as:

- `46-60`;
- `51-70`;
- `56-75`;
- `61-80`;
- `66-85`;
- `71-90`;

were stored in Source Version 1 as:

- `0-60`;
- `0-70`;
- `0-75`;
- `0-80`;
- `0-85`;
- `0-90`.

The full-population distribution is therefore consistent with a broad
Source Version 1 convention that frequently preserves the **upper rating
boundary** of Flat races while representing the lower boundary as zero.

### Important limitation

This population result does **not** mean that all 48,675 Flat `0-X` values are
incorrect.

Official BHA Flat programmes can genuinely contain `0-X` races, including
Nursery Handicaps.

The database alone cannot tell us which Flat `0-X` values:

- reproduce an official BHA `0-X` band;
- or have lost a positive official lower boundary.

That distinction requires authoritative race-level evidence.

### Consequence

The existing Database v4 Flat `rating_lower_bound` cannot be treated as an
authoritative British race-condition boundary merely because it parsed
successfully.

For Flat racing in particular:

> **canonical syntax does not imply authoritative semantics.**

The upper boundary appears much more promising: it agreed in every non-blank
BHA pilot comparison, including all 14 lower-bound disagreements.

This creates a potentially substantial source-quality issue rather than a
small collection of anomalous rows.

## Next bounded question

> **What are the 15 exceptional Flat races in Database v4 that preserve a
> positive lower rating boundary?**

Inspecting those 15 races may reveal whether they represent a distinct race
type, period or source convention and help us understand how Source Version 1
constructed the Flat rating-band field.

In [14]:
# Inspect all Flat races whose stored canonical rating band has a
# positive lower boundary.
#
# Why we are doing this:
#
# Across the full GB population, almost every non-blank Flat rating band
# in Database v4 is stored as 0-X. Only 15 Flat races preserve an N-X form
# such as 61-80.
#
# Because there are only 15 exceptions, we can inspect every one rather
# than sample them.
#
# We want to find out whether these 15 races share something obvious:
#
#   - the same historical period;
#   - particular racecourses;
#   - particular Classes;
#   - particular age conditions;
#   - similar race names;
#   - or particular rating-band values.
#
# If they cluster, that may tell us why Source Version 1 occasionally
# preserves the positive lower boundary while usually reducing Flat bands
# to 0-X.
#
# The study overlay requires the source race-identity and advertised-time
# fields because it applies verified post-release corrections. We therefore
# include those fields in the base query alongside all the descriptive
# columns needed for this inspection.

flat_exception_base_query = """
SELECT
    source_race_occurrence_code,

    raw_date,
    raw_course,
    raw_off,

    governed_racecourse_name,
    advertised_start_course_local,

    race_name_raw,
    race_type_raw,

    class_raw,
    class_number,
    age_band_raw,

    rating_band_raw,
    rating_lower_bound,
    rating_upper_bound,
    rating_band_parse_status

FROM view_gb_reconciled_race_occurrences_with_racecourse
"""

flat_exception_overlay_sql, flat_exception_overlay_params = (
    build_race_overlay_query(flat_exception_base_query)
)


# Now select only the unusual Flat cases:
#
#   race_type_study = 'Flat'
#       uses the study-facing race type after verified corrections.
#
#   rating_band_parse_status = 'canonical'
#       limits us to properly parsed N-N rating bands.
#
#   rating_lower_bound > 0
#       isolates the tiny group where the source actually retained
#       a positive lower boundary rather than storing 0-X.
#
# We display the governed advertised time from the overlay rather than
# the raw source time.

with connect_read_only(DATABASE) as connection:
    flat_positive_lower = pd.read_sql_query(
        f"""
        SELECT
            source_race_occurrence_code,
            raw_date AS race_date,
            governed_racecourse_name,
            advertised_start_course_local_study AS advertised_start_course_local,

            race_name_raw,
            class_raw,
            class_number,
            age_band_raw,

            rating_band_raw,
            rating_lower_bound,
            rating_upper_bound,
            rating_band_parse_status

        FROM ({flat_exception_overlay_sql})

        WHERE
            race_type_study = 'Flat'
            AND rating_band_parse_status = 'canonical'
            AND rating_lower_bound > 0

        ORDER BY
            raw_date,
            governed_racecourse_name,
            advertised_start_course_local_study
        """,
        connection,
        params=flat_exception_overlay_params,
    )


# The previous population summary established that exactly 15 such races
# exist. This assertion makes sure the detailed inspection is looking at
# precisely that same population.

assert len(flat_positive_lower) == 15, (
    f"Expected 15 Flat positive-lower rating bands, "
    f"found {len(flat_positive_lower)}"
)

flat_positive_lower

,source_race_occurrence_code,race_date,governed_racecourse_name,advertised_start_course_local,race_name_raw,class_raw,class_number,age_band_raw,rating_band_raw,rating_lower_bound,rating_upper_bound,rating_band_parse_status
0,race:77b5dbbbfdee69d4d92a5826:000007234,2015-06-10,Brighton,2015-06-10T16:20:00+01:00,Star Sports Bet Handicap,Class 5,5,3yo,51-70,51,70,canonical
1,race:77b5dbbbfdee69d4d92a5826:000007232,2015-06-10,Brighton,2015-06-10T17:20:00+01:00,checkatrade.com Handicap,Class 5,5,3yo+,51-70,51,70,canonical
2,race:77b5dbbbfdee69d4d92a5826:000007527,2015-06-15,Nottingham,NaN,Des Walker Handicap,Class 4,4,3yo,66-85,66,85,canonical
3,race:77b5dbbbfdee69d4d92a5826:000010990,2015-08-24,Leicester,2015-08-24T18:25:00+01:00,John Smiths Extra Smooth Handicap,Class 5,5,3yo+,61-75,61,75,canonical
4,race:77b5dbbbfdee69d4d92a5826:000010992,2015-08-24,Leicester,2015-08-24T19:25:00+01:00,Strongbow Cloudy Apple Handicap,Class 5,5,3yo+,56-70,56,70,canonical
5,race:77b5dbbbfdee69d4d92a5826:000010981,2015-08-24,Leicester,2015-08-24T19:55:00+01:00,Foxton Locks Inn Handicap,Class 4,4,3yo+,66-80,66,80,canonical
6,race:77b5dbbbfdee69d4d92a5826:000011622,2015-09-05,Ascot,2015-09-05T16:35:00+01:00,Bibendum Wine Ltd Handicap,Class 2,2,3yo+,86-105,86,105,canonical
7,race:77b5dbbbfdee69d4d92a5826:000011804,2015-09-10,Epsom Downs,2015-09-10T16:25:00+01:00,JRA Handicap,Class 5,5,3yo,61-75,61,75,canonical
8,race:77b5dbbbfdee69d4d92a5826:000012954,2015-10-03,Ascot,2015-10-03T16:55:00+01:00,AP Security Handicap,Class 3,3,3yo+,81-95,81,95,canonical
9,race:77b5dbbbfdee69d4d92a5826:000013147,2015-10-07,Nottingham,2015-10-07T16:10:00+01:00,£10 Free At 32Red.com Handicap,Class 5,5,3yo+,59-73,59,73,canonical


## Evidence note 16 — the 15 positive-lower Flat bands are temporally clustered source exceptions

All 15 Flat races in Database v4 whose stored canonical rating band has a
positive lower boundary were inspected exhaustively.

They do not form an obvious sporting category.

The races span:

- Classes 2 through 6;
- both `3yo` and `3yo+` age conditions;
- multiple racecourses;
- and ordinary Flat handicap rating bands including `51-70`, `61-75`,
  `66-80`, `66-85`, `81-95` and `86-105`.

The striking feature is instead their chronology.

### Temporal distribution

Of the 15 races:

- 10 occur during 2015;
- the remaining 5 all occur at Newcastle on 18 December 2019;
- none occur from 2020 onward.

The 2015 cases themselves are concentrated on a small number of racecards,
including multiple positive-lower bands on the same card at Brighton and
Leicester.

The five 2019 cases likewise occur on a single Newcastle card.

### Interpretation

This pattern gives no evidence that positive-lower Flat rating bands were
retained because of Class, age restriction or another obvious race-condition
category.

Instead, their concentration on a handful of historical racecards is
consistent with a source-acquisition or source-representation difference.

That interpretation also fits the wider population result:

> among 48,690 Flat races with canonical non-blank rating bands, only 15
> preserve a positive lower boundary.

The 15 exceptions therefore should not be used to infer a general semantic
rule for Source Version 1.

They appear to be exceptional records within a source that otherwise
overwhelmingly represents Flat rating bands as `0-X`.

### Current conclusion on the field

The evidence now supports:

> **Source Version 1 does not reliably preserve the official lower boundary of
> British Flat handicap rating bands.**

In the BHA pilot, official positive lower boundaries were systematically
replaced by zero.

Across the full Database v4 population, nearly every stored Flat rating band
uses the same `0-X` convention.

The 15 positive-lower exceptions are highly clustered historically and do not
provide evidence of a different sporting meaning.

The exact provenance of those exceptional records remains unresolved.

## Next bounded question

> **On the racecards containing the 15 exceptions, is the positive-lower form
> a card-level source behaviour or does it occur alongside `0-X` Flat
> handicaps on the same course and date?**

This can distinguish a whole-card/source-feed effect from isolated
race-level exceptions.

In [15]:
# Test whether the unusual positive-lower rating bands occur as a whole-card
# source behaviour.
#
# The 15 exceptions are concentrated on a small number of course-date cards.
# If all rated Flat handicaps on those cards use N-X notation, that would
# support the idea that the source captured those particular cards differently.
#
# If N-X and 0-X appear side by side on the same card, the explanation must
# operate at a finer race level instead.
#
# We therefore:
#   1. take the unique course-date combinations represented by the 15 cases;
#   2. load every Flat race on those same cards;
#   3. retain the race name and governed rating-band representation;
#   4. display the complete card context for inspection.

exception_cards = (
    flat_positive_lower[
        ["race_date", "governed_racecourse_name"]
    ]
    .drop_duplicates()
    .sort_values(
        ["race_date", "governed_racecourse_name"]
    )
    .reset_index(drop=True)
)

card_conditions = " OR ".join(
    [
        "(raw_date = ? AND governed_racecourse_name = ?)"
        for _ in range(len(exception_cards))
    ]
)

card_params = []

for row in exception_cards.itertuples(index=False):
    card_params.extend(
        [
            row.race_date,
            row.governed_racecourse_name,
        ]
    )


with connect_read_only(DATABASE) as connection:
    exception_card_context = pd.read_sql_query(
        f"""
        SELECT
            raw_date AS race_date,
            governed_racecourse_name,
            advertised_start_course_local,
            race_name_raw,
            race_type_raw,
            class_raw,
            age_band_raw,
            rating_band_raw,
            rating_lower_bound,
            rating_upper_bound,
            rating_band_parse_status

        FROM view_gb_reconciled_race_occurrences_with_racecourse

        WHERE
            ({card_conditions})
            AND race_type_raw = 'Flat'

        ORDER BY
            raw_date,
            governed_racecourse_name,
            advertised_start_course_local
        """,
        connection,
        params=card_params,
    )

exception_card_context

,race_date,governed_racecourse_name,advertised_start_course_local,race_name_raw,race_type_raw,class_raw,age_band_raw,rating_band_raw,rating_lower_bound,rating_upper_bound,rating_band_parse_status
0,2015-06-10,Brighton,2015-06-10T14:20:00+01:00,Genting Casino Brighton Maiden Stakes,Flat,Class 5,3yo,,NaN,NaN,blank
1,2015-06-10,Brighton,2015-06-10T14:50:00+01:00,Thistle Hotel Brighton Handicap,Flat,Class 5,4yo+,0-70,0.0,70.0,canonical
2,2015-06-10,Brighton,2015-06-10T15:20:00+01:00,Mears Group Handicap,Flat,Class 5,3yo,0-75,0.0,75.0,canonical
3,2015-06-10,Brighton,2015-06-10T15:50:00+01:00,James Ross Jewellers Handicap,Flat,Class 6,4yo+,0-55,0.0,55.0,canonical
4,2015-06-10,Brighton,2015-06-10T16:20:00+01:00,Star Sports Bet Handicap,Flat,Class 5,3yo,51-70,51.0,70.0,canonical
5,2015-06-10,Brighton,2015-06-10T16:50:00+01:00,Brighton & Hove Buses Free Raceday Shuttle Han...,Flat,Class 6,3yo+,0-55,0.0,55.0,canonical
6,2015-06-10,Brighton,2015-06-10T17:20:00+01:00,checkatrade.com Handicap,Flat,Class 5,3yo+,51-70,51.0,70.0,canonical
7,2015-06-15,Nottingham,NaN,Gary Birtles Handicap,Flat,Class 6,4yo+,0-60,0.0,60.0,canonical
8,2015-06-15,Nottingham,NaN,Des Walker Handicap,Flat,Class 4,3yo,66-85,66.0,85.0,canonical
9,2015-06-15,Nottingham,NaN,Jeff Whitefoot Shredall Median Auction Maiden ...,Flat,Class 5,2yo,,NaN,NaN,blank


## Evidence note 17 — the positive-lower Flat exceptions are race-level, not card-level

The full racecards containing the 15 positive-lower Flat rating bands were
inspected.

The result rules out a simple card-level source behaviour.

On the same racecards, Source Version 1 can contain all three forms:

- no rating band;
- a canonical `0-X` rating band;
- a canonical positive-lower `N-X` rating band.

### Examples

At Brighton on 10 June 2015 the same card contains:

- `0-70`;
- `0-75`;
- `0-55`;
- `51-70`;
- `0-55`;
- `51-70`.

At Leicester on 24 August 2015 the same card contains:

- blank;
- `0-70`;
- `61-75`;
- `0-80`;
- `56-70`;
- `66-80`.

At Newcastle on 18 December 2019 the same card contains:

- `61-75`;
- `0-75`;
- blank;
- `0-70`;
- `46-55`;
- `70-85`;
- `45-60`;
- `61-75`.

Therefore:

> **the positive-lower values are not caused by an entire racecard being
> represented through a different source format.**

The distinction exists at individual-race level.

### This also changes how `0-X` should be interpreted

The source clearly had the ability to distinguish `N-X` from `0-X` on the same
card.

Therefore it would be wrong to conclude that Source Version 1 always converted
Flat rating bands mechanically into `0-X`.

Some stored `0-X` values may represent genuine official `0-X` race conditions.

Others, as demonstrated by the 2026 BHA pilot, can represent races for which
the official BHA structured rating band actually has a positive lower boundary.

The problem is therefore more subtle:

> **Source Version 1's `rating_band` field cannot by itself tell us whether a
> Flat `0-X` value is an authoritative `0-X` condition or a partial
> representation that has lost the official lower boundary.**

### What has now been ruled out

The evidence does not support explaining the 15 exceptions by:

- race Class;
- age category;
- racecourse;
- a single historical period;
- or whole-card source behaviour.

The exact reason why these particular races retain their positive lower bounds
remains unresolved.

### Next bounded question

The Newcastle card of 18 December 2019 is especially useful because it contains
several positive-lower bands and two `0-X` Nursery bands on the same card.

> **What does the authoritative BHA structured record report for the rating
> bands on that card?**

If the BHA reproduces the same mixture, we will have direct evidence that some
historical Source Version 1 `0-X` values are genuine while the later 2026 pilot
shows that other `0-X` values have lost information.

That would establish precisely why the source field cannot be interpreted
without authoritative race-level validation.

In [16]:
# Test the mixed Newcastle card of 18 December 2019 against the BHA.
#
# Why this card matters:
#
# Database v4 contains both forms on the SAME Flat card:
#
#   positive-lower bands:
#       61-75, 46-55, 70-85, 45-60, 61-75
#
#   zero-lower bands:
#       0-75, 0-70
#
# The two 0-X races are Nurseries.
#
# This makes the card a useful historical test of whether Source Version 1
# sometimes preserved the official distinction correctly.
#
# The analysis proceeds in four stages:
#
#   1. ask the BHA for result-bearing fixtures on 18 December 2019;
#   2. identify the Newcastle fixture;
#   3. retrieve the BHA race list for that fixture;
#   4. reconcile BHA races with Database v4 using course-local advertised time
#      and compare the two rating-band values.
#
# We are not assuming that the BHA API necessarily reaches back to 2019.
# If no historical fixture is returned, that itself is evidence about the
# current API's historical availability and we stop rather than inventing a
# match.

historical_date = "2019-12-18"

historical_fixture_params = {
    "fromdate": historical_date,
    "todate": historical_date,
    "resultsAvailable": 1,
    "order": "asc",
    "page": 1,
    "per_page": 100,
}

historical_fixture_url = (
    "https://api09.horseracing.software/bha/v1/fixtures/?"
    + urlencode(historical_fixture_params)
)

historical_fixture_request = Request(
    historical_fixture_url,
    headers={
        "Authorization": authorization_value,
        "Accept": "application/json",
        "Origin": "https://www.britishhorseracing.com",
        "Referer": "https://www.britishhorseracing.com/",
        "User-Agent": USER_AGENT,
    },
)

with urlopen(historical_fixture_request, timeout=30) as response:
    historical_fixture_payload = json.loads(
        response.read().decode("utf-8")
    )

historical_fixtures = historical_fixture_payload["data"]

newcastle_fixtures = [
    fixture
    for fixture in historical_fixtures
    if "newcastle" in fixture["courseName"].lower()
]

print(
    "Result-bearing BHA fixtures returned:",
    len(historical_fixtures),
)

print(
    "Newcastle fixtures found:",
    len(newcastle_fixtures),
)

display(
    pd.DataFrame(
        [
            {
                "fixtureYear": fixture["fixtureYear"],
                "fixtureId": fixture["fixtureId"],
                "courseName": fixture["courseName"],
            }
            for fixture in newcastle_fixtures
        ]
    )
)

assert len(newcastle_fixtures) == 1, (
    "Expected exactly one Newcastle fixture. "
    "Inspect the returned fixture list before continuing."
)

newcastle_fixture = newcastle_fixtures[0]


# ---------------------------------------------------------------------------
# Retrieve every official BHA race belonging to that Newcastle fixture.
# ---------------------------------------------------------------------------

newcastle_races_url = (
    "https://api09.horseracing.software/bha/v1/fixtures/"
    f"{newcastle_fixture['fixtureYear']}/"
    f"{newcastle_fixture['fixtureId']}/races"
)

newcastle_races_request = Request(
    newcastle_races_url,
    headers={
        "Authorization": authorization_value,
        "Accept": "application/json",
        "Origin": "https://www.britishhorseracing.com",
        "Referer": "https://www.britishhorseracing.com/",
        "User-Agent": USER_AGENT,
    },
)

with urlopen(newcastle_races_request, timeout=30) as response:
    newcastle_races_payload = json.loads(
        response.read().decode("utf-8")
    )

bha_newcastle_2019 = pd.DataFrame(
    [
        {
            "race_date": race["raceDate"],
            "advertised_time": race["raceTime"],
            "bha_race_id": race["raceId"],
            "bha_race_name": race["raceName"],
            "bha_rating_band": race["ratingBand"],
            "bha_race_class": race["raceClass"],
        }
        for race in newcastle_races_payload["data"]
    ]
)


# ---------------------------------------------------------------------------
# Load the same card from Database v4.
#
# The governed advertised course-local timestamp is reduced to its local
# HH:MM:SS component so it can be matched directly with BHA `raceTime`.
# ---------------------------------------------------------------------------

with connect_read_only(DATABASE) as connection:
    v4_newcastle_2019 = pd.read_sql_query(
        """
        SELECT
            source_race_occurrence_code,
            raw_date AS race_date,
            substr(advertised_start_course_local, 12, 8) AS advertised_time,
            race_name_raw,
            rating_band_raw,
            class_number,
            age_band_raw
        FROM view_gb_reconciled_race_occurrences_with_racecourse
        WHERE
            raw_date = ?
            AND governed_racecourse_name = 'Newcastle'
        ORDER BY advertised_start_course_local
        """,
        connection,
        params=[historical_date],
    )


# ---------------------------------------------------------------------------
# Reconcile the official BHA card with Database v4 and display the values
# side by side.
#
# We use date + advertised local time here because the racecourse and date
# have already been fixed to this single Newcastle fixture.
# ---------------------------------------------------------------------------

newcastle_rating_comparison = bha_newcastle_2019.merge(
    v4_newcastle_2019,
    on=[
        "race_date",
        "advertised_time",
    ],
    how="outer",
    indicator=True,
)

newcastle_rating_comparison[
    [
        "advertised_time",
        "bha_race_name",
        "bha_rating_band",
        "rating_band_raw",
        "age_band_raw",
        "bha_race_class",
        "class_number",
        "_merge",
    ]
].sort_values("advertised_time")

Result-bearing BHA fixtures returned: 4
Newcastle fixtures found: 1


,fixtureYear,fixtureId,courseName
0,2019,12714,Newcastle


,advertised_time,bha_race_name,bha_rating_band,rating_band_raw,age_band_raw,bha_race_class,class_number,_merge
0,15:45:00,THE BOMBARDIER BRITISH HOPPED AMBER BEER HANDI...,61-75,61-75,3yo+,5,5,both
1,16:15:00,THE LADBROKES WHERE THE NATION PLAYS NURSERY H...,0-75,0-75,2yo,5,5,both
2,16:45:00,THE LADBROKES HOME OF THE ODDS BOOST NOVICE ME...,,,2yo,6,6,both
3,17:15:00,"THE LADBROKES ""PLAY 1-2 FREE"" ON FOOTBALL NURS...",0-70,0-70,2yo,5,5,both
4,17:45:00,THE BOMBARDIER GOLDEN BEER HANDICAP STAKES (CL...,46-55,46-55,3yo+,6,6,both
5,18:15:00,"THE BOMBARDIER ""MARCH TO YOUR OWN DRUM"" HANDIC...",71-85,70-85,3yo+,4,4,both
6,18:45:00,THE BETWAY CASINO HANDICAP STAKES (CLASS 6),46-60,45-60,3yo+,6,6,both
7,19:15:00,THE #BETYOURWAY AT BETWAY HANDICAP STAKES (CLA...,61-75,61-75,3yo+,5,6,both


In [17]:
# Test whether Source Version 1's race-level `rating_band` may have been
# influenced by the ratings of the horses appearing in the race.
#
# Why this test matters:
#
# For Newcastle on 18 December 2019 the BHA says:
#
#   18:15  official band = 71-85
#          source band   = 70-85
#
#   18:45  official band = 46-60
#          source band   = 45-60
#
# One possible explanation is that the source allowed the rating of an
# actual participant to alter the lower edge of the race-level band.
#
# We therefore compare, for every race on that card:
#
#   - the authoritative BHA race band already retrieved;
#   - Source Version 1's stored race-level rating_band;
#   - the minimum and maximum runner-level source `or` values.
#
# `or` is a runner-level source rating field governed by Notebook 18.
#
# We use raw date + course + off ONLY to locate the exact immutable source
# race records. It is not being used here as a sporting race-time measure.

from pathlib import Path
import sqlite3


# ---------------------------------------------------------------------------
# Recover the authorised Source Version 1 identity for each Newcastle race
# from Database v4.
# ---------------------------------------------------------------------------

with connect_read_only(DATABASE) as connection:
    newcastle_source_identities = pd.read_sql_query(
        """
        SELECT
            source_race_occurrence_code,
            raw_date,
            raw_course,
            raw_off,
            substr(advertised_start_course_local, 12, 8) AS advertised_time,
            race_name_raw,
            rating_band_raw
        FROM view_gb_reconciled_race_occurrences_with_racecourse
        WHERE
            raw_date = '2019-12-18'
            AND governed_racecourse_name = 'Newcastle'
        ORDER BY advertised_start_course_local
        """,
        connection,
    )


# ---------------------------------------------------------------------------
# Open immutable Source Version 1 read-only.
#
# We query only the eight exact source races identified above and retain
# each horse's raw `or` value so the source evidence remains visible.
# ---------------------------------------------------------------------------

project_root = Path(DATABASE).resolve().parents[4]

source_database = (
    project_root
    / "data"
    / "raw"
    / "form_2015-present"
    / "form_2015-present"
    / "raceform.db"
)

assert source_database.exists()

target_values = ", ".join(
    ["(?, ?, ?)"] * len(newcastle_source_identities)
)

target_params = []

for row in newcastle_source_identities.itertuples(index=False):
    target_params.extend(
        [
            row.raw_date,
            row.raw_course,
            row.raw_off,
        ]
    )

source_connection = sqlite3.connect(
    f"file:{source_database}?mode=ro",
    uri=True,
)

source_runner_ratings = pd.read_sql_query(
    f"""
    WITH targets(source_date, source_course, source_off) AS (
        VALUES {target_values}
    )

    SELECT
        d.date AS raw_date,
        d.course AS raw_course,
        d.off AS raw_off,
        d.race_name,
        d.rating_band,
        d.horse,
        d.pos,
        d."or" AS or_raw

    FROM data AS d

    INNER JOIN targets AS t
        ON t.source_date = d.date
       AND t.source_course = d.course
       AND t.source_off = d.off

    WHERE d.rowid <> 1

    ORDER BY
        d.date,
        d.course,
        d.off,
        d.num
    """,
    source_connection,
    params=target_params,
)

source_connection.close()


# ---------------------------------------------------------------------------
# Convert usable runner OR values to numbers for the bounded comparison.
#
# The untouched source value remains in `or_raw`; conversion failures become
# missing rather than being silently assigned a rating.
# ---------------------------------------------------------------------------

source_runner_ratings["or_numeric"] = pd.to_numeric(
    source_runner_ratings["or_raw"],
    errors="coerce",
)


# ---------------------------------------------------------------------------
# Reduce the runner rows back to one row per source race.
#
# This tells us the lowest and highest runner rating represented in the
# source for each race, without treating those extrema as the race condition.
# ---------------------------------------------------------------------------

source_rating_extrema = (
    source_runner_ratings
    .groupby(
        [
            "raw_date",
            "raw_course",
            "raw_off",
            "race_name",
            "rating_band",
        ],
        dropna=False,
    )
    .agg(
        runners=("horse", "size"),
        runners_with_or=("or_numeric", "count"),
        minimum_runner_or=("or_numeric", "min"),
        maximum_runner_or=("or_numeric", "max"),
    )
    .reset_index()
)


# ---------------------------------------------------------------------------
# Attach governed advertised time, then attach the BHA rating band from the
# historical reconciliation we have already completed.
# ---------------------------------------------------------------------------

source_rating_extrema = source_rating_extrema.merge(
    newcastle_source_identities[
        [
            "raw_date",
            "raw_course",
            "raw_off",
            "advertised_time",
        ]
    ],
    on=[
        "raw_date",
        "raw_course",
        "raw_off",
    ],
    how="left",
    validate="one_to_one",
)

rating_derivation_test = source_rating_extrema.merge(
    newcastle_rating_comparison[
        [
            "advertised_time",
            "bha_rating_band",
        ]
    ],
    on="advertised_time",
    how="left",
    validate="one_to_one",
)


# ---------------------------------------------------------------------------
# Show the prospective BHA condition, the source's race-level value and the
# realised runner-rating range side by side.
#
# The two key rows are 18:15 and 18:45.
# ---------------------------------------------------------------------------

rating_derivation_test[
    [
        "advertised_time",
        "race_name",
        "bha_rating_band",
        "rating_band",
        "minimum_runner_or",
        "maximum_runner_or",
        "runners",
        "runners_with_or",
    ]
].sort_values("advertised_time")

,advertised_time,race_name,bha_rating_band,rating_band,minimum_runner_or,maximum_runner_or,runners,runners_with_or
0,15:45:00,Bombardier British Hopped Amber Beer Handicap,61-75,61-75,65.0,74.0,14,14
1,16:15:00,Ladbrokes Where The Nation Plays Nursery Handicap,0-75,0-75,62.0,75.0,9,9
2,16:45:00,Ladbrokes Home Of The Odds Boost Novice Median...,,,73.0,82.0,9,3
3,17:15:00,Ladbrokes Play 1-2 Free On Football Nursery Ha...,0-70,0-70,49.0,69.0,14,14
4,17:45:00,Bombardier Golden Beer Handicap,46-55,46-55,46.0,55.0,14,14
5,18:15:00,Bombardier March To Your Own Drum Handicap,71-85,70-85,75.0,86.0,11,11
6,18:45:00,Betway Casino Handicap,46-60,45-60,45.0,61.0,11,11
7,19:15:00,#betyourway At Betway Handicap,61-75,61-75,68.0,75.0,13,13


In [18]:
# Test whether the source rating-band discrepancies can be explained by
# the official ratings of the horses that appeared in those races.
#
# The Newcastle card gives us two known discrepancies:
#
#   BHA 71-85  versus source 70-85
#   BHA 46-60  versus source 45-60
#
# The hypothesis is that the source race-level band may sometimes have been
# influenced by a runner's own official rating.
#
# We test all eight races on the card, not just the two discrepancies, so
# the other six races act as controls.
#
# Identification proceeds from the BHA race time:
#
#   BHA raceTime
#       -> governed advertised course-local time in Database v4
#       -> source_race_occurrence_code
#       -> governed runner records
#
# Runner `or_governed` is the official pre-race handicap mark applicable
# to that runner for this race, as established by Notebook 18.


# ---------------------------------------------------------------------------
# Start from the BHA card we already retrieved.
#
# `advertised_time` here came directly from BHA `raceTime`.
# Keep the BHA rating band beside it because that is the authoritative
# prospective race condition we are testing against.
# ---------------------------------------------------------------------------

bha_newcastle_test = (
    newcastle_rating_comparison[
        [
            "race_date",
            "advertised_time",
            "bha_race_name",
            "bha_rating_band",
            "rating_band_raw",
        ]
    ]
    .copy()
)


# ---------------------------------------------------------------------------
# Identify the corresponding Database v4 race occurrence using the governed
# course-local advertised start time.
#
# The date and racecourse fix the Newcastle card; the BHA clock time then
# identifies each individual race on that card.
# ---------------------------------------------------------------------------

with connect_read_only(DATABASE) as connection:
    v4_newcastle_races = pd.read_sql_query(
        """
        SELECT
            source_race_occurrence_code,
            raw_date AS race_date,
            substr(advertised_start_course_local, 12, 8) AS advertised_time,
            race_name_raw
        FROM view_gb_reconciled_race_occurrences_with_racecourse
        WHERE
            raw_date = '2019-12-18'
            AND governed_racecourse_name = 'Newcastle'
        ORDER BY advertised_start_course_local
        """,
        connection,
    )

race_matches = bha_newcastle_test.merge(
    v4_newcastle_races,
    on=[
        "race_date",
        "advertised_time",
    ],
    how="left",
    validate="one_to_one",
)

assert len(race_matches) == 8
assert race_matches["source_race_occurrence_code"].notna().all()


# ---------------------------------------------------------------------------
# Retrieve the runner-level official ratings for those exact eight race
# occurrences.
#
# Only ratings whose governed status is `available` are used when calculating
# the minimum and maximum runner rating. Missing ratings remain missing rather
# than being treated as zero or inferred.
# ---------------------------------------------------------------------------

race_codes = race_matches[
    "source_race_occurrence_code"
].tolist()

placeholders = ", ".join("?" for _ in race_codes)

with connect_read_only(DATABASE) as connection:
    runner_ratings = pd.read_sql_query(
        f"""
        SELECT
            source_race_occurrence_code,
            raw_horse AS horse,
            raw_or,
            or_governed,
            or_status
        FROM view_reconciled_source_runner_participations
        WHERE source_race_occurrence_code IN ({placeholders})
        ORDER BY
            source_race_occurrence_code,
            or_governed,
            raw_horse
        """,
        connection,
        params=race_codes,
    )


# ---------------------------------------------------------------------------
# Summarise the realised runner-rating range for each race.
#
# This does NOT redefine the race's rating band from its runners.
# It simply lets us test whether the unusual source boundary happens to
# coincide with one of the official ratings present in the realised field.
# ---------------------------------------------------------------------------

available_runner_ratings = runner_ratings[
    runner_ratings["or_status"] == "available"
].copy()

runner_rating_summary = (
    available_runner_ratings
    .groupby(
        "source_race_occurrence_code",
        as_index=False,
    )
    .agg(
        runners_with_or=("or_governed", "count"),
        minimum_runner_or=("or_governed", "min"),
        maximum_runner_or=("or_governed", "max"),
        runner_ors=(
            "or_governed",
            lambda values: ", ".join(
                str(int(value))
                for value in sorted(values)
            ),
        ),
    )
)


# ---------------------------------------------------------------------------
# Put prospective race condition and realised runner ratings side by side.
#
# The key questions are:
#
#   - does the 18:15 source lower bound of 70 coincide with a runner OR of 70?
#   - does the 18:45 source lower bound of 45 coincide with a runner OR of 45?
#
# The six agreement races tell us whether such coincidences are normal or
# distinctive to the discrepant cases.
# ---------------------------------------------------------------------------

rating_band_runner_test = (
    race_matches
    .merge(
        runner_rating_summary,
        on="source_race_occurrence_code",
        how="left",
        validate="one_to_one",
    )
    [
        [
            "advertised_time",
            "bha_race_name",
            "bha_rating_band",
            "rating_band_raw",
            "minimum_runner_or",
            "maximum_runner_or",
            "runner_ors",
            "runners_with_or",
        ]
    ]
    .sort_values("advertised_time")
    .reset_index(drop=True)
)

rating_band_runner_test

,advertised_time,bha_race_name,bha_rating_band,rating_band_raw,minimum_runner_or,maximum_runner_or,runner_ors,runners_with_or
0,15:45:00,THE BOMBARDIER BRITISH HOPPED AMBER BEER HANDI...,61-75,61-75,65.0,74.0,"65, 67, 70, 70, 70, 70, 71, 72, 73, 73, 73, 73...",14
1,16:15:00,THE LADBROKES WHERE THE NATION PLAYS NURSERY H...,0-75,0-75,62.0,75.0,"62, 63, 63, 65, 67, 67, 71, 71, 75",9
2,16:45:00,THE LADBROKES HOME OF THE ODDS BOOST NOVICE ME...,,,73.0,82.0,"73, 74, 82",3
3,17:15:00,"THE LADBROKES ""PLAY 1-2 FREE"" ON FOOTBALL NURS...",0-70,0-70,49.0,69.0,"49, 51, 59, 64, 64, 65, 65, 66, 66, 66, 68, 69...",14
4,17:45:00,THE BOMBARDIER GOLDEN BEER HANDICAP STAKES (CL...,46-55,46-55,46.0,55.0,"46, 46, 47, 48, 49, 50, 50, 52, 52, 53, 54, 54...",14
5,18:15:00,"THE BOMBARDIER ""MARCH TO YOUR OWN DRUM"" HANDIC...",71-85,70-85,75.0,86.0,"75, 76, 78, 78, 81, 82, 84, 84, 84, 85, 86",11
6,18:45:00,THE BETWAY CASINO HANDICAP STAKES (CLASS 6),46-60,45-60,45.0,61.0,"45, 46, 47, 50, 55, 56, 58, 58, 59, 61, 61",11
7,19:15:00,THE #BETYOURWAY AT BETWAY HANDICAP STAKES (CLA...,61-75,61-75,68.0,75.0,"68, 70, 70, 70, 70, 71, 72, 72, 73, 74, 75, 75...",13


In [19]:
# Test whether the 15 exceptional Flat races with a positive stored lower
# rating-band boundary are associated with runners rated ABOVE the stored
# upper boundary.
#
# Why we are doing this:
#
# On the Newcastle 18 December 2019 card, the ONLY two races where Source
# Version 1 disagreed with the BHA rating band were also the only two races
# containing a horse rated 1 lb above the official BHA upper boundary.
#
# That raises a specific hypothesis:
#
#   perhaps the unusual source representation is associated with the
#   realised handicap/weight situation when an above-band horse participates.
#
# We are NOT assuming that hypothesis is true.
#
# The 15 positive-lower Flat races give us a small, exhaustive historical
# population in which to test whether above-band runners are common.
#
# Race identification uses only the durable project-owned
# `source_race_occurrence_code`.
#
# Runner `or_governed` is the official pre-race handicap mark applicable
# to that horse for the race.


# ---------------------------------------------------------------------------
# Get the exact 15 race occurrence codes and their stored rating bands.
# ---------------------------------------------------------------------------

positive_lower_races = flat_positive_lower[
    [
        "source_race_occurrence_code",
        "race_date",
        "governed_racecourse_name",
        "advertised_start_course_local",
        "race_name_raw",
        "rating_band_raw",
        "rating_lower_bound",
        "rating_upper_bound",
    ]
].copy()

race_codes = positive_lower_races[
    "source_race_occurrence_code"
].tolist()

placeholders = ", ".join("?" for _ in race_codes)


# ---------------------------------------------------------------------------
# Retrieve governed runner ORs for those exact races.
#
# Only `available` OR values are used in the numerical comparison.
# ---------------------------------------------------------------------------

with connect_read_only(DATABASE) as connection:
    positive_lower_runner_ratings = pd.read_sql_query(
        f"""
        SELECT
            source_race_occurrence_code,
            raw_horse AS horse,
            or_governed,
            or_status
        FROM view_reconciled_source_runner_participations
        WHERE source_race_occurrence_code IN ({placeholders})
        ORDER BY
            source_race_occurrence_code,
            or_governed,
            raw_horse
        """,
        connection,
        params=race_codes,
    )

available_positive_lower_or = positive_lower_runner_ratings[
    positive_lower_runner_ratings["or_status"] == "available"
].copy()


# ---------------------------------------------------------------------------
# Reduce the runner rows to one row per race.
#
# We retain:
#
#   - minimum runner OR;
#   - maximum runner OR;
#   - the complete sorted OR list;
#   - number of runners with usable OR.
#
# The maximum is the key value for the current hypothesis.
# ---------------------------------------------------------------------------

positive_lower_or_summary = (
    available_positive_lower_or
    .groupby(
        "source_race_occurrence_code",
        as_index=False,
    )
    .agg(
        runners_with_or=("or_governed", "count"),
        minimum_runner_or=("or_governed", "min"),
        maximum_runner_or=("or_governed", "max"),
        runner_ors=(
            "or_governed",
            lambda values: ", ".join(
                str(int(value))
                for value in sorted(values)
            ),
        ),
    )
)


# ---------------------------------------------------------------------------
# Attach the stored race-band boundaries.
#
# Then calculate how far the highest-rated runner sits above or below the
# stored upper boundary.
#
# Examples:
#
#   0  = highest-rated runner equals the stored upper boundary
#   1  = a +1 runner is present
#   2  = a +2 runner is present
#  -3  = highest-rated runner is 3 lb below the stored upper boundary
# ---------------------------------------------------------------------------

positive_lower_runner_test = (
    positive_lower_races
    .merge(
        positive_lower_or_summary,
        on="source_race_occurrence_code",
        how="left",
        validate="one_to_one",
    )
)

positive_lower_runner_test["max_or_minus_upper"] = (
    positive_lower_runner_test["maximum_runner_or"]
    - positive_lower_runner_test["rating_upper_bound"]
)

positive_lower_runner_test["above_stored_upper"] = (
    positive_lower_runner_test["max_or_minus_upper"] > 0
)


# ---------------------------------------------------------------------------
# Display every one of the 15 races.
# ---------------------------------------------------------------------------

positive_lower_runner_test[
    [
        "race_date",
        "governed_racecourse_name",
        "advertised_start_course_local",
        "race_name_raw",
        "rating_band_raw",
        "minimum_runner_or",
        "maximum_runner_or",
        "max_or_minus_upper",
        "above_stored_upper",
        "runner_ors",
        "runners_with_or",
    ]
].sort_values(
    [
        "race_date",
        "governed_racecourse_name",
        "advertised_start_course_local",
    ]
)

,race_date,governed_racecourse_name,advertised_start_course_local,race_name_raw,rating_band_raw,minimum_runner_or,maximum_runner_or,max_or_minus_upper,above_stored_upper,runner_ors,runners_with_or
0,2015-06-10,Brighton,2015-06-10T16:20:00+01:00,Star Sports Bet Handicap,51-70,62,69,-1,False,"62, 63, 64, 67, 67, 69, 69",7
1,2015-06-10,Brighton,2015-06-10T17:20:00+01:00,checkatrade.com Handicap,51-70,56,72,2,True,"56, 63, 64, 64, 69, 72",6
2,2015-06-15,Nottingham,NaN,Des Walker Handicap,66-85,71,78,-7,False,"71, 72, 77, 78",4
3,2015-08-24,Leicester,2015-08-24T18:25:00+01:00,John Smiths Extra Smooth Handicap,61-75,64,75,0,False,"64, 67, 69, 72, 73, 75, 75, 75",8
4,2015-08-24,Leicester,2015-08-24T19:25:00+01:00,Strongbow Cloudy Apple Handicap,56-70,56,69,-1,False,"56, 58, 59, 60, 65, 69",6
5,2015-08-24,Leicester,2015-08-24T19:55:00+01:00,Foxton Locks Inn Handicap,66-80,69,78,-2,False,"69, 74, 75, 76, 76, 76, 78, 78",8
6,2015-09-05,Ascot,2015-09-05T16:35:00+01:00,Bibendum Wine Ltd Handicap,86-105,87,103,-2,False,"87, 87, 89, 90, 90, 93, 94, 95, 96, 96, 97, 97...",17
7,2015-09-10,Epsom Downs,2015-09-10T16:25:00+01:00,JRA Handicap,61-75,61,75,0,False,"61, 65, 65, 68, 72, 74, 74, 75",8
8,2015-10-03,Ascot,2015-10-03T16:55:00+01:00,AP Security Handicap,81-95,80,94,-1,False,"80, 81, 82, 83, 84, 85, 85, 86, 87, 87, 87, 87...",16
9,2015-10-07,Nottingham,2015-10-07T16:10:00+01:00,£10 Free At 32Red.com Handicap,59-73,67,73,0,False,"67, 67, 69, 70, 70, 70, 72, 72, 72, 73, 73, 73",12


In [20]:
# Decisive control test for the "above-band runner changes source band" idea.
#
# Brighton 17:20 on 10 June 2015 is stored as 51-70 in Source Version 1,
# but the realised field contains a horse rated 72 (+2 above that upper bound).
#
# If BHA also says the official race was 51-70, then the presence of an
# above-band runner clearly does NOT itself cause Source Version 1 to alter
# the race's rating band.
#
# We identify the race using the BHA's own raceTime.

historical_date = "2015-06-10"

fixture_params = {
    "fromdate": historical_date,
    "todate": historical_date,
    "resultsAvailable": 1,
    "order": "asc",
    "page": 1,
    "per_page": 100,
}

fixture_url = (
    "https://api09.horseracing.software/bha/v1/fixtures/?"
    + urlencode(fixture_params)
)

fixture_request = Request(
    fixture_url,
    headers={
        "Authorization": authorization_value,
        "Accept": "application/json",
        "Origin": "https://www.britishhorseracing.com",
        "Referer": "https://www.britishhorseracing.com/",
        "User-Agent": USER_AGENT,
    },
)

with urlopen(fixture_request, timeout=30) as response:
    fixture_payload = json.loads(
        response.read().decode("utf-8")
    )

brighton_fixtures = [
    fixture
    for fixture in fixture_payload["data"]
    if "brighton" in fixture["courseName"].lower()
]

print("Brighton fixtures found:", len(brighton_fixtures))

assert len(brighton_fixtures) == 1

brighton_fixture = brighton_fixtures[0]


# Retrieve the official BHA races on that fixture.

races_url = (
    "https://api09.horseracing.software/bha/v1/fixtures/"
    f"{brighton_fixture['fixtureYear']}/"
    f"{brighton_fixture['fixtureId']}/races"
)

races_request = Request(
    races_url,
    headers={
        "Authorization": authorization_value,
        "Accept": "application/json",
        "Origin": "https://www.britishhorseracing.com",
        "Referer": "https://www.britishhorseracing.com/",
        "User-Agent": USER_AGENT,
    },
)

with urlopen(races_request, timeout=30) as response:
    races_payload = json.loads(
        response.read().decode("utf-8")
    )

bha_brighton = pd.DataFrame(
    [
        {
            "advertised_time": race["raceTime"],
            "bha_race_name": race["raceName"],
            "bha_rating_band": race["ratingBand"],
            "bha_race_class": race["raceClass"],
        }
        for race in races_payload["data"]
    ]
)

bha_brighton[
    bha_brighton["advertised_time"] == "17:20:00"
]

Brighton fixtures found: 1


,advertised_time,bha_race_name,bha_rating_band,bha_race_class
6,17:20:00,THE checkatrade.com HANDICAP STAKES (CLASS 5),51-70,5


### Evidence note — realised runner ratings do not explain the source discrepancies

A possible explanation for the two Newcastle rating-band disagreements was
that Source Version 1 had modified the race-level band in response to horses
running above the official upper boundary.

That hypothesis was tested against the 15 Flat races in Database v4 that
retain a positive lower rating-band boundary.

Only three of the 15 contained a runner rated above the stored upper boundary:

- Brighton, 10 June 2015, 17:20: stored `51-70`, maximum runner OR `72`;
- Newcastle, 18 December 2019, 18:15: stored `70-85`, maximum runner OR `86`;
- Newcastle, 18 December 2019, 18:45: stored `45-60`, maximum runner OR `61`.

The Brighton race provides a decisive control.

The authoritative BHA structured record reports the Brighton 17:20 race as
`51-70`, exactly matching Source Version 1, despite a horse rated 72 competing
in the race.

Therefore:

> **The presence of a horse rated above the published rating band does not
> itself cause Source Version 1 to alter the stored race-level rating band.**

The earlier hypothesis that the Newcastle one-pound discrepancies were caused
by above-band runners is rejected.

The two Newcastle cases should instead be treated as upstream Source Version 1
discrepancies of unresolved origin:

- BHA `71-85` versus source `70-85`;
- BHA `46-60` versus source `45-60`.

Database v4 faithfully preserves those source values and is not responsible
for introducing the discrepancy.

The broader conceptual finding remains important:

> **The ratings of the horses that ultimately participate do not define the
> prospective rating band of the race.**

A horse may compete outside the principal published rating range where the
applicable handicap rules permit it, while the programmed race condition
remains unchanged.

In [21]:
# First bounded test of what Source Version 1's `rating_band` actually represents.
#
# We have manually verified four Racing Post race pages from 27 May 2026.
# On each page Racing Post presents TWO different rating-band representations:
#
#   headline shorthand       e.g. 0-75
#   detailed race condition  e.g. 56-75
#
# We now compare both with:
#
#   - the authoritative BHA structured `ratingBand`;
#   - Database v4's preserved Source Version 1 `rating_band_raw`.
#
# This is deliberately a small provenance-controlled sample before we attempt
# any wider historical publication comparison.
#
# Race matching uses BHA raceTime and the governed course-local advertised
# start time. No source off-time field is involved.


from urllib.parse import urlencode
from urllib.request import Request, urlopen
import json
import pandas as pd


# ---------------------------------------------------------------------------
# Manually verified Racing Post observations.
#
# These values come from the SAME Racing Post page for each race:
#
#   `rp_headline_band`   = compact band printed in the race heading
#   `rp_conditions_band` = "Rated N-X" inside the detailed race conditions
#
# The page URL is retained as study provenance.
# ---------------------------------------------------------------------------

rp_rating_band_evidence = pd.DataFrame(
    [
        {
            "race_date": "2026-05-27",
            "governed_racecourse_name": "Beverley",
            "advertised_time": "15:15:00",
            "rp_headline_band": "0-75",
            "rp_conditions_band": "56-75",
            "rp_url": (
                "https://www.racingpost.com/racecards/6/"
                "beverley/2026-05-27/918962/raceday-live/"
            ),
        },
        {
            "race_date": "2026-05-27",
            "governed_racecourse_name": "Beverley",
            "advertised_time": "15:45:00",
            "rp_headline_band": "0-75",
            "rp_conditions_band": "56-75",
            "rp_url": (
                "https://www.racingpost.com/racecards/6/"
                "beverley/2026-05-27/918960/raceday-live/"
            ),
        },
        {
            "race_date": "2026-05-27",
            "governed_racecourse_name": "Hamilton Park",
            "advertised_time": "15:35:00",
            "rp_headline_band": "0-80",
            "rp_conditions_band": "61-80",
            "rp_url": (
                "https://www.racingpost.com/racecards/22/"
                "hamilton/2026-05-27/918969/raceday-live/"
            ),
        },
        {
            "race_date": "2026-05-27",
            "governed_racecourse_name": "Kempton Park",
            "advertised_time": "17:25:00",
            "rp_headline_band": "0-70",
            "rp_conditions_band": "51-70",
            "rp_url": (
                "https://www.racingpost.com/racecards/1079/"
                "kempton-aw/2026-05-27/919104/raceday-live/"
            ),
        },
    ]
)


# ---------------------------------------------------------------------------
# Retrieve BHA's authoritative structured race records for the same date.
# ---------------------------------------------------------------------------

fixture_url = (
    "https://api09.horseracing.software/bha/v1/fixtures/?"
    + urlencode(
        {
            "fromdate": "2026-05-27",
            "todate": "2026-05-27",
            "resultsAvailable": 1,
            "order": "asc",
            "page": 1,
            "per_page": 100,
        }
    )
)

request_headers = {
    "Authorization": authorization_value,
    "Accept": "application/json",
    "Origin": "https://www.britishhorseracing.com",
    "Referer": "https://www.britishhorseracing.com/",
    "User-Agent": USER_AGENT,
}

with urlopen(
    Request(fixture_url, headers=request_headers),
    timeout=30,
) as response:
    fixture_payload = json.loads(response.read().decode("utf-8"))


target_bha_courses = {
    "Beverley": "Beverley",
    "Hamilton Park": "Hamilton Park",
    "Kempton Park": "Kempton Park",
}

bha_rows = []

for fixture in fixture_payload["data"]:

    bha_course_name = fixture["courseName"]

    if bha_course_name not in target_bha_courses:
        continue

    races_url = (
        "https://api09.horseracing.software/bha/v1/fixtures/"
        f"{fixture['fixtureYear']}/{fixture['fixtureId']}/races"
    )

    with urlopen(
        Request(races_url, headers=request_headers),
        timeout=30,
    ) as response:
        races_payload = json.loads(response.read().decode("utf-8"))

    for race in races_payload["data"]:

        bha_rows.append(
            {
                "race_date": "2026-05-27",
                "governed_racecourse_name": target_bha_courses[bha_course_name],
                "advertised_time": str(race["raceTime"])[:5] + ":00",
                "bha_race_name": race["raceName"],
                "bha_rating_band": race["ratingBand"],
            }
        )

bha_rating_bands = pd.DataFrame(bha_rows)


# ---------------------------------------------------------------------------
# Retrieve Source Version 1's preserved rating-band values from Database v4.
#
# Individual race matching uses the already governed advertised course-local
# time, corresponding to BHA raceTime.
# ---------------------------------------------------------------------------

with connect_read_only(DATABASE) as connection:
    v4_rating_bands = pd.read_sql_query(
        """
        SELECT
            raw_date AS race_date,
            governed_racecourse_name,
            substr(advertised_start_course_local, 12, 8) AS advertised_time,
            race_name_raw,
            rating_band_raw
        FROM view_gb_reconciled_race_occurrences_with_racecourse
        WHERE raw_date = '2026-05-27'
          AND governed_racecourse_name IN (
              'Beverley',
              'Hamilton Park',
              'Kempton Park'
          )
        """,
        connection,
    )


# ---------------------------------------------------------------------------
# Join the four independently represented versions of each race.
# ---------------------------------------------------------------------------

comparison = (
    rp_rating_band_evidence
    .merge(
        bha_rating_bands,
        on=[
            "race_date",
            "governed_racecourse_name",
            "advertised_time",
        ],
        how="left",
        validate="one_to_one",
    )
    .merge(
        v4_rating_bands,
        on=[
            "race_date",
            "governed_racecourse_name",
            "advertised_time",
        ],
        how="left",
        validate="one_to_one",
    )
)

assert comparison["bha_rating_band"].notna().all()
assert comparison["rating_band_raw"].notna().all()


# ---------------------------------------------------------------------------
# Explicitly test which publication representation Source Version 1 matches.
# ---------------------------------------------------------------------------

comparison["source_matches_bha"] = (
    comparison["rating_band_raw"]
    == comparison["bha_rating_band"]
)

comparison["source_matches_rp_headline"] = (
    comparison["rating_band_raw"]
    == comparison["rp_headline_band"]
)

comparison["source_matches_rp_conditions"] = (
    comparison["rating_band_raw"]
    == comparison["rp_conditions_band"]
)


comparison[
    [
        "governed_racecourse_name",
        "advertised_time",
        "bha_rating_band",
        "rp_headline_band",
        "rp_conditions_band",
        "rating_band_raw",
        "source_matches_bha",
        "source_matches_rp_headline",
        "source_matches_rp_conditions",
    ]
]

,governed_racecourse_name,advertised_time,bha_rating_band,rp_headline_band,rp_conditions_band,rating_band_raw,source_matches_bha,source_matches_rp_headline,source_matches_rp_conditions
0,Beverley,15:15:00,56-75,0-75,56-75,0-75,False,True,False
1,Beverley,15:45:00,56-75,0-75,56-75,0-75,False,True,False
2,Hamilton Park,15:35:00,61-80,0-80,61-80,0-80,False,True,False
3,Kempton Park,17:25:00,51-70,0-70,51-70,0-70,False,True,False


### Evidence note — Source Version 1 rating bands align with Racing Post headline shorthand in the first direct comparison

A four-race comparison was made between:

1. the authoritative BHA structured `ratingBand`;
2. the compact rating-band representation shown in the Racing Post race heading;
3. the detailed `Rated N-X` condition shown by Racing Post for the same race;
4. Source Version 1's preserved `rating_band_raw`.

The four races were from Beverley, Hamilton Park and Kempton Park on
27 May 2026.

Results:

- Source Version 1 matched the BHA structured rating band in 0/4 races.
- Source Version 1 matched the Racing Post headline representation in 4/4 races.
- Racing Post's detailed race condition matched the BHA structured rating band
  in all four cases.

Examples include:

- BHA `56-75` / Racing Post detailed `56-75`
  / Racing Post headline `0-75` / Source Version 1 `0-75`;
- BHA `61-80` / Racing Post detailed `61-80`
  / Racing Post headline `0-80` / Source Version 1 `0-80`;
- BHA `51-70` / Racing Post detailed `51-70`
  / Racing Post headline `0-70` / Source Version 1 `0-70`.

This establishes that a Source Version 1 `0-X` value cannot automatically be
interpreted as an erroneous attempt to reproduce the official BHA rating band.

Instead, the first direct publication comparison is consistent with the source
preserving a compact publication-style representation of the race.

The comparison does not establish that Source Version 1 was obtained from
Racing Post, nor that every source rating-band value follows Racing Post.
Those remain hypotheses requiring wider historical testing.

For British race-condition analysis, `rating_band_raw` therefore remains useful
source evidence but should not be treated as an authoritative representation
of the BHA programmed rating band.

### Evidence note — Source Version 1 rating-band values can reflect publication shorthand rather than the official BHA race condition

The rating-band investigation established an important distinction between the
formal BHA race condition and the compact representation used by Racing Post.

For several directly verified races on 27 May 2026, the BHA structured record
and the detailed Racing Post race conditions agreed on the formal rating band,
while the Racing Post race heading used a broader `0-X` shorthand.

Examples:

- BHA `56-75` / Racing Post detailed condition `56-75`
  / Racing Post headline `0-75`
  / Source Version 1 `0-75`;
- BHA `61-80` / Racing Post detailed condition `61-80`
  / Racing Post headline `0-80`
  / Source Version 1 `0-80`;
- BHA `51-70` / Racing Post detailed condition `51-70`
  / Racing Post headline `0-70`
  / Source Version 1 `0-70`.

Across the four directly compared races:

- Source Version 1 matched the BHA structured rating band in `0/4`;
- Source Version 1 matched the Racing Post headline representation in `4/4`;
- Racing Post's detailed race condition matched the BHA structured rating band
  in all four cases.

This does not establish that Source Version 1 was obtained from Racing Post,
but it demonstrates that its `rating_band_raw` field can preserve the same
publication-style representation.

The two forms describe different aspects of the race.

A BHA band such as `56-75` is the formal programmed handicap range. Horses
rated below the lower boundary may nevertheless be eligible to compete under
the applicable handicap conditions, and horses above the upper boundary may
also be eligible in specified circumstances.

Racing Post can summarise that wider eligibility in a compact heading such as
`0-75`, while still stating the formal `Rated 56-75` condition in the detailed
race information.

Therefore:

> **`rating_band_raw` must not automatically be interpreted as the official
> BHA programmed rating band. It is a preserved source representation whose
> semantics may reflect publication shorthand.**

This distinction also means that a source value such as `0-75` is inherently
ambiguous without authoritative race-condition evidence. It may represent:

1. a genuine BHA `0-75` programmed race; or
2. publication shorthand for a race whose formal BHA band has a positive lower
   boundary, such as `56-75`.

Consequently, the raw source value should continue to be preserved unchanged,
but analyses requiring the formal prospective race condition should use or
verify the BHA rating band rather than infer it from `rating_band_raw`.

The investigation also reinforces the broader Study 05 distinction:

> **The programmed rating band belongs to the race specification; the ratings
> of the horses that ultimately participate belong to the realised field and do
> not redefine that prospective race condition.**

In [22]:
# Examine how the BHA structured data represents the age condition of a race.
#
# Analytical question:
#
#   What values does BHA actually place in `ageLimit`, and how do those values
#   correspond to the full human-readable race names?
#
# We inspect the raw BHA representation before comparing it with Database v4:
#
#   BHA concept -> BHA structured representation -> Inside Rails representation
#
# The 27 May 2026 pilot date is useful because we already know all 34 BHA
# race occurrences on that date reconcile with Database v4.
#
# Full race names are displayed without truncation so we can compare the
# structured `ageLimit` value with the wording of the actual race condition.

from urllib.parse import urlencode
from urllib.request import Request, urlopen
import json
import pandas as pd


fixture_url = (
    "https://api09.horseracing.software/bha/v1/fixtures/?"
    + urlencode(
        {
            "fromdate": "2026-05-27",
            "todate": "2026-05-27",
            "resultsAvailable": 1,
            "order": "asc",
            "page": 1,
            "per_page": 100,
        }
    )
)

request_headers = {
    "Authorization": authorization_value,
    "Accept": "application/json",
    "Origin": "https://www.britishhorseracing.com",
    "Referer": "https://www.britishhorseracing.com/",
    "User-Agent": USER_AGENT,
}

with urlopen(
    Request(fixture_url, headers=request_headers),
    timeout=30,
) as response:
    fixture_payload = json.loads(response.read().decode("utf-8"))


bha_age_rows = []

for fixture in fixture_payload["data"]:

    races_url = (
        "https://api09.horseracing.software/bha/v1/fixtures/"
        f"{fixture['fixtureYear']}/{fixture['fixtureId']}/races"
    )

    with urlopen(
        Request(races_url, headers=request_headers),
        timeout=30,
    ) as response:
        races_payload = json.loads(response.read().decode("utf-8"))

    for race in races_payload["data"]:

        bha_age_rows.append(
            {
                "course": fixture["courseName"],
                "advertised_time": race["raceTime"],
                "race_type": race["raceCriteriaRaceType"],
                "age_limit": race["ageLimit"],
                "race_name": race["raceName"],
            }
        )


bha_age_sample = pd.DataFrame(bha_age_rows)

assert len(bha_age_sample) == 34


# Prevent long race names from being truncated in the notebook output.
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", None)


display(
    bha_age_sample
    .sort_values(
        [
            "age_limit",
            "course",
            "advertised_time",
        ],
        na_position="first",
    )
    [
        [
            "course",
            "advertised_time",
            "race_type",
            "age_limit",
            "race_name",
        ]
    ]
)

,course,advertised_time,race_type,age_limit,race_name
7,Beverley,14:15:00,FLAT,2YO,THE HAPPY BIRTHDAY JOE MCCABE CLAIMING STAKES (CLASS 6)
8,Beverley,14:45:00,FLAT,2YO,"THE TIGERS TRUST RESTRICTED NOVICE STAKES (CLASS 4) (for horses in Bands B, C and D) (GBB RACE)"
0,Hamilton Park,14:05:00,FLAT,2YO,THE WYVIS ROOFING EBF MAIDEN STAKES (CLASS 4) (Hamilton Park 2yo Series Qualifier) (GBB RACE)
21,Kempton Park,18:00:00,FLAT,2YO,THE TRY UNIBET'S NEW IMPROVED ACCA BOOSTS FILLIES' NOVICE STAKES (CLASS 4) (GBB RACE)
1,Hamilton Park,14:35:00,FLAT,3-5YO,THE ONE STOP ROOFING SUPPLIES LTD RESTRICTED NOVICE STAKES (CLASS 4) (for horses in Bands C and D) (GBB RACE)
2,Hamilton Park,15:05:00,FLAT,3YO,THE ICB (WATER PROOFING) HANDICAP STAKES (CLASS 4)
5,Hamilton Park,16:35:00,FLAT,3YO,THE FIXING POINT CLADDING FASTENERS HANDICAP STAKES (CLASS 5)
25,Kempton Park,20:00:00,FLAT,3YO,THE UNIBET SUPPORTS SAFER GAMBLING HANDICAP STAKES (CLASS 4)
22,Kempton Park,18:30:00,FLAT,3YO+,THE TRY UNIBET'S NEW SMARTVIEW RACECARDS NOVICE STAKES (CLASS 4) (DIV I) (GBB RACE)
23,Kempton Park,19:00:00,FLAT,3YO+,THE TRY UNIBET'S NEW SMARTVIEW RACECARDS NOVICE STAKES (CLASS 4) (DIV II) (GBB RACE)


In [23]:
# Compare the BHA structured age condition with Database v4's preserved
# Source Version 1 age-band representation.
#
# Analytical question:
#
#   Does `age_band_raw` represent the same prospective age eligibility
#   condition as BHA `ageLimit`?
#
# We match races using:
#
#   race date
#   governed racecourse identity
#   BHA raceTime -> governed advertised course-local start time
#
# We do not use any source off-time field.


# ---------------------------------------------------------------------------
# Prepare the BHA side of the comparison.
# ---------------------------------------------------------------------------

bha_age_comparison = bha_age_sample.rename(
    columns={
        "course": "governed_racecourse_name",
        "age_limit": "bha_age_limit",
        "race_name": "bha_race_name",
    }
).copy()

# Normalise BHA raceTime to HH:MM:SS for matching.
bha_age_comparison["advertised_time"] = (
    bha_age_comparison["advertised_time"]
    .astype(str)
    .str[:5]
    + ":00"
)


# ---------------------------------------------------------------------------
# Retrieve the corresponding v4 race-level records.
# ---------------------------------------------------------------------------

with connect_read_only(DATABASE) as connection:
    v4_age_bands = pd.read_sql_query(
        """
        SELECT
            raw_date AS race_date,
            governed_racecourse_name,
            substr(advertised_start_course_local, 12, 8) AS advertised_time,
            race_name_raw,
            age_band_raw
        FROM view_gb_reconciled_race_occurrences_with_racecourse
        WHERE raw_date = '2026-05-27'
        """,
        connection,
    )


# ---------------------------------------------------------------------------
# Restrict v4 to the five BHA fixtures in our already reconciled 34-race pilot.
# ---------------------------------------------------------------------------

target_courses = {
    "Beverley",
    "Hamilton Park",
    "Kempton Park",
    "Newton Abbot",
    "Cartmel",
}

v4_age_bands = v4_age_bands[
    v4_age_bands["governed_racecourse_name"].isin(target_courses)
].copy()


# ---------------------------------------------------------------------------
# Join BHA and v4 one race at a time.
# ---------------------------------------------------------------------------

age_comparison = bha_age_comparison.merge(
    v4_age_bands,
    on=[
        "governed_racecourse_name",
        "advertised_time",
    ],
    how="left",
    validate="one_to_one",
)


assert len(age_comparison) == 34
assert age_comparison["age_band_raw"].notna().all()


# ---------------------------------------------------------------------------
# Test exact agreement before attempting any interpretation or transformation.
# ---------------------------------------------------------------------------

age_comparison["exact_match"] = (
    age_comparison["bha_age_limit"]
    == age_comparison["age_band_raw"]
)


pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", None)

display(
    age_comparison[
        [
            "governed_racecourse_name",
            "advertised_time",
            "bha_age_limit",
            "age_band_raw",
            "exact_match",
            "bha_race_name",
        ]
    ].sort_values(
        [
            "exact_match",
            "governed_racecourse_name",
            "advertised_time",
        ]
    )
)

print()
print("Comparison summary:")
display(
    age_comparison["exact_match"]
    .value_counts(dropna=False)
    .rename_axis("exact_match")
    .reset_index(name="races")
)

,governed_racecourse_name,advertised_time,bha_age_limit,age_band_raw,exact_match,bha_race_name
7,Beverley,14:15:00,2YO,2yo,False,THE HAPPY BIRTHDAY JOE MCCABE CLAIMING STAKES (CLASS 6)
8,Beverley,14:45:00,2YO,2yo,False,"THE TIGERS TRUST RESTRICTED NOVICE STAKES (CLASS 4) (for horses in Bands B, C and D) (GBB RACE)"
9,Beverley,15:15:00,4YO+,4yo+,False,THE WARD HOMES YORKSHIRE 10TH ANNIVERSARY HANDICAP STAKES (CLASS 5)
10,Beverley,15:45:00,4YO+,4yo+,False,THE DR EDDIE MOLL HANDICAP STAKES (CLASS 5)
11,Beverley,16:15:00,4YO+,4yo+,False,THE CONNEXIN'S FULL FIBRE FOR ALL HANDICAP STAKES (CLASS 6)
12,Beverley,16:50:00,4YO+,4yo+,False,THE UP THE TIGERS HANDICAP STAKES (CLASS 6)
13,Beverley,17:20:00,4YO+,4yo+,False,THE RACING AGAIN THIS SATURDAY APPRENTICE HANDICAP STAKES (CLASS 6)
28,Cartmel,18:08:00,4YO+,4yo+,False,THE WILLIAM HILL MAIDEN HURDLE RACE (CLASS 4) (GBB RACE)
29,Cartmel,18:38:00,4YO+,4yo+,False,THE IN MEMORY OF JIMMY LATHAM SELLING HANDICAP HURDLE RACE (CLASS 5)
30,Cartmel,19:08:00,4YO+,4yo+,False,THE TRAFFIC MANAGEMENT HANDICAP HURDLE RACE (CLASS 4)



Comparison summary:


,exact_match,races
0,False,34


In [24]:
# Test semantic agreement after removing the purely presentational
# difference in capitalisation between BHA and Source Version 1.
#
# Examples:
#
#   BHA     4YO+
#   source  4yo+
#
# These are the same age condition, so case should not count as a
# substantive disagreement.

age_comparison["normalised_bha_age_limit"] = (
    age_comparison["bha_age_limit"]
    .astype(str)
    .str.strip()
    .str.lower()
)

age_comparison["normalised_source_age_band"] = (
    age_comparison["age_band_raw"]
    .astype(str)
    .str.strip()
    .str.lower()
)

age_comparison["semantic_match"] = (
    age_comparison["normalised_bha_age_limit"]
    == age_comparison["normalised_source_age_band"]
)

display(
    age_comparison[
        [
            "governed_racecourse_name",
            "advertised_time",
            "bha_age_limit",
            "age_band_raw",
            "semantic_match",
        ]
    ].sort_values(
        [
            "semantic_match",
            "governed_racecourse_name",
            "advertised_time",
        ]
    )
)

print()
print("Semantic comparison summary:")

display(
    age_comparison["semantic_match"]
    .value_counts(dropna=False)
    .rename_axis("semantic_match")
    .reset_index(name="races")
)

,governed_racecourse_name,advertised_time,bha_age_limit,age_band_raw,semantic_match
7,Beverley,14:15:00,2YO,2yo,True
8,Beverley,14:45:00,2YO,2yo,True
9,Beverley,15:15:00,4YO+,4yo+,True
10,Beverley,15:45:00,4YO+,4yo+,True
11,Beverley,16:15:00,4YO+,4yo+,True
12,Beverley,16:50:00,4YO+,4yo+,True
13,Beverley,17:20:00,4YO+,4yo+,True
28,Cartmel,18:08:00,4YO+,4yo+,True
29,Cartmel,18:38:00,4YO+,4yo+,True
30,Cartmel,19:08:00,4YO+,4yo+,True



Semantic comparison summary:


,semantic_match,races
0,True,34


In [25]:
# Inventory every age-band representation in the complete GB Database v4
# race population.
#
# Analytical question:
#
#   What prospective age-condition forms occur across British races from
#   2015 to the end of Database v4 coverage?
#
# The 34-race BHA pilot established semantic agreement between BHA `ageLimit`
# and Source Version 1 `age_band_raw` after normalising letter case.
#
# We now inspect the full GB population to determine whether additional age
# forms exist that were absent from that one-day sample.
#
# This is an inventory only: we do not infer the meaning of unfamiliar forms
# until we have seen what actually occurs.

with connect_read_only(DATABASE) as connection:
    gb_age_inventory = pd.read_sql_query(
        """
        SELECT
            CASE
                WHEN age_band_raw IS NULL
                     OR trim(age_band_raw) = ''
                    THEN '[blank]'
                ELSE trim(age_band_raw)
            END AS age_band_raw,
            COUNT(*) AS races,
            MIN(raw_date) AS earliest_date,
            MAX(raw_date) AS latest_date
        FROM view_gb_reconciled_race_occurrences_with_racecourse
        GROUP BY
            CASE
                WHEN age_band_raw IS NULL
                     OR trim(age_band_raw) = ''
                    THEN '[blank]'
                ELSE trim(age_band_raw)
            END
        ORDER BY races DESC, age_band_raw
        """,
        connection,
    )


print("Distinct age-band forms:", len(gb_age_inventory))
print("GB races represented:", gb_age_inventory["races"].sum())

display(gb_age_inventory)

Distinct age-band forms: 24
GB races represented: 111634


,age_band_raw,races,earliest_date,latest_date
0,4yo+,42663,2015-01-01,2026-05-27
1,3yo+,29554,2015-01-07,2026-05-27
2,2yo,12318,2015-03-28,2026-05-27
3,3yo,11876,2015-01-01,2026-05-27
4,5yo+,9259,2015-01-01,2026-05-27
5,4-6yo,2603,2015-01-01,2026-04-24
6,3-5yo,1076,2015-01-01,2026-05-27
7,3-4yo,505,2015-01-09,2026-05-04
8,4yo,499,2015-01-01,2026-05-22
9,4-5yo,419,2015-02-22,2026-05-26


In [26]:
# Classify the complete GB age-condition population into simple structural forms.
#
# Analytical question:
#
#   Can every Source Version 1 `age_band_raw` value be interpreted directly
#   as one of:
#
#       exact age       e.g. 3yo
#       bounded range   e.g. 3-5yo
#       minimum age     e.g. 4yo+
#
# This is a structural interpretation of the notation only. It does not add
# any sporting meaning beyond what is explicitly encoded in the source value.

import re


def parse_age_condition(value):
    value = value.strip().lower()

    # Exact age: 2yo, 3yo, 4yo
    match = re.fullmatch(r"(\d+)yo", value)
    if match:
        age = int(match.group(1))
        return pd.Series(
            {
                "age_condition_type": "exact",
                "minimum_age": age,
                "maximum_age": age,
            }
        )

    # Open-ended minimum age: 3yo+, 4yo+, 10yo+
    match = re.fullmatch(r"(\d+)yo\+", value)
    if match:
        return pd.Series(
            {
                "age_condition_type": "minimum_age",
                "minimum_age": int(match.group(1)),
                "maximum_age": pd.NA,
            }
        )

    # Bounded age range: 3-5yo, 4-6yo, etc.
    match = re.fullmatch(r"(\d+)-(\d+)yo", value)
    if match:
        return pd.Series(
            {
                "age_condition_type": "bounded_range",
                "minimum_age": int(match.group(1)),
                "maximum_age": int(match.group(2)),
            }
        )

    return pd.Series(
        {
            "age_condition_type": "unresolved",
            "minimum_age": pd.NA,
            "maximum_age": pd.NA,
        }
    )


parsed_age_inventory = pd.concat(
    [
        gb_age_inventory,
        gb_age_inventory["age_band_raw"].apply(parse_age_condition),
    ],
    axis=1,
)

display(parsed_age_inventory)

print()
print("Races by structural age-condition type:")

display(
    parsed_age_inventory
    .groupby("age_condition_type", dropna=False)["races"]
    .sum()
    .reset_index()
    .sort_values("races", ascending=False)
)

print()
print(
    "Unresolved age-band forms:",
    (parsed_age_inventory["age_condition_type"] == "unresolved").sum(),
)

,age_band_raw,races,earliest_date,latest_date,age_condition_type,minimum_age,maximum_age
0,4yo+,42663,2015-01-01,2026-05-27,minimum_age,4,<NA>
1,3yo+,29554,2015-01-07,2026-05-27,minimum_age,3,<NA>
2,2yo,12318,2015-03-28,2026-05-27,exact,2,2
3,3yo,11876,2015-01-01,2026-05-27,exact,3,3
4,5yo+,9259,2015-01-01,2026-05-27,minimum_age,5,<NA>
5,4-6yo,2603,2015-01-01,2026-04-24,bounded_range,4,6
6,3-5yo,1076,2015-01-01,2026-05-27,bounded_range,3,5
7,3-4yo,505,2015-01-09,2026-05-04,bounded_range,3,4
8,4yo,499,2015-01-01,2026-05-22,exact,4,4
9,4-5yo,419,2015-02-22,2026-05-26,bounded_range,4,5



Races by structural age-condition type:


,age_condition_type,races
2,minimum_age,82022
1,exact,24693
0,bounded_range,4919



Unresolved age-band forms: 0


### Finding — age restriction is a prospective race condition and is cleanly represented in Source Version 1

The BHA structured race record exposes an `ageLimit` describing the ages of
horses eligible for the programmed race.

Examples include:

- `2YO` — exactly two-year-olds;
- `3YO` — exactly three-year-olds;
- `3-5YO` — horses aged three through five;
- `3YO+` — horses aged three and older;
- `4YO+` — horses aged four and older;
- `5YO+` — horses aged five and older.

This condition belongs to the prospective race specification. It is not
derived from the ages of the horses that ultimately participate.

A 34-race comparison between BHA structured data and Database v4 found
semantic agreement in all 34 races. The only difference was presentational
capitalisation, for example:

- BHA `4YO+`
- Source Version 1 `4yo+`

The complete Database v4 GB population contains 24 distinct source
age-band forms covering all 111,634 races.

Every form can be interpreted using one of three simple structures:

- **exact age** — e.g. `2yo`, `3yo`, `4yo`;
- **bounded range** — e.g. `3-5yo`, `4-6yo`, `5-8yo`;
- **minimum age** — e.g. `3yo+`, `4yo+`, `10yo+`.

Population totals are:

- minimum-age conditions: 82,022 races;
- exact-age conditions: 24,693 races;
- bounded age ranges: 4,919 races.

No age-band values remained structurally unresolved.

Therefore:

> **The age condition is a prospective eligibility property of the race,
> specifying an exact age, bounded age range, or minimum permitted age.**

Within the evidence tested here, Source Version 1's `age_band_raw` provides a
clean representation of the same race-level concept as BHA `ageLimit`.

### Evidence note — age eligibility operates at more than one level

The BHA `ageLimit` field represents an important prospective race condition,
but it does not by itself describe every age-related rule that may determine
whether a horse can compete.

There are at least three relevant layers.

#### 1. Official racing age

A racehorse's age for racing purposes advances on **1 January**, rather than on
the horse's individual birthday.

Therefore a horse foaled during 2023 is treated as a three-year-old throughout
the 2026 racing year, including before the anniversary of its actual foaling
date.

This official racing age is the age used when applying race conditions such as
`2YO`, `3YO`, `4YO+`, or `3-5YO`.

#### 2. General age qualifications imposed by the Rules or race type

Before considering the age condition of a particular programmed race, broader
rules can already restrict whether a horse of a given age may participate in a
particular form of racing.

For example:

- Flat horses can begin racing during the year in which they are officially
  two years old;
- Jump racing has its own minimum-age qualification;
- particular race types can impose additional age requirements.

This means that age eligibility is not necessarily created entirely by the
individual race's `ageLimit`.

#### 3. The individual race's prospective age condition

The BHA structured race record exposes this race-specific condition through
`ageLimit`.

Observed examples include:

- `2YO` — exactly two-year-olds;
- `3YO` — exactly three-year-olds;
- `4YO` — exactly four-year-olds;
- `3-5YO` — horses aged three, four or five;
- `4-6YO` — horses aged four, five or six;
- `3YO+` — horses aged three or older;
- `4YO+` — horses aged four or older;
- `10YO+` — horses aged ten or older.

This is a genuine prospective eligibility restriction. It specifies which
official racing ages are permitted in that particular programmed race.

The 34-race BHA-v4 comparison showed semantic agreement between BHA
`ageLimit` and Source Version 1 `age_band_raw` in all 34 races, differing only
in letter case.

Across all 111,634 GB races in Database v4, every observed `age_band_raw`
value falls into one of three simple structural forms:

- exact age;
- bounded age range;
- minimum age with no stated upper limit.

No source age-band form remained structurally unresolved.

#### Age can also affect competitive terms

Age is not only an eligibility question.

Where horses of different ages compete together, age can also affect the
weights under which they race. The BHA weight-for-age system can provide
younger horses with allowances intended to account for differences in
maturity.

Therefore age can affect both:

1. **whether a horse is eligible to compete**, and
2. **the competitive terms under which it competes**.

This gives the following working model:

> **official racing age**
> → **general code/race-type age qualification**
> → **individual race age condition (`ageLimit`)**
> → **where applicable, age-dependent competitive terms such as weight-for-age**

Consequently:

> **`ageLimit` is a prospective race-level eligibility condition, but it should
> not yet be assumed to be a complete description of every age-related rule
> governing participation in the race.**

The next question is therefore whether all operative age restrictions of an
individual British race are explicitly represented by `ageLimit`, or whether
some remain implicit in the governing race type or wider Rules of Racing.

In [27]:
# Examine the age conditions attached specifically to National Hunt Flat races.
#
# Analytical question:
#
#   Does the explicit race-level age condition completely describe the ages
#   permitted in a National Hunt Flat race, or do wider BHA rules impose
#   additional age restrictions?
#
# BHA states that:
#
#   - only horses aged seven or younger may race in British NH Flat races;
#   - from the 2023/24 season, six-year-olds cease to be eligible for NH Flat
#     races after the end of that Jump season.
#
# We therefore inspect the age-band forms actually attached to every NH Flat
# race in the GB population.
#
# This is still observation before interpretation: first establish what the
# programmed race records themselves say.

with connect_read_only(DATABASE) as connection:
    nh_flat_age_conditions = pd.read_sql_query(
        """
        SELECT
            age_band_raw,
            COUNT(*) AS races,
            MIN(raw_date) AS earliest_date,
            MAX(raw_date) AS latest_date
        FROM view_gb_reconciled_race_occurrences_with_racecourse
        WHERE lower(trim(race_type_raw)) IN (
            'nh flat',
            'national hunt flat'
        )
        GROUP BY age_band_raw
        ORDER BY races DESC, age_band_raw
        """,
        connection,
    )


print("NH Flat races represented:", nh_flat_age_conditions["races"].sum())
print("Distinct NH Flat age conditions:", len(nh_flat_age_conditions))

display(nh_flat_age_conditions)

NH Flat races represented: 3100
Distinct NH Flat age conditions: 11


,age_band_raw,races,earliest_date,latest_date
0,4-6yo,2351,2015-01-01,2026-04-24
1,4-5yo,412,2015-02-22,2026-05-26
2,3yo,138,2015-10-04,2025-12-20
3,4yo+,78,2018-03-02,2023-03-05
4,4yo,60,2015-01-01,2026-05-22
5,3-5yo,29,2015-10-21,2025-12-31
6,5yo+,13,2018-03-02,2021-02-09
7,4-7yo,11,2015-01-03,2023-03-25
8,3-6yo,3,2016-12-19,2020-11-02
9,5-6yo,3,2015-01-09,2016-01-28


In [28]:
# Find the latest National Hunt Flat races whose explicit source age condition
# is open-ended (`4yo+` or `5yo+`).
#
# Analytical question:
#
#   Can we identify a concrete historical race where the race-level age
#   condition itself has no upper boundary?
#
# We will then retrieve that exact race from the BHA API and test whether BHA's
# own structured `ageLimit` is also open-ended.
#
# This query uses governed racecourse identity and advertised course-local time.
# No source off-time field is used.

with connect_read_only(DATABASE) as connection:
    open_ended_nh_flat_age_cases = pd.read_sql_query(
        """
        SELECT
            source_race_occurrence_code,
            raw_date AS race_date,
            governed_racecourse_name,
            substr(advertised_start_course_local, 12, 8) AS advertised_time,
            race_type_raw,
            age_band_raw,
            race_name_raw
        FROM view_gb_reconciled_race_occurrences_with_racecourse
        WHERE lower(trim(race_type_raw)) IN (
            'nh flat',
            'national hunt flat'
        )
          AND lower(trim(age_band_raw)) IN ('4yo+', '5yo+')
        ORDER BY raw_date DESC,
                 governed_racecourse_name,
                 advertised_start_course_local
        """,
        connection,
    )


print(
    "Open-ended NH Flat age-condition races:",
    len(open_ended_nh_flat_age_cases),
)

pd.set_option("display.max_colwidth", None)

display(
    open_ended_nh_flat_age_cases.head(20)
)

Open-ended NH Flat age-condition races: 91


,source_race_occurrence_code,race_date,governed_racecourse_name,advertised_time,race_type_raw,age_band_raw,race_name_raw
0,race:77b5dbbbfdee69d4d92a5826:000133771,2023-03-05,Ffos Las,17:40:00,NH Flat,4yo+,DragonBet Proud To Be Welsh Open Maiden National Hunt Flat Race (Category 3 Elimination) (GBB Race)
1,race:77b5dbbbfdee69d4d92a5826:000119895,2022-05-03,Sedgefield,NaN,NH Flat,4yo+,Watch Chester On Sky Sports Racing Mares Open National Hunt Flat Race (Cat 1) (GBB Race)
2,race:77b5dbbbfdee69d4d92a5826:000098253,2021-02-15,Lingfield Park,14:10:00,NH Flat,4yo+,Sky Sports Racing Sky 415 Jumpers Bumper National Hunt Flat Race (Div I)
3,race:77b5dbbbfdee69d4d92a5826:000098237,2021-02-15,Lingfield Park,14:40:00,NH Flat,4yo+,Sky Sports Racing Sky 415 Jumpers Bumper National Hunt Flat Race (Div II)
4,race:77b5dbbbfdee69d4d92a5826:000098246,2021-02-15,Lingfield Park,15:15:00,NH Flat,4yo+,Free Tips Daily On attheraces.com Jumpers Bumper National Hunt Flat Race
5,race:77b5dbbbfdee69d4d92a5826:000098254,2021-02-15,Lingfield Park,15:45:00,NH Flat,4yo+,Follow At The Races On Twitter Jumpers Bumper National Hunt Flat Race
6,race:77b5dbbbfdee69d4d92a5826:000098255,2021-02-15,Lingfield Park,16:20:00,NH Flat,4yo+,Download The At The Races App Mares Jumpers Bumper National Hunt Flat Race
7,race:77b5dbbbfdee69d4d92a5826:000098238,2021-02-15,Lingfield Park,16:50:00,NH Flat,4yo+,Sky Sports Racing HD Virgin 535 Jumpers Bumper National Hunt Flat Race
8,race:77b5dbbbfdee69d4d92a5826:000098097,2021-02-11,Kempton Park,14:05:00,NH Flat,4yo+,vbet.co.uk Jumpers Bumper National Hunt Flat Race (Div I) (AW)
9,race:77b5dbbbfdee69d4d92a5826:000098104,2021-02-11,Kempton Park,14:40:00,NH Flat,4yo+,vbet.co.uk Jumpers Bumper National Hunt Flat Race (Div II) (AW)


In [29]:
# Check the authoritative BHA structured age condition for the latest
# open-ended NH Flat example found in Database v4.
#
# Analytical question:
#
#   Did BHA itself describe this race as `4YO+`, or is the open-ended age
#   condition peculiar to Source Version 1?
#
# Target:
#
#   Ffos Las
#   5 March 2023
#   advertised 17:40
#   DragonBet Proud To Be Welsh Open Maiden National Hunt Flat Race
#
# Race matching uses BHA raceTime and the governed advertised course-local
# time established in Database v4. No source off-time field is used.

from urllib.parse import urlencode
from urllib.request import Request, urlopen
import json
import pandas as pd


target_date = "2023-03-05"
target_course = "Ffos Las"
target_time = "17:40:00"


# ---------------------------------------------------------------------------
# Find the BHA fixture at Ffos Las on the target date.
# ---------------------------------------------------------------------------

fixture_url = (
    "https://api09.horseracing.software/bha/v1/fixtures/?"
    + urlencode(
        {
            "fromdate": target_date,
            "todate": target_date,
            "resultsAvailable": 1,
            "order": "asc",
            "page": 1,
            "per_page": 100,
        }
    )
)

request_headers = {
    "Authorization": authorization_value,
    "Accept": "application/json",
    "Origin": "https://www.britishhorseracing.com",
    "Referer": "https://www.britishhorseracing.com/",
    "User-Agent": USER_AGENT,
}

with urlopen(
    Request(fixture_url, headers=request_headers),
    timeout=30,
) as response:
    fixture_payload = json.loads(response.read().decode("utf-8"))


ffos_fixtures = [
    fixture
    for fixture in fixture_payload["data"]
    if fixture["courseName"] == target_course
]

print("Ffos Las fixtures found:", len(ffos_fixtures))

assert len(ffos_fixtures) == 1

fixture = ffos_fixtures[0]


# ---------------------------------------------------------------------------
# Retrieve all races in that fixture and inspect BHA's structured ageLimit.
# ---------------------------------------------------------------------------

races_url = (
    "https://api09.horseracing.software/bha/v1/fixtures/"
    f"{fixture['fixtureYear']}/{fixture['fixtureId']}/races"
)

with urlopen(
    Request(races_url, headers=request_headers),
    timeout=30,
) as response:
    races_payload = json.loads(response.read().decode("utf-8"))


bha_ffos_races = pd.DataFrame(
    [
        {
            "advertised_time": race["raceTime"],
            "bha_race_name": race["raceName"],
            "bha_race_type": race["raceCriteriaRaceType"],
            "bha_age_limit": race["ageLimit"],
            "bha_rating_band": race["ratingBand"],
            "bha_race_class": race["raceClass"],
        }
        for race in races_payload["data"]
    ]
)


pd.set_option("display.max_colwidth", None)

display(
    bha_ffos_races[
        bha_ffos_races["advertised_time"] == target_time
    ]
)

Ffos Las fixtures found: 1


,advertised_time,bha_race_name,bha_race_type,bha_age_limit,bha_rating_band,bha_race_class
6,17:40:00,THE DRAGONBET PROUD TO BE WELSH OPEN MAIDEN NATIONAL HUNT FLAT RACE (CLASS 5) (Category 3 Elimination) (GBB RACE),JUMP,4YO+,,5


### Interim finding — BHA `ageLimit` can be formally open-ended

The Ffos Las National Hunt Flat race at 17:40 on 5 March 2023 provides a
direct historical example of an open-ended BHA age condition.

Database v4 records:

- race type: `NH Flat`;
- age condition: `4yo+`.

The authoritative BHA structured historical record independently reports:

- race type: `JUMP`;
- `ageLimit`: `4YO+`.

Therefore the open-ended form is not a Source Version 1 simplification or
publication convention. It is the race-level age condition recorded by the
BHA itself.

This establishes that BHA `ageLimit` can formally state a minimum age without
an explicit maximum age.

However, this does not yet prove that the horse was genuinely eligible at any
older age.

National Hunt Flat racing is governed by additional race-type rules, and the
historical rules applying on the race date must be established before deciding
whether an implicit upper age restriction operated alongside the explicit
`4YO+` condition.

In particular, the later rule preventing six-year-olds from running in
National Hunt Flat races after the end of the Jump season did not yet apply
to this race on 5 March 2023.

The next question is therefore:

> **What wider National Hunt Flat age qualification applied under the BHA Rules
> on 5 March 2023, and did it impose an upper age boundary not expressed by
> `ageLimit = 4YO+`?**

In [30]:
# Examine the realised runner ages in all 91 NH Flat races whose explicit
# race-level age condition is open-ended (`4yo+` or `5yo+`).
#
# Analytical questions:
#
#   1. What ages actually participated in these races?
#   2. Did any horse older than seven run?
#   3. If so, were those cases concentrated in exceptional "Jumpers' Bumper"
#      races rather than ordinary National Hunt Flat races?
#
# This does NOT use realised runner ages to redefine the prospective age
# condition. The purpose is to test whether wider race-type rules appear to
# constrain an explicitly open-ended BHA ageLimit in practice.
#
# Race identity is carried by the durable project-owned
# `source_race_occurrence_code`.

open_ended_race_codes = (
    open_ended_nh_flat_age_cases["source_race_occurrence_code"]
    .drop_duplicates()
    .tolist()
)

placeholders = ",".join("?" for _ in open_ended_race_codes)


with connect_read_only(DATABASE) as connection:
    open_ended_nh_flat_runners = pd.read_sql_query(
        f"""
        SELECT
            source_race_occurrence_code,
            raw_horse AS horse,
            age_recorded,
            age_interpretation_status
        FROM view_reconciled_source_runner_participations
        WHERE source_race_occurrence_code IN ({placeholders})
        """,
        connection,
        params=open_ended_race_codes,
    )


# Add the prospective race information from the 91-race table already built.
open_ended_runner_age_check = (
    open_ended_nh_flat_runners
    .merge(
        open_ended_nh_flat_age_cases[
            [
                "source_race_occurrence_code",
                "race_date",
                "governed_racecourse_name",
                "advertised_time",
                "age_band_raw",
                "race_name_raw",
            ]
        ],
        on="source_race_occurrence_code",
        how="left",
        validate="many_to_one",
    )
)


# Summarise the realised age range in each race.
race_age_summary = (
    open_ended_runner_age_check
    .groupby(
        [
            "source_race_occurrence_code",
            "race_date",
            "governed_racecourse_name",
            "advertised_time",
            "age_band_raw",
            "race_name_raw",
        ],
        dropna=False,
    )
    .agg(
        runners=("horse", "size"),
        runners_with_age=("age_recorded", "count"),
        minimum_runner_age=("age_recorded", "min"),
        maximum_runner_age=("age_recorded", "max"),
    )
    .reset_index()
    .sort_values(
        ["maximum_runner_age", "race_date"],
        ascending=[False, False],
    )
)


print("Open-ended NH Flat races:", len(race_age_summary))
print(
    "Highest realised runner age:",
    race_age_summary["maximum_runner_age"].max(),
)

print()
print("Races containing a runner aged over seven:")

display(
    race_age_summary[
        race_age_summary["maximum_runner_age"] > 7
    ]
)


print()
print("Complete race-age summary:")

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", None)

display(race_age_summary)

Open-ended NH Flat races: 91
Highest realised runner age: 13

Races containing a runner aged over seven:


,source_race_occurrence_code,race_date,governed_racecourse_name,advertised_time,age_band_raw,race_name_raw,runners,runners_with_age,minimum_runner_age,maximum_runner_age
65,race:77b5dbbbfdee69d4d92a5826:000098043,2021-02-08,Newcastle,13:30:00,4yo+,quinnbet.com Jumpers Bumper National Hunt Flat Race (Div II) (AW),9,9,5,13
23,race:77b5dbbbfdee69d4d92a5826:000086380,2020-02-21,Newcastle,15:43:00,4yo+,Allingtons Kia Jumpers Bumper National Hunt Flat Race (All-Weather),12,12,4,13
82,race:77b5dbbbfdee69d4d92a5826:000098105,2021-02-11,Kempton Park,16:15:00,4yo+,Vbet Jumpers Bumper National Hunt Flat Race (Div I) (AW),14,14,6,12
69,race:77b5dbbbfdee69d4d92a5826:000098047,2021-02-08,Newcastle,15:05:00,5yo+,Quinnbet Quarterback Jumpers Bumper National Hunt Flat Race (AW),8,8,7,12
55,race:77b5dbbbfdee69d4d92a5826:000097804,2021-02-02,Newcastle,14:55:00,4yo+,Quinncasino Jumpers Bumper National Hunt Flat Race (Div II) (AW),12,12,5,12
61,race:77b5dbbbfdee69d4d92a5826:000097816,2021-02-02,Newcastle,16:25:00,4yo+,Quinnbet Jumpers Bumper National Hunt Flat Race (AW),11,11,7,12
35,race:77b5dbbbfdee69d4d92a5826:000097310,2021-01-18,Lingfield Park,14:50:00,4yo+,Sky Sports Jumpers Bumper National Hunt Flat Race,13,13,5,12
40,race:77b5dbbbfdee69d4d92a5826:000097317,2021-01-18,Lingfield Park,13:15:00,4yo+,Watch Sky Sports Racing In HD Jumpers Bumper National Hunt Flat Race,8,8,6,12
25,race:77b5dbbbfdee69d4d92a5826:000096926,2021-01-08,Lingfield Park,15:20:00,4yo+,Sky Sports Racing On Virgin 535 Jumpers Bumper National Hunt Flat Race,9,9,6,12
32,race:77b5dbbbfdee69d4d92a5826:000096933,2021-01-08,Newcastle,14:10:00,5yo+,Download The Quinnbet App Jumpers Bumper National Hunt Flat Race (AW),8,8,5,12



Complete race-age summary:


,source_race_occurrence_code,race_date,governed_racecourse_name,advertised_time,age_band_raw,race_name_raw,runners,runners_with_age,minimum_runner_age,maximum_runner_age
65,race:77b5dbbbfdee69d4d92a5826:000098043,2021-02-08,Newcastle,13:30:00,4yo+,quinnbet.com Jumpers Bumper National Hunt Flat Race (Div II) (AW),9,9,5,13
23,race:77b5dbbbfdee69d4d92a5826:000086380,2020-02-21,Newcastle,15:43:00,4yo+,Allingtons Kia Jumpers Bumper National Hunt Flat Race (All-Weather),12,12,4,13
82,race:77b5dbbbfdee69d4d92a5826:000098105,2021-02-11,Kempton Park,16:15:00,4yo+,Vbet Jumpers Bumper National Hunt Flat Race (Div I) (AW),14,14,6,12
69,race:77b5dbbbfdee69d4d92a5826:000098047,2021-02-08,Newcastle,15:05:00,5yo+,Quinnbet Quarterback Jumpers Bumper National Hunt Flat Race (AW),8,8,7,12
55,race:77b5dbbbfdee69d4d92a5826:000097804,2021-02-02,Newcastle,14:55:00,4yo+,Quinncasino Jumpers Bumper National Hunt Flat Race (Div II) (AW),12,12,5,12
61,race:77b5dbbbfdee69d4d92a5826:000097816,2021-02-02,Newcastle,16:25:00,4yo+,Quinnbet Jumpers Bumper National Hunt Flat Race (AW),11,11,7,12
35,race:77b5dbbbfdee69d4d92a5826:000097310,2021-01-18,Lingfield Park,14:50:00,4yo+,Sky Sports Jumpers Bumper National Hunt Flat Race,13,13,5,12
40,race:77b5dbbbfdee69d4d92a5826:000097317,2021-01-18,Lingfield Park,13:15:00,4yo+,Watch Sky Sports Racing In HD Jumpers Bumper National Hunt Flat Race,8,8,6,12
25,race:77b5dbbbfdee69d4d92a5826:000096926,2021-01-08,Lingfield Park,15:20:00,4yo+,Sky Sports Racing On Virgin 535 Jumpers Bumper National Hunt Flat Race,9,9,6,12
32,race:77b5dbbbfdee69d4d92a5826:000096933,2021-01-08,Newcastle,14:10:00,5yo+,Download The Quinnbet App Jumpers Bumper National Hunt Flat Race (AW),8,8,5,12


In [31]:
# Inspect the full BHA race-detail record for the Ffos Las 17:40 NH Flat race.
#
# Analytical question:
#
#   BHA's fixture race list records ageLimit = `4YO+`.
#
#   Does the more detailed BHA race record contain additional structured
#   eligibility information that further constrains age, or is the upper-age
#   qualification supplied by wider race-type rules outside `ageLimit`?
#
# We identify the race using BHA's official raceTime and then retrieve the
# corresponding race-detail endpoint.
#
# No source off-time field is used.

target_race = next(
    race
    for race in races_payload["data"]
    if race["raceTime"] == "17:40:00"
)

print("Race-list identity:")
print("raceId:", target_race["raceId"])
print("divisionSequence:", target_race.get("divisionSequence"))
print("ageLimit:", target_race["ageLimit"])
print("raceName:", target_race["raceName"])
print()


# ---------------------------------------------------------------------------
# Retrieve BHA's detailed structured record for this race occurrence.
# ---------------------------------------------------------------------------

race_detail_url = (
    "https://api09.horseracing.software/bha/v1/races/"
    f"{target_race['yearOfRace']}/"
    f"{target_race['raceId']}/"
    f"{target_race['divisionSequence']}"
)

with urlopen(
    Request(race_detail_url, headers=request_headers),
    timeout=30,
) as response:
    race_detail_payload = json.loads(response.read().decode("utf-8"))


# ---------------------------------------------------------------------------
# Inspect the API response shape before interpreting it.
#
# This endpoint returns `data` as a list, even when it contains information
# for a single requested race.
# ---------------------------------------------------------------------------

race_detail_data = race_detail_payload["data"]

print("Type of BHA detail data:", type(race_detail_data).__name__)
print("Records returned:", len(race_detail_data))
print()


# ---------------------------------------------------------------------------
# Print every field from every returned record.
#
# We are deliberately not selecting likely age-related fields yet. The purpose
# is to discover what BHA actually exposes at race-detail level before deciding
# whether age qualification exists anywhere beyond `ageLimit`.
# ---------------------------------------------------------------------------

for record_number, record in enumerate(race_detail_data, start=1):

    print("=" * 100)
    print(f"RECORD {record_number}")
    print("=" * 100)

    if isinstance(record, dict):
        for key, value in record.items():
            print(f"{key}: {value}")
    else:
        print(record)

    print()

Race-list identity:
raceId: 61997
divisionSequence: 0
ageLimit: 4YO+
raceName: THE DRAGONBET PROUD TO BE WELSH OPEN MAIDEN NATIONAL HUNT FLAT RACE (CLASS 5) (Category 3 Elimination) (GBB RACE)

Type of BHA detail data: list
Records returned: 1

RECORD 1
raceId: 61997
yearOfRace: 2023
divisionSequence: 0
raceNumber: 21327
fixtureId: 1443
raceDate: 2023-03-05
raceTime: 17:40:00
rawDistanceText: 1m 7f 182y
distanceUnits: None
distanceValue: 3482
distanceText: 1m 7f 182y
distanceFullText: about TWO MILES (1m 7f 182yds)
raceName: THE DRAGONBET PROUD TO BE WELSH OPEN MAIDEN NATIONAL HUNT FLAT RACE (CLASS 5) (Category 3 Elimination) (GBB RACE)
ageLimit: 4YO+
sexLimit: ALL
currentStageCode: 99
transparentWindowStatus: DECLARATIONS_OPEN
goingText: Good, Good to Soft in places
riderType: None
animalType: MAIDEN
ratingBand: -1--1
prizeAmount: 4800
prizeCurrency: None
tvCoverage: NOT_COVERED
racingUK: 0
atTheRaces: 1
offTime: None
winTime: 3m 42.75s
runners: 7
resultsAvailable: 1
racecardAvailable

In [34]:
# Test the complete GB runner population against the explicit prospective
# age condition without loading the whole runner population into pandas.
#
# Analytical question:
#
#   Do any realised runners fall outside the stated race-level age condition?
#
# SQLite performs the population-wide comparison and returns:
#
#   1. a small population summary;
#   2. only the exceptional runner records requiring investigation.
#
# This avoids retaining ~985,000 runner rows in notebook memory.
#
# Race identity uses `source_race_occurrence_code`.
# No source off-time field is used.


# ---------------------------------------------------------------------------
# Population-level compliance summary.
# ---------------------------------------------------------------------------

with connect_read_only(DATABASE) as connection:
    age_compliance_summary = pd.read_sql_query(
        """
        SELECT
            CASE
                WHEN runner.age_recorded IS NULL
                    THEN 'runner_age_unavailable'

                WHEN race.stated_minimum_age IS NOT NULL
                 AND runner.age_recorded < race.stated_minimum_age
                    THEN 'below_stated_minimum'

                WHEN race.stated_maximum_age IS NOT NULL
                 AND runner.age_recorded > race.stated_maximum_age
                    THEN 'above_stated_maximum'

                ELSE 'within_stated_age_condition'
            END AS age_compliance_status,

            COUNT(*) AS runner_records

        FROM view_gb_reconciled_race_occurrences_with_racecourse AS race

        JOIN view_reconciled_source_runner_participations AS runner
          ON runner.source_race_occurrence_code =
             race.source_race_occurrence_code

        GROUP BY age_compliance_status

        ORDER BY runner_records DESC
        """,
        connection,
    )


print(
    "Runner records tested:",
    int(age_compliance_summary["runner_records"].sum()),
)

print()
print("Age-condition compliance:")

display(age_compliance_summary)


# ---------------------------------------------------------------------------
# Retrieve ONLY the runners outside the literal stated age condition.
#
# This small dataframe is the input for the following investigation into the
# apparent exceptions.
# ---------------------------------------------------------------------------

with connect_read_only(DATABASE) as connection:
    age_band_violations = pd.read_sql_query(
        """
        SELECT
            race.source_race_occurrence_code,
            race.raw_date AS race_date,
            race.governed_racecourse_name,
            race.advertised_start_course_local,
            race.race_type_raw,
            race.race_name_raw,
            race.age_band_raw,
            race.stated_minimum_age,
            race.stated_maximum_age,

            runner.raw_horse AS horse,
            runner.age_recorded AS runner_age,
            runner.age_interpretation_status,

            CASE
                WHEN race.stated_minimum_age IS NOT NULL
                 AND runner.age_recorded < race.stated_minimum_age
                    THEN 'below_stated_minimum'

                WHEN race.stated_maximum_age IS NOT NULL
                 AND runner.age_recorded > race.stated_maximum_age
                    THEN 'above_stated_maximum'
            END AS age_compliance_status

        FROM view_gb_reconciled_race_occurrences_with_racecourse AS race

        JOIN view_reconciled_source_runner_participations AS runner
          ON runner.source_race_occurrence_code =
             race.source_race_occurrence_code

        WHERE runner.age_recorded IS NOT NULL

          AND (
                (
                    race.stated_minimum_age IS NOT NULL
                    AND runner.age_recorded < race.stated_minimum_age
                )
                OR
                (
                    race.stated_maximum_age IS NOT NULL
                    AND runner.age_recorded > race.stated_maximum_age
                )
          )

        ORDER BY
            race.raw_date,
            race.governed_racecourse_name,
            race.advertised_start_course_local,
            runner.raw_horse
        """,
        connection,
    )


print()
print(
    "Runner records outside stated age condition:",
    len(age_band_violations),
)

print(
    "Affected races:",
    age_band_violations["source_race_occurrence_code"].nunique(),
)


pd.set_option("display.max_colwidth", None)

display(
    age_band_violations[
        [
            "race_date",
            "governed_racecourse_name",
            "advertised_start_course_local",
            "race_type_raw",
            "age_band_raw",
            "stated_minimum_age",
            "stated_maximum_age",
            "horse",
            "runner_age",
            "age_compliance_status",
            "race_name_raw",
        ]
    ]
)

Runner records tested: 984757

Age-condition compliance:


,age_compliance_status,runner_records
0,within_stated_age_condition,984730
1,below_stated_minimum,26
2,above_stated_maximum,1



Runner records outside stated age condition: 27
Affected races: 25


,race_date,governed_racecourse_name,advertised_start_course_local,race_type_raw,age_band_raw,stated_minimum_age,stated_maximum_age,horse,runner_age,age_compliance_status,race_name_raw
0,2015-06-20,Ascot,2015-06-20T16:20:00+01:00,Flat,4yo+,4,NaN,Brazen Beau (AUS),3,below_stated_minimum,Diamond Jubilee Stakes (British Champions Series & Global Sprint Challenge)
1,2015-06-20,Ascot,2015-06-20T16:20:00+01:00,Flat,4yo+,4,NaN,Wandjina (AUS),3,below_stated_minimum,Diamond Jubilee Stakes (British Champions Series & Global Sprint Challenge)
2,2016-06-18,Ascot,2016-06-18T16:20:00+01:00,Flat,4yo+,4,NaN,Holler (AUS),3,below_stated_minimum,Diamond Jubilee Stakes (British Champions Series & Global Sprint Challenge)
3,2017-06-21,Ascot,2017-06-21T15:40:00+01:00,Flat,4yo+,4,NaN,Greta G (ARG),3,below_stated_minimum,Duke Of Cambridge Stakes (Fillies & Mares)
4,2017-07-27,Great Yarmouth,2017-07-27T13:40:00+01:00,Flat,2yo,2,2.0,Millies Kiss (GB),3,above_stated_maximum,Read Silvestre De Sousa At 188Bet Novice Auction Stakes (Plus 10 Race)
5,2018-04-30,Windsor,2018-04-30T17:40:00+01:00,Flat,2yo,2,2.0,Anthem Of Peace (AUS),1,below_stated_minimum,British EBF Novice Stakes (Plus 10 Race)
6,2018-05-10,Chester,2018-05-10T16:05:00+01:00,Flat,2yo,2,2.0,Anthem Of Peace (AUS),1,below_stated_minimum,British Stallion Studs EBF Maiden Stakes (Plus 10 Race)
7,2018-06-01,Bath,2018-06-01T18:20:00+01:00,Flat,2yo,2,2.0,Anthem Of Peace (AUS),1,below_stated_minimum,EBF Novice Stakes
8,2018-06-10,Goodwood,2018-06-10T15:10:00+01:00,Flat,2yo,2,2.0,Anthem Of Peace (AUS),1,below_stated_minimum,Sutton Winson Backing The NSPCC Selling Stakes
9,2018-06-18,Windsor,2018-06-18T19:00:00+01:00,Flat,2yo,2,2.0,Anthem Of Peace (AUS),1,below_stated_minimum,Sky Bet Best Odds Guaranteed Selling Stakes


In [35]:
# Classify the 27 apparent age-band exceptions by breeding jurisdiction.
#
# This dataframe contains only the exceptional runners identified by SQLite,
# so the analysis remains deliberately small in memory.

import re


SOUTHERN_HEMISPHERE_SUFFIXES = {
    "AUS",
    "NZ",
    "SAF",
    "ARG",
    "BRZ",
    "CHI",
    "URU",
}


# Extract the final breeding-jurisdiction suffix from labels such as:
#
#   Brazen Beau (AUS)
#   Givinitsum (SAF)
#   Millies Kiss (GB)

age_violation_classification = age_band_violations.copy()

age_violation_classification["breeding_suffix"] = (
    age_violation_classification["horse"]
    .str.extract(r"\(([A-Z]{2,3})\)\s*$", expand=False)
)

age_violation_classification["southern_hemisphere_bred"] = (
    age_violation_classification["breeding_suffix"]
    .isin(SOUTHERN_HEMISPHERE_SUFFIXES)
)


# Calculate distance from the relevant stated age boundary without row-wise
# Python apply().
#
# Below minimum:
#   runner age - minimum age  -> negative
#
# Above maximum:
#   runner age - maximum age  -> positive

age_violation_classification["age_boundary_difference"] = 0.0

below_mask = (
    age_violation_classification["age_compliance_status"]
    == "below_stated_minimum"
)

above_mask = (
    age_violation_classification["age_compliance_status"]
    == "above_stated_maximum"
)

age_violation_classification.loc[
    below_mask,
    "age_boundary_difference",
] = (
    age_violation_classification.loc[below_mask, "runner_age"]
    - age_violation_classification.loc[below_mask, "stated_minimum_age"]
)

age_violation_classification.loc[
    above_mask,
    "age_boundary_difference",
] = (
    age_violation_classification.loc[above_mask, "runner_age"]
    - age_violation_classification.loc[above_mask, "stated_maximum_age"]
)


print(
    "Apparent age-band breaches:",
    len(age_violation_classification),
)

print()
print("By breeding jurisdiction / hemisphere:")

display(
    age_violation_classification
    .groupby(
        [
            "breeding_suffix",
            "southern_hemisphere_bred",
            "age_boundary_difference",
        ],
        dropna=False,
    )
    .size()
    .reset_index(name="runner_records")
    .sort_values("runner_records", ascending=False)
)


print()
print("Individual apparent breaches:")

display(
    age_violation_classification[
        [
            "race_date",
            "governed_racecourse_name",
            "race_type_raw",
            "age_band_raw",
            "horse",
            "breeding_suffix",
            "runner_age",
            "age_compliance_status",
            "age_boundary_difference",
            "southern_hemisphere_bred",
            "race_name_raw",
        ]
    ]
)

Apparent age-band breaches: 27

By breeding jurisdiction / hemisphere:


,breeding_suffix,southern_hemisphere_bred,age_boundary_difference,runner_records
1,AUS,True,-1.0,22
3,SAF,True,-1.0,3
0,ARG,True,-1.0,1
2,GB,False,1.0,1



Individual apparent breaches:


,race_date,governed_racecourse_name,race_type_raw,age_band_raw,horse,breeding_suffix,runner_age,age_compliance_status,age_boundary_difference,southern_hemisphere_bred,race_name_raw
0,2015-06-20,Ascot,Flat,4yo+,Brazen Beau (AUS),AUS,3,below_stated_minimum,-1.0,True,Diamond Jubilee Stakes (British Champions Series & Global Sprint Challenge)
1,2015-06-20,Ascot,Flat,4yo+,Wandjina (AUS),AUS,3,below_stated_minimum,-1.0,True,Diamond Jubilee Stakes (British Champions Series & Global Sprint Challenge)
2,2016-06-18,Ascot,Flat,4yo+,Holler (AUS),AUS,3,below_stated_minimum,-1.0,True,Diamond Jubilee Stakes (British Champions Series & Global Sprint Challenge)
3,2017-06-21,Ascot,Flat,4yo+,Greta G (ARG),ARG,3,below_stated_minimum,-1.0,True,Duke Of Cambridge Stakes (Fillies & Mares)
4,2017-07-27,Great Yarmouth,Flat,2yo,Millies Kiss (GB),GB,3,above_stated_maximum,1.0,False,Read Silvestre De Sousa At 188Bet Novice Auction Stakes (Plus 10 Race)
5,2018-04-30,Windsor,Flat,2yo,Anthem Of Peace (AUS),AUS,1,below_stated_minimum,-1.0,True,British EBF Novice Stakes (Plus 10 Race)
6,2018-05-10,Chester,Flat,2yo,Anthem Of Peace (AUS),AUS,1,below_stated_minimum,-1.0,True,British Stallion Studs EBF Maiden Stakes (Plus 10 Race)
7,2018-06-01,Bath,Flat,2yo,Anthem Of Peace (AUS),AUS,1,below_stated_minimum,-1.0,True,EBF Novice Stakes
8,2018-06-10,Goodwood,Flat,2yo,Anthem Of Peace (AUS),AUS,1,below_stated_minimum,-1.0,True,Sutton Winson Backing The NSPCC Selling Stakes
9,2018-06-18,Windsor,Flat,2yo,Anthem Of Peace (AUS),AUS,1,below_stated_minimum,-1.0,True,Sky Bet Best Odds Guaranteed Selling Stakes


In [36]:
# Classify every apparent age-band breach by the horse's breeding jurisdiction.
#
# Analytical question:
#
#   Are the 26 runners below the stated minimum age a coherent Southern
#   Hemisphere pattern rather than ordinary breaches of the race condition?
#
# The source horse label includes a breeding-country suffix such as:
#
#   (AUS)
#   (SAF)
#   (ARG)
#   (GB)
#
# Australia, South Africa and Argentina are Southern Hemisphere breeding
# jurisdictions. BHA uses separate Southern Hemisphere weight-for-age scales
# because their breeding season is approximately six months offset from the
# Northern Hemisphere.
#
# We are NOT yet asserting that hemisphere automatically explains eligibility.
# This cell establishes the population pattern first.

import re


SOUTHERN_HEMISPHERE_SUFFIXES = {
    "AUS",
    "NZ",
    "SAF",
    "ARG",
    "BRZ",
    "CHI",
    "URU",
}


def extract_breeding_suffix(horse_label):
    if not isinstance(horse_label, str):
        return None

    match = re.search(r"\(([A-Z]{2,3})\)\s*$", horse_label.strip())

    if match:
        return match.group(1)

    return None


age_violation_classification = age_band_violations.copy()

age_violation_classification["breeding_suffix"] = (
    age_violation_classification["horse"]
    .apply(extract_breeding_suffix)
)

age_violation_classification["southern_hemisphere_bred"] = (
    age_violation_classification["breeding_suffix"]
    .isin(SOUTHERN_HEMISPHERE_SUFFIXES)
)

# Measure how far each runner sits outside the literal stated condition.
#
# Negative values mean below the minimum.
# Positive values mean above the maximum.

def age_band_difference(row):

    if row["age_compliance_status"] == "below_stated_minimum":
        return row["runner_age"] - row["stated_minimum_age"]

    if row["age_compliance_status"] == "above_stated_maximum":
        return row["runner_age"] - row["stated_maximum_age"]

    return 0


age_violation_classification["age_boundary_difference"] = (
    age_violation_classification.apply(
        age_band_difference,
        axis=1,
    )
)


# ---------------------------------------------------------------------------
# Population summary.
# ---------------------------------------------------------------------------

print("Apparent age-band breaches:", len(age_violation_classification))

print()
print("By breeding jurisdiction / hemisphere:")

display(
    age_violation_classification
    .groupby(
        [
            "breeding_suffix",
            "southern_hemisphere_bred",
            "age_boundary_difference",
        ],
        dropna=False,
    )
    .size()
    .reset_index(name="runner_records")
    .sort_values(
        "runner_records",
        ascending=False,
    )
)


# ---------------------------------------------------------------------------
# Show every individual case.
# ---------------------------------------------------------------------------

print()
print("Individual apparent breaches:")

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", None)

display(
    age_violation_classification[
        [
            "race_date",
            "governed_racecourse_name",
            "race_type_raw",
            "age_band_raw",
            "horse",
            "breeding_suffix",
            "runner_age",
            "age_compliance_status",
            "age_boundary_difference",
            "southern_hemisphere_bred",
            "race_name_raw",
        ]
    ].sort_values(
        [
            "southern_hemisphere_bred",
            "race_date",
            "horse",
        ],
        ascending=[True, True, True],
    )
)

Apparent age-band breaches: 27

By breeding jurisdiction / hemisphere:


,breeding_suffix,southern_hemisphere_bred,age_boundary_difference,runner_records
1,AUS,True,-1.0,22
3,SAF,True,-1.0,3
0,ARG,True,-1.0,1
2,GB,False,1.0,1



Individual apparent breaches:


,race_date,governed_racecourse_name,race_type_raw,age_band_raw,horse,breeding_suffix,runner_age,age_compliance_status,age_boundary_difference,southern_hemisphere_bred,race_name_raw
4,2017-07-27,Great Yarmouth,Flat,2yo,Millies Kiss (GB),GB,3,above_stated_maximum,1.0,False,Read Silvestre De Sousa At 188Bet Novice Auction Stakes (Plus 10 Race)
0,2015-06-20,Ascot,Flat,4yo+,Brazen Beau (AUS),AUS,3,below_stated_minimum,-1.0,True,Diamond Jubilee Stakes (British Champions Series & Global Sprint Challenge)
1,2015-06-20,Ascot,Flat,4yo+,Wandjina (AUS),AUS,3,below_stated_minimum,-1.0,True,Diamond Jubilee Stakes (British Champions Series & Global Sprint Challenge)
2,2016-06-18,Ascot,Flat,4yo+,Holler (AUS),AUS,3,below_stated_minimum,-1.0,True,Diamond Jubilee Stakes (British Champions Series & Global Sprint Challenge)
3,2017-06-21,Ascot,Flat,4yo+,Greta G (ARG),ARG,3,below_stated_minimum,-1.0,True,Duke Of Cambridge Stakes (Fillies & Mares)
5,2018-04-30,Windsor,Flat,2yo,Anthem Of Peace (AUS),AUS,1,below_stated_minimum,-1.0,True,British EBF Novice Stakes (Plus 10 Race)
6,2018-05-10,Chester,Flat,2yo,Anthem Of Peace (AUS),AUS,1,below_stated_minimum,-1.0,True,British Stallion Studs EBF Maiden Stakes (Plus 10 Race)
7,2018-06-01,Bath,Flat,2yo,Anthem Of Peace (AUS),AUS,1,below_stated_minimum,-1.0,True,EBF Novice Stakes
8,2018-06-10,Goodwood,Flat,2yo,Anthem Of Peace (AUS),AUS,1,below_stated_minimum,-1.0,True,Sutton Winson Backing The NSPCC Selling Stakes
9,2018-06-18,Windsor,Flat,2yo,Anthem Of Peace (AUS),AUS,1,below_stated_minimum,-1.0,True,Sky Bet Best Odds Guaranteed Selling Stakes


In [37]:
# Re-test the apparent age-band exceptions efficiently after the kernel restart.
#
# The previous approach loaded ~985,000 runner rows into pandas and then applied
# Python functions row by row. That was unnecessarily memory-heavy.
#
# This version pushes the work into SQLite and returns ONLY runners whose
# recorded age lies outside the explicit race-level age condition.
#
# Analytical question:
#
#   After accounting for the observed one-year Southern Hemisphere age
#   convention, how many apparent age-condition breaches remain?
#
# No source off-time field is used.

from pathlib import Path

import pandas as pd

from inside_rails.source_sqlite import connect_read_only


# ---------------------------------------------------------------------------
# Locate the accepted Database v4 release from wherever the notebook is run.
# ---------------------------------------------------------------------------

PROJECT_ROOT = Path.cwd()

while (
    not (PROJECT_ROOT / "data").exists()
    and PROJECT_ROOT.parent != PROJECT_ROOT
):
    PROJECT_ROOT = PROJECT_ROOT.parent

DATABASE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "database"
    / "releases"
    / "inside_rails_v4.sqlite3"
)

assert DATABASE.exists(), f"Database not found: {DATABASE}"


# ---------------------------------------------------------------------------
# Ask SQLite for ONLY literal age-band exceptions.
#
# This should reproduce the 27 cases already observed:
#
#   26 below the stated minimum
#    1 above the stated maximum
# ---------------------------------------------------------------------------

with connect_read_only(DATABASE) as connection:
    age_exceptions = pd.read_sql_query(
        """
        SELECT
            race.source_race_occurrence_code,
            race.raw_date AS race_date,
            race.governed_racecourse_name,
            race.race_type_raw,
            race.race_name_raw,
            race.age_band_raw,
            race.stated_minimum_age,
            race.stated_maximum_age,

            runner.raw_horse AS horse,
            runner.age_recorded AS runner_age,

            CASE
                WHEN runner.age_recorded < race.stated_minimum_age
                    THEN 'below_stated_minimum'

                WHEN race.stated_maximum_age IS NOT NULL
                 AND runner.age_recorded > race.stated_maximum_age
                    THEN 'above_stated_maximum'
            END AS literal_status,

            CASE
                WHEN runner.raw_horse LIKE '% (AUS)'
                  OR runner.raw_horse LIKE '% (NZ)'
                  OR runner.raw_horse LIKE '% (SAF)'
                  OR runner.raw_horse LIKE '% (ARG)'
                  OR runner.raw_horse LIKE '% (BRZ)'
                  OR runner.raw_horse LIKE '% (CHI)'
                  OR runner.raw_horse LIKE '% (URU)'
                    THEN 1
                ELSE 0
            END AS southern_hemisphere_bred

        FROM view_gb_reconciled_race_occurrences_with_racecourse AS race

        JOIN view_reconciled_source_runner_participations AS runner
          ON runner.source_race_occurrence_code =
             race.source_race_occurrence_code

        WHERE runner.age_recorded IS NOT NULL

          AND (
                (
                    race.stated_minimum_age IS NOT NULL
                    AND runner.age_recorded < race.stated_minimum_age
                )
                OR
                (
                    race.stated_maximum_age IS NOT NULL
                    AND runner.age_recorded > race.stated_maximum_age
                )
          )

        ORDER BY
            race.raw_date,
            race.governed_racecourse_name,
            runner.raw_horse
        """,
        connection,
    )


# ---------------------------------------------------------------------------
# Apply the observed +1 comparison only to these few Southern Hemisphere
# exceptions. This does NOT alter the preserved runner age.
# ---------------------------------------------------------------------------

age_exceptions["age_for_band_comparison"] = (
    age_exceptions["runner_age"]
    + age_exceptions["southern_hemisphere_bred"]
)


def adjusted_status(row):

    age = row["age_for_band_comparison"]

    if age < row["stated_minimum_age"]:
        return "below_stated_minimum"

    if (
        pd.notna(row["stated_maximum_age"])
        and age > row["stated_maximum_age"]
    ):
        return "above_stated_maximum"

    return "within_stated_age_condition"


age_exceptions["adjusted_status"] = age_exceptions.apply(
    adjusted_status,
    axis=1,
)


remaining_exceptions = age_exceptions[
    age_exceptions["adjusted_status"]
    != "within_stated_age_condition"
].copy()


print("Literal apparent exceptions:", len(age_exceptions))
print(
    "Southern Hemisphere apparent exceptions:",
    int(age_exceptions["southern_hemisphere_bred"].sum()),
)
print(
    "Remaining after +1 comparison:",
    len(remaining_exceptions),
)

print()
print("Remaining cases:")

pd.set_option("display.max_colwidth", None)

display(
    remaining_exceptions[
        [
            "race_date",
            "governed_racecourse_name",
            "race_type_raw",
            "age_band_raw",
            "horse",
            "runner_age",
            "age_for_band_comparison",
            "adjusted_status",
            "race_name_raw",
        ]
    ]
)

Literal apparent exceptions: 27
Southern Hemisphere apparent exceptions: 26
Remaining after +1 comparison: 1

Remaining cases:


,race_date,governed_racecourse_name,race_type_raw,age_band_raw,horse,runner_age,age_for_band_comparison,adjusted_status,race_name_raw
4,2017-07-27,Great Yarmouth,Flat,2yo,Millies Kiss (GB),3,3,above_stated_maximum,Read Silvestre De Sousa At 188Bet Novice Auction Stakes (Plus 10 Race)


### Finding — apparent runner-age breaches are explained by Southern Hemisphere age treatment except for one documented wrong-horse incident

The complete GB source-backed runner population was tested against the explicit
prospective age condition of each race.

A literal numerical comparison identified 27 runner records outside the stated
age band.

Of those 27:

- 26 runners were bred in Southern Hemisphere jurisdictions:
  - Australia: 22;
  - South Africa: 3;
  - Argentina: 1;
- all 26 were recorded exactly one year below the stated minimum age;
- the remaining runner was British-bred and was one year above the stated
  maximum.

The Southern Hemisphere pattern was completely systematic.

When a separate analytical comparison age of `recorded age + 1` was used for
Southern Hemisphere-bred horses, all 26 apparent breaches moved inside the
stated race condition.

This adjustment does **not** replace or correct the preserved runner age. It is
an analytical test of the observed age-convention relationship.

The result is consistent with BHA's separate treatment of Southern Hemisphere
horses, whose breeding season is offset from that of Northern Hemisphere
horses and for whom separate age/weight-for-age provisions exist.

After that comparison, only one runner remained outside the stated race age
condition:

- **Millies Kiss (GB)**, recorded aged three in a race restricted to
  two-year-olds at Great Yarmouth on 27 July 2017.

That case is not an unexplained eligibility exception. It is the documented
incident in which the wrong horse was mistakenly sent out to race.

Therefore:

> **There are no unexplained literal age-band breaches among the 984,757
> source-backed GB runner records tested.**

However, the investigation also shows that age eligibility cannot safely be
tested by comparing the displayed numerical runner age with `ageLimit` in
isolation.

For Southern Hemisphere-bred horses, the relationship between recorded age and
British race-age conditions requires the applicable international/BHA age
convention to be considered.

The age condition remains a genuine prospective eligibility property of the
race, but determining whether an individual horse satisfies it may require
additional horse-level context.